# 01. Configuracoes e caminhos

**v66_35 - EXPERIMENTAL: NEC industrial antes da alocacao; rateio por componente de demanda dentro da prioridade CLASS_ROT. Matriz de Cortes original preservada.**

Derivada diretamente da v66_26. Unica mudanca de regra: dentro de cada PRIOR_MATPAR e PRIOR_ROT, produtos com menor CLASS_ROT consomem capacidade antes de classes maiores. Permanece rateio proporcional somente entre itens da mesma classe. Preservadas as restricoes globais REC/FER, o transbordamento por PRIOR_ROT, a Matriz de Cortes e as regras industriais. Nao homologada; validar contra a v66_26.

In [ ]:

# ============================================================
# 00. CONFIGURAÇÃO
# ============================================================



try:
    from functions import *
except Exception:
    import time
    class Temporizador:
        def iniciar(self):
            self._inicio = time.time()
            return None
        def parar(self):
            fim = time.time()
            inicio = getattr(self, '_inicio', fim)
            return fim - inicio
        def finalizar(self):
            return self.parar()
from pathlib import Path
import time
from datetime import datetime
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.utils import get_column_letter
try:
    from IPython.display import display
except Exception:
    display = print
    
# Iniciando Temporizador
timer = Temporizador()
timer.iniciar()
_TEMPO_TOTAL_INICIO_V58 = time.perf_counter()

PIPELINE_VERSION = "v66_36"
OUTPUT_SOLVER_FILENAME = "bd_SOLVER_v66_36.xlsx"
ESTRUTURA_FILENAME = "estruturas.xlsx"

# Resolve o projeto pela localização do notebook/script, sem depender de usuário, máquina ou OneDrive.
try:
    CAMINHO_EXECUCAO = Path(__file__).resolve().parent
except NameError:
    CAMINHO_EXECUCAO = Path.cwd().resolve()

def _resolver_raiz_ltp_v58(caminho_inicial):
    # Procura a raiz real do LTP pela presença simultânea das pastas operacionais do projeto.
    candidatos = [caminho_inicial, caminho_inicial.parent, caminho_inicial.parent.parent]
    for candidato in candidatos:
        if (candidato / "01_INPUT").is_dir() and (candidato / "02_OUTPUT").is_dir():
            return candidato
    # Em execução isolada, mantém o diretório corrente como raiz para facilitar testes locais.
    return caminho_inicial

RAIZ_LTP = _resolver_raiz_ltp_v58(CAMINHO_EXECUCAO)
PASTA_INPUT = RAIZ_LTP / "01_INPUT"
PASTA_OUTPUT = RAIZ_LTP / "02_OUTPUT"

# Se o notebook já estiver dentro de 02_OUTPUT, usa a própria pasta como saída e a irmã 01_INPUT.
if CAMINHO_EXECUCAO.name.upper() == "02_OUTPUT":
    PASTA_OUTPUT = CAMINHO_EXECUCAO
    PASTA_INPUT = CAMINHO_EXECUCAO.parent / "01_INPUT"

INPUT_FILE = PASTA_OUTPUT / "bd_LTP_NEC_SOLVER.xlsx"
OUTPUT_SOLVER_FILE = PASTA_OUTPUT / OUTPUT_SOLVER_FILENAME
ESTRUTURA_FILE = PASTA_INPUT / ESTRUTURA_FILENAME

# Compatibilidade apenas para teste local quando o arquivo oficial foi anexado com sufixo numérico.
if not INPUT_FILE.exists():
    candidatos_input = sorted(CAMINHO_EXECUCAO.glob("bd_LTP_NEC_SOLVER*.xlsx"))
    if candidatos_input:
        INPUT_FILE = candidatos_input[-1]

TOL = 1e-7
DATA_HORA_EXECUCAO = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
TEMPOS_ETAPAS_V58 = {}

def _iniciar_etapa_v58(nome):
    TEMPOS_ETAPAS_V58[nome] = -time.perf_counter()

def _finalizar_etapa_v58(nome):
    TEMPOS_ETAPAS_V58[nome] = TEMPOS_ETAPAS_V58.get(nome, 0.0) + time.perf_counter()
    return TEMPOS_ETAPAS_V58[nome]

print("=" * 60)
print(" KRONA - SOLVER LTP v66_36 - NEC PRE-ALOCACAO")
print("=" * 60)


# ============================================================
# CONTROLE DE EXPORTACAO DAS GUIAS
# ============================================================
# Mantem todos os calculos internos do Solver.
# As flags abaixo controlam somente quais DataFrames serao gravados no Excel final.
GUIAS_EXPORTAR = {
    "03_PLANO_PRODUCAO": True,
    "04_AUDITORIA_MATRIZ_CORTES": True,
}


# 02. Leitura e preparação das bases


In [ ]:
print("[01/04] Carregando e preparando bases de entrada...")
_iniciar_etapa_v58("PREPARACAO_BASES")

# ============================================================
# 01. FUNÇÕES AUXILIARES
# ============================================================

def to_num(s):
    return pd.to_numeric(s, errors="coerce").fillna(0)


def ensure_columns(df: pd.DataFrame, cols, default=0):
    for c in cols:
        if c not in df.columns:
            df[c] = default
    return df


def ajustar_nec_lote_embalagem(nec, lote_min, qtd_emb):
    """Aplica as regras industriais diretamente na NEC, de forma vetorizada.

    Regras:
    - NEC <= 0 permanece 0;
    - NEC positiva abaixo do LOTE_MIN sobe para o LOTE_MIN;
    - depois, quando QTD_EMB > 0, arredonda para cima para o próximo múltiplo válido.
    """
    nec = pd.to_numeric(nec, errors="coerce").fillna(0.0)
    lote_min = pd.to_numeric(lote_min, errors="coerce").fillna(0.0).clip(lower=0.0)
    qtd_emb = pd.to_numeric(qtd_emb, errors="coerce").fillna(0.0).clip(lower=0.0)

    nec_ajustada = nec.clip(lower=0.0)
    mask_pos = nec_ajustada > 0
    nec_ajustada = nec_ajustada.where(~mask_pos, np.maximum(nec_ajustada, lote_min))

    mask_emb = mask_pos & (qtd_emb > 0)
    nec_ajustada.loc[mask_emb] = np.ceil((nec_ajustada.loc[mask_emb] - 1e-12) / qtd_emb.loc[mask_emb]) * qtd_emb.loc[mask_emb]
    return nec_ajustada


def is_valid_alt(s):
    """Converte a coluna ALTERNATIVA_VALIDA para booleano de forma segura.

    Aceita booleanos reais e textos comuns como TRUE, SIM, YES, 1, VALIDA e ALTERNATIVA_VALIDA.
    Qualquer valor vazio ou diferente disso vira False.
    """
    if isinstance(s, pd.Series):
        if str(s.dtype).lower() == "bool":
            return s.fillna(False)
        txt = s.fillna(False).astype(str).str.strip().str.upper()
        return txt.isin(["TRUE", "VERDADEIRO", "SIM", "YES", "Y", "1", "VALIDA", "VÁLIDA", "ALTERNATIVA_VALIDA"])
    return bool(s)


def calcular_nec_pcs_se_necessario(df: pd.DataFrame, aplicar_regras_industriais: bool = True) -> pd.DataFrame:
    """Calcula NEC_PCS e permite separar otimização contínua do fechamento industrial.

    aplicar_regras_industriais=True:
        preserva o comportamento histórico da V66, aplicando LIMIT_PCS, LOTE_MIN e QTD_EMB.
    aplicar_regras_industriais=False:
        calcula a NEC contínua usada nas iterações da Matriz, sem LIMIT_PCS, LOTE_MIN ou QTD_EMB.
    """
    df = df.copy()
    if "NEC_PCS" in df.columns and aplicar_regras_industriais:
        df = ensure_columns(df, ["LOTE_MIN", "QTD_EMB"], 0)
        df["NEC_PCS"] = to_num(df["NEC_PCS"]).clip(lower=0.0)
        df["NEC_PCS"] = ajustar_nec_lote_embalagem(df["NEC_PCS"], df["LOTE_MIN"], df["QTD_EMB"])
        if "NEC_ESTRUTURA_PAI_PCS" not in df.columns:
            df["NEC_DIRETA_BRUTA_PCS"] = df["NEC_PCS"]
            df["NEC_ESTRUTURA_PAI_PCS"] = df["NEC_PCS"]
        else:
            df["NEC_ESTRUTURA_PAI_PCS"] = to_num(df["NEC_ESTRUTURA_PAI_PCS"]).clip(lower=0.0)
            df["NEC_ESTRUTURA_PAI_PCS"] = ajustar_nec_lote_embalagem(
                df["NEC_ESTRUTURA_PAI_PCS"], df["LOTE_MIN"], df["QTD_EMB"]
            )
        return df

    # Em modo continuo, NEC_PCS preexistente nao e fonte de verdade, pois pode
    # carregar LIMIT_PCS/LOTE_MIN/QTD_EMB de uma etapa anterior. A NEC e
    # reconstruida abaixo diretamente pelos componentes LTP.

    required = [
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS",
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_COMP_NEC_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    # Demanda bruta antes de deduzir estoque/ORI/transito.
    # Esta demanda é usada como base da explosão da estrutura, seguindo o comportamento do motor antigo
    # que explodia componentes antes de o corte/solver eliminar itens por estoque/capacidade.
    demanda_bruta = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
    )
    mesma_reg_nao = df.get("MESMA_REG", "SIM").astype(str).str.upper().eq("NAO")
    demanda_bruta = np.where(mesma_reg_nao, demanda_bruta + df["LTP_SALDO_PREV_PROX_MES_PCS"], demanda_bruta)
    demanda_bruta = pd.Series(demanda_bruta, index=df.index).clip(lower=0)
    demanda_bruta = demanda_bruta + df["LTP_COMP_NEC_PCS"]
    if aplicar_regras_industriais:
        demanda_bruta = np.maximum(demanda_bruta, df["LIMIT_PCS"])

    # Demanda líquida para o solver: deduz estoque/transito/ORI/triangulação.
    base_comum = (
        df["LTP_CART_ARR_MES_ANT"] + df["LTP_CART_MES_ATUAL"] + df["LTP_SALDO_PREV_PCS"] + df["LTP_EST_SEG_PCS"]
        - df["LTP_EST_INI_PCS"] - df["LTP_EST_TRANS_PCS"] - df["ORI_TOT_PCS"] - df["TRIANG_TOT_PCS"]
    )
    nec = np.where(mesma_reg_nao, base_comum + df["LTP_SALDO_PREV_PROX_MES_PCS"], base_comum)
    nec = pd.Series(nec, index=df.index).clip(lower=0)
    nec = nec + df["LTP_COMP_NEC_PCS"]
    if aplicar_regras_industriais:
        nec = np.maximum(nec, df["LIMIT_PCS"])

    nec = pd.Series(nec, index=df.index).fillna(0.0).clip(lower=0.0)
    if aplicar_regras_industriais:
        df["NEC_PCS"] = ajustar_nec_lote_embalagem(nec, df["LOTE_MIN"], df["QTD_EMB"])
    else:
        df["NEC_PCS"] = nec

    # Demanda bruta calculada para auditoria e para exceções de explosão estrutural.
    # Por padrão, a explosão usa a NEC líquida do pai (NEC_PCS).
    # Exceções de retorno de componente/transferência são ativadas depois que a estrutura é conhecida.
    nec_estrutura_bruta = demanda_bruta.copy()
    df["NEC_DIRETA_BRUTA_PCS"] = pd.Series(nec_estrutura_bruta, index=df.index).fillna(0)
    df["NEC_ESTRUTURA_PAI_PCS"] = df["NEC_PCS"]
    return df


def preparar_base(df: pd.DataFrame):
    df = df.copy()
    for c in ["PRIOR_MATPAR", "PRIOR_ROT", "PCS_HORA", "HOR_REC", "HOR_FER", "NEC_PCS"]:
        if c in df.columns:
            df[c] = to_num(df[c])
    df["HOR_CAP"] = df[["HOR_REC", "HOR_FER"]].min(axis=1)
    mes_ref = df["MES_REF"].min()
    bd_mes = df[df["MES_REF"].eq(mes_ref)].copy()
    return bd_mes


def classificar_motivo_invalida(row):
    motivos = []
    if pd.isna(row.get("ALOC_REC")) or str(row.get("ALOC_REC", "")).strip() == "":
        motivos.append("ALOC_REC_AUSENTE")
    if pd.isna(row.get("COD_FER_UNID")) or str(row.get("COD_FER_UNID", "")).strip() == "":
        motivos.append("COD_FER_UNID_AUSENTE")
    if row.get("PCS_HORA", 0) <= TOL:
        motivos.append("PCS_HORA_ZERO_OU_INVALIDO")
    if row.get("HOR_REC", 0) <= TOL:
        motivos.append("HOR_REC_ZERO_OU_NEGATIVO")
    if row.get("HOR_FER", 0) <= TOL:
        motivos.append("HOR_FER_ZERO_OU_NEGATIVO")
    if row.get("HOR_CAP", 0) <= TOL:
        motivos.append("HOR_CAP_ZERO_OU_NEGATIVO")
    return " | ".join(dict.fromkeys(motivos)) if motivos else "ALTERNATIVA_VALIDA"


def formatar_excel(caminho_arquivo):
    wb = load_workbook(caminho_arquivo)
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(color="FFFFFF", bold=True)
    thin_gray = Side(style="thin", color="D9E2F3")
    for ws in wb.worksheets:
        ws.freeze_panes = "A2"
        ws.sheet_view.showGridLines = False
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal="center", vertical="center")
            cell.border = Border(bottom=thin_gray)
        for col_idx, column_cells in enumerate(ws.columns, start=1):
            max_len = 0
            for cell in column_cells:
                if cell.value is not None:
                    max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 10), 45)
        ws.auto_filter.ref = ws.dimensions
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                if isinstance(cell.value, (int, float)):
                    cell.number_format = '#,##0.00'
    wb.save(caminho_arquivo)



# ============================================================
# 01B. FUNÇÕES - EXPLOSÃO DE ESTRUTURA COM ESTOQUE
# ============================================================

from collections import defaultdict, deque


def normalizar_codigo_serie(s: pd.Series) -> pd.Series:
    return (
        s.astype("string")
         .fillna("")
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
    )


def ler_estrutura_produto(caminho_estrutura: Path) -> pd.DataFrame:
    if not caminho_estrutura.exists():
        print("Arquivo de estrutura não encontrado. A explosão será ignorada:", caminho_estrutura)
        return pd.DataFrame()

    estrutura = pd.read_excel(
        caminho_estrutura,
        sheet_name="estruturas",
        dtype={"cod_prod_acabado": str, "cod_insumo": str, "empresa": str}
    )

    required = ["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]
    missing = [c for c in required if c not in estrutura.columns]
    if missing:
        raise ValueError(f"Colunas obrigatórias ausentes na estrutura: {missing}")

    estrutura = estrutura.copy()
    estrutura["empresa"] = estrutura["empresa"].astype("string").fillna("").str.strip()
    estrutura["cod_prod_acabado"] = normalizar_codigo_serie(estrutura["cod_prod_acabado"])
    estrutura["cod_insumo"] = normalizar_codigo_serie(estrutura["cod_insumo"])
    estrutura["qtd_utilizada_pcs"] = to_num(estrutura["qtd_utilizada_pcs"])
    estrutura = estrutura[
        (estrutura["empresa"] != "")
        & (estrutura["cod_prod_acabado"] != "")
        & (estrutura["cod_insumo"] != "")
    ].copy()

    estrutura = estrutura.drop_duplicates(["empresa", "cod_prod_acabado", "cod_insumo"], keep="first")
    return estrutura


def calcular_estoque_componentes_pos_demanda_direta(df: pd.DataFrame, aplicar_regras_industriais: bool = True) -> pd.DataFrame:
    """Calcula estoque disponível para a explosão após consumir a demanda direta/bruta do próprio item.

    Mantém a lógica do pipeline antigo: estoque total = estoque inicial + trânsito + ORI + triangulação.
    Depois abate a demanda bruta do item, sem descontar estoque nessa demanda, para evitar usar na estrutura
    o estoque que já estaria comprometido com a necessidade direta do próprio produto.
    """
    df = df.copy()
    required = [
        "LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS",
        "LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS",
        "LTP_SALDO_PREV_PROX_MES_PCS", "LTP_EST_SEG_PCS", "LIMIT_PCS", "LOTE_MIN", "QTD_EMB"
    ]
    df = ensure_columns(df, required, 0)
    for c in required:
        df[c] = to_num(df[c])

    df["COD_PROD"] = normalizar_codigo_serie(df["COD_PROD"])
    df["UNID_PROD"] = df["UNID_PROD"].astype("string").fillna("").str.strip()
    if "UNID_FAT" not in df.columns:
        df["UNID_FAT"] = df["UNID_PROD"]
    df["UNID_FAT"] = df["UNID_FAT"].astype("string").fillna("").str.strip()
    tipo_prod = df.get("TIPO_PROD", "").astype(str).str.upper()
    mesma_reg = df.get("MESMA_REG", "SIM").astype(str).str.upper()

    estoque_total = (
        df["LTP_EST_INI_PCS"]
        + df["LTP_EST_TRANS_PCS"]
        + df["ORI_TOT_PCS"]
        + df["TRIANG_TOT_PCS"]
    )

    demanda_base = (
        df["LTP_CART_ARR_MES_ANT"]
        + df["LTP_CART_MES_ATUAL"]
        + df["LTP_SALDO_PREV_PCS"]
        + df["LTP_EST_SEG_PCS"]
    )
    mask_mesma_reg_nao = mesma_reg.eq("NAO")
    demanda_base = demanda_base.where(~mask_mesma_reg_nao, demanda_base + df["LTP_SALDO_PREV_PROX_MES_PCS"])

    if aplicar_regras_industriais:
        demanda_bruta = np.maximum(demanda_base, df["LIMIT_PCS"])
    else:
        demanda_bruta = np.maximum(demanda_base, 0.0)

    df["ESTOQUE_TOTAL_PCS"] = estoque_total
    df["DEMANDA_BRUTA_DIRETA_PCS"] = np.maximum(demanda_bruta, 0)
    df["ESTOQUE_DISP_EXPLOSAO_PCS"] = np.maximum(df["ESTOQUE_TOTAL_PCS"] - df["DEMANDA_BRUTA_DIRETA_PCS"], 0)

    # Regra herdada do motor de cortes:
    # o estoque disponível para a explosão é localizado pela unidade de faturamento/destino.
    # Isso permite que uma demanda Nordeste do pai gere necessidade em uma linha do componente
    # produzida em Matriz, desde que a linha do componente tenha UNID_FAT = Nordeste.
    estoque = (
        df.groupby(["COD_PROD", "UNID_FAT"], as_index=False)
        .agg(
            ESTOQUE_TOTAL_PCS=("ESTOQUE_TOTAL_PCS", "max"),
            DEMANDA_BRUTA_DIRETA_PCS=("DEMANDA_BRUTA_DIRETA_PCS", "max"),
            ESTOQUE_DISP_EXPLOSAO_PCS=("ESTOQUE_DISP_EXPLOSAO_PCS", "max")
        )
        .rename(columns={"UNID_FAT": "UNID_PROD"})
    )
    return estoque


def explodir_estrutura_multinivel_com_estoque(
    bd_estrutura: pd.DataFrame,
    bd_demanda_raiz: pd.DataFrame,
    bd_estoque_componentes: pd.DataFrame,
    bd_mapa_componentes: pd.DataFrame = None,
    max_nivel: int = 30
):
    cols = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM",
        "NEC_RAIZ_PCS", "COD_PROD_ACAB", "COD_INSUMO", "QTD_UTIL_PCS",
        "NEC_GERADA_BRUTA_PCS", "ESTOQUE_COMP_ANTES_PCS", "ESTOQUE_CONSUMIDO_PCS",
        "NEC_GERADA_LIQ_PCS", "ESTOQUE_COMP_DEPOIS_PCS", "NIVEL", "TRILHA"
    ]
    if bd_estrutura.empty or bd_demanda_raiz.empty:
        return pd.DataFrame(columns=cols), pd.DataFrame(columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])

    estrutura_idx = defaultdict(lambda: defaultdict(list))
    for emp, pai, filho, qtd in bd_estrutura[["empresa", "cod_prod_acabado", "cod_insumo", "qtd_utilizada_pcs"]].itertuples(index=False, name=None):
        estrutura_idx[str(emp)][str(pai)].append((str(filho), float(qtd or 0)))

    estoque_dict = {
        (str(unid), str(cod)): float(est or 0)
        for cod, unid, est in bd_estoque_componentes[["COD_PROD", "UNID_PROD", "ESTOQUE_DISP_EXPLOSAO_PCS"]].itertuples(index=False, name=None)
    }

    # Regra oficial de chaveamento da explosão/retorno:
    # A estrutura é explodida na UNID_PROD corrente e essa unidade é preservada
    # na necessidade gerada. No retorno ao LTP, o casamento é:
    #   COD_INSUMO -> COD_PROD
    #   UNID_PROD  -> UNID_FAT
    # Para continuar a explosão multinível, localiza-se o componente no LTP
    # por COD_PROD + UNID_FAT e usa-se a UNID_PROD dessa linha no próximo nível.
    if bd_mapa_componentes is None or bd_mapa_componentes.empty:
        mapa_comp = pd.DataFrame(columns=["COD_PROD", "UNID_FAT", "UNID_PROD"])
    else:
        mapa_comp = bd_mapa_componentes.copy()
    for _c in ["COD_PROD", "UNID_FAT", "UNID_PROD"]:
        if _c not in mapa_comp.columns:
            mapa_comp[_c] = ""
        mapa_comp[_c] = mapa_comp[_c].astype("string").fillna("").str.strip()
    mapa_comp["COD_PROD"] = normalizar_codigo_serie(mapa_comp["COD_PROD"])
    mapa_comp = mapa_comp.drop_duplicates(["COD_PROD", "UNID_FAT"], keep="first")
    comp_por_fat = {
        (str(cod), str(unid_fat)): str(unid_prod)
        for cod, unid_fat, unid_prod
        in mapa_comp[["COD_PROD", "UNID_FAT", "UNID_PROD"]].itertuples(index=False, name=None)
    }

    def resolver_unidade_prod_componente(cod_filho, unid_prod_explosao):
        cod_filho = str(cod_filho)
        unid_prod_explosao = str(unid_prod_explosao or "").strip()
        return comp_por_fat.get((cod_filho, unid_prod_explosao), unid_prod_explosao)

    resultados = []
    problemas = []

    ordem_cols = ["MES_REF", "UNID_PROD", "COD_PROD", "ID_PROD_UNID_FAT"]
    bd_demanda_raiz_ord = bd_demanda_raiz.sort_values([c for c in ordem_cols if c in bd_demanda_raiz.columns]).copy()

    for row in bd_demanda_raiz_ord[["MES_REF", "UNID_PROD", "UNID_FAT_ORIGEM", "ID_PROD_UNID_FAT", "COD_PROD", "NEC_PCS"]].itertuples(index=False, name=None):
        mes_ref, unid_estrutura, unid_fat_origem, id_origem, cod_origem, nec_raiz = row
        emp = str(unid_estrutura)
        unid_fat_origem = str(unid_fat_origem or emp).strip()
        root = str(cod_origem)
        nec_raiz = float(nec_raiz or 0)
        if nec_raiz <= TOL:
            continue
        estrutura_empresa = estrutura_idx.get(emp, {})
        if root not in estrutura_empresa:
            continue

        # Na fila entra a necessidade líquida do pai.
        # A fila carrega a unidade estrutural do nó para localizar seus filhos, mantendo separada a UNID_FAT da origem.
        fila = deque([(emp, root, 0, nec_raiz, root, {root})])
        while fila:
            emp_no, pai, nivel, nec_pai_liq, trilha, caminho = fila.popleft()
            if nivel >= max_nivel:
                problemas.append((emp_no, root, trilha, "MAX_NIVEL_ATINGIDO"))
                continue

            estrutura_empresa_no = estrutura_idx.get(str(emp_no), {})
            for filho, qtd_util in estrutura_empresa_no.get(pai, []):
                nivel_filho = nivel + 1
                trilha_filho = f"{trilha} -> {filho}"
                ciclo = filho in caminho
                nec_bruta = float(nec_pai_liq or 0) * float(qtd_util or 0)

                if ciclo:
                    problemas.append((emp_no, root, trilha_filho, "CICLO_ESTRUTURA"))
                    continue
                if nec_bruta <= TOL:
                    continue

                # A linha explodida preserva a UNID_PROD que gerou a necessidade.
                # No retorno ao LTP: UNID_PROD -> UNID_FAT e COD_INSUMO -> COD_PROD.
                unid_retorno = str(emp_no).strip()
                unid_prod_componente = resolver_unidade_prod_componente(filho, unid_retorno)
                chave_estoque = (unid_retorno, str(filho))
                estoque_antes = float(estoque_dict.get(chave_estoque, 0.0))
                estoque_consumido = min(estoque_antes, nec_bruta)
                nec_liq = max(nec_bruta - estoque_consumido, 0.0)
                estoque_depois = max(estoque_antes - estoque_consumido, 0.0)
                estoque_dict[chave_estoque] = estoque_depois

                resultados.append((
                    mes_ref, unid_retorno, id_origem, root, nec_raiz, pai, filho, qtd_util,
                    nec_bruta, estoque_antes, estoque_consumido, nec_liq, estoque_depois,
                    nivel_filho, trilha_filho
                ))

                # Só explode abaixo do componente se ainda existe necessidade líquida dele.
                # Para localizar os filhos do componente, usa a unidade produtiva do componente.
                if nec_liq > TOL:
                    fila.append((unid_prod_componente, filho, nivel_filho, nec_liq, trilha_filho, caminho | {filho}))

    bd_explodida = pd.DataFrame.from_records(resultados, columns=cols)
    bd_problemas = pd.DataFrame.from_records(problemas, columns=["UNID_PROD", "COD_PROD_ORIGEM", "TRILHA", "MOTIVO"])
    return bd_explodida, bd_problemas

def consolidar_necessidade_com_estrutura(bd_mes: pd.DataFrame, bd_estrutura_explodida: pd.DataFrame, aplicar_regras_industriais: bool = True, ajustar_nec_inicial: bool = False):
    """Consolida demanda direta e demanda de componente seguindo o motor antigo.

    Regra principal:
    - a explosão da estrutura nasce pela unidade produtiva do pai;
    - a necessidade líquida do componente volta para o LTP pelo chaveamento definido pelo cliente;
    - a chave de retorno é:
        componente.COD_PROD = COD_INSUMO explodido
        componente.UNID_FAT = UNID_PROD registrada na explosão
    - o solver passa a trabalhar com duas naturezas de demanda:
        NEC_DIRETA_PCS = demanda própria do item;
        NEC_COMP_PCS   = demanda líquida gerada por pais/componentes;
        NEC_PCS        = NEC_DIRETA_PCS + NEC_COMP_PCS.
    """
    bd_mes_out = bd_mes.copy()
    if "UNID_FAT" not in bd_mes_out.columns:
        bd_mes_out["UNID_FAT"] = bd_mes_out["UNID_PROD"]
    bd_mes_out["COD_PROD"] = normalizar_codigo_serie(bd_mes_out["COD_PROD"])
    bd_mes_out["UNID_PROD"] = bd_mes_out["UNID_PROD"].astype("string").fillna("").str.strip()
    bd_mes_out["UNID_FAT"] = bd_mes_out["UNID_FAT"].astype("string").fillna("").str.strip()

    bd_mes_out["NEC_PCS_ORIGINAL"] = to_num(bd_mes_out["NEC_PCS"])
    bd_mes_out["NEC_DIRETA_PCS"] = bd_mes_out["NEC_PCS_ORIGINAL"]

    if bd_estrutura_explodida is None or bd_estrutura_explodida.empty:
        nec_comp_retorno = pd.DataFrame(columns=[
            "MES_REF", "UNID_FAT", "COD_PROD",
            "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_COMP_PCS"
        ])
    else:
        nec_comp_retorno = (
            bd_estrutura_explodida
            .groupby(["MES_REF", "UNID_PROD", "COD_INSUMO"], as_index=False)
            .agg(
                NEC_EXPLODIDA_BRUTA_PCS=("NEC_GERADA_BRUTA_PCS", "sum"),
                ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=("ESTOQUE_CONSUMIDO_PCS", "sum"),
                NEC_COMP_PCS=("NEC_GERADA_LIQ_PCS", "sum")
            )
            .rename(columns={"UNID_PROD": "UNID_FAT", "COD_INSUMO": "COD_PROD"})
        )
        nec_comp_retorno["COD_PROD"] = normalizar_codigo_serie(nec_comp_retorno["COD_PROD"])
        nec_comp_retorno["UNID_FAT"] = nec_comp_retorno["UNID_FAT"].astype("string").fillna("").str.strip()

    bd_mes_out = bd_mes_out.merge(
        nec_comp_retorno,
        on=["MES_REF", "UNID_FAT", "COD_PROD"],
        how="left"
    )

    for c in ["NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_COMP_PCS"]:
        bd_mes_out[c] = to_num(bd_mes_out[c])

    # Alias mantido para compatibilidade com guias antigas.
    bd_mes_out["NEC_EXPLODIDA_PCS"] = bd_mes_out["NEC_COMP_PCS"]
    bd_mes_out["LTP_COMP_NEC_PCS"] = bd_mes_out["NEC_COMP_PCS"]
    # v66_30: o NEC real so e conhecido APOS a consolidacao direta + PA->PI.
    # A explosao estrutural acima permanece baseada na NEC real, evitando criar
    # demanda de filhos pelo mero arredondamento industrial do pai.
    bd_mes_out["NEC_PCS_BASE_CALCULADA_INTERNA"] = (
        bd_mes_out["NEC_DIRETA_PCS"] + bd_mes_out["NEC_COMP_PCS"]
    ).clip(lower=0.0)
    bd_mes_out["NEC_PCS_CONSOLIDADA"] = bd_mes_out["NEC_PCS_BASE_CALCULADA_INTERNA"].copy()
    if ajustar_nec_inicial or aplicar_regras_industriais:
        bd_mes_out = ensure_columns(bd_mes_out, ["LIMIT_PCS", "LOTE_MIN", "QTD_EMB"], 0)
        base = to_num(bd_mes_out["NEC_PCS_BASE_CALCULADA_INTERNA"]).clip(lower=0.0)
        limite = to_num(bd_mes_out["LIMIT_PCS"]).clip(lower=0.0)
        # LIMIT_PCS e piso da quantidade positiva, como no fechamento industrial.
        alvo = base.where(base <= 0, np.maximum(base, limite))
        bd_mes_out["NEC_PCS_CONSOLIDADA"] = ajustar_nec_lote_embalagem(
            alvo, bd_mes_out["LOTE_MIN"], bd_mes_out["QTD_EMB"]
        )
    bd_mes_out["NEC_ACRESCIMO_INDUSTRIAL_PRE_ALOCACAO"] = (
        bd_mes_out["NEC_PCS_CONSOLIDADA"] - bd_mes_out["NEC_PCS_BASE_CALCULADA_INTERNA"]
    ).clip(lower=0.0)
    bd_mes_out["NEC_PCS"] = bd_mes_out["NEC_PCS_CONSOLIDADA"]

    chave_item = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT"]
    extras = ["DESC_PROD", "TIPO_PROD"]
    agg = {
        "NEC_DIRETA_PCS": "max",
        "NEC_COMP_PCS": "max",
        "NEC_EXPLODIDA_BRUTA_PCS": "max",
        "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS": "max",
        "NEC_EXPLODIDA_PCS": "max",
        "NEC_PCS_CONSOLIDADA": "max",
        "NEC_PCS_BASE_CALCULADA_INTERNA": "max",
        "NEC_ACRESCIMO_INDUSTRIAL_PRE_ALOCACAO": "max",
    }
    for c in extras:
        if c in bd_mes_out.columns:
            agg[c] = "first"
    nec_consolidada = (
        bd_mes_out.groupby([c for c in chave_item if c in bd_mes_out.columns], as_index=False)
        .agg(agg)
    )
    nec_consolidada["TEM_ROTEIRO_NO_SOLVER"] = nec_consolidada["ID_PROD_UNID_FAT"].notna()

    return bd_mes_out, nec_consolidada

# ============================================================
# 01C. FUNÇÕES - AJUSTE PÓS-SOLVER PAI/FILHO - rotina operacional MOTIVO AJUSTADO
# ============================================================

def calcular_ajuste_pos_solver_pai_filho(estrutura_explodida: pd.DataFrame, nec_consolidada_solver: pd.DataFrame, nao_atend_item: pd.DataFrame, plano_producao: pd.DataFrame):
    """Calcula diagnóstico pós-solver de viabilidade pai/filho.

    rotina:
    - o filho/componente sustenta o pai com estoque já consumido na explosão + produção do filho pelo solver;
    - a produção do filho continua distribuída proporcionalmente entre demanda direta e estrutural quando houver necessidade total compartilhada;
    - todo corte do pai recalcula a capacidade liberada para realocação posterior.
    """
    cols_ajuste = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM", "COD_PROD_ACAB", "COD_INSUMO", "NIVEL", "TRILHA", "QTD_UTIL_PCS",
        "NEC_RAIZ_PCS", "QTD_PAI_SOLVER", "FATOR_ATEND_PAI_SOLVER", "NEC_GERADA_LIQ_PCS", "ESTOQUE_CONSUMIDO_PCS", "NEC_FILHO_POS_CORTE_PAI_PCS",
        "NEC_DIRETA_FILHO_PCS", "NEC_TOTAL_FILHO_SOLVER_PCS", "QTD_FILHO_SOLVER_TOTAL", "QTD_FILHO_SOLVER_ESTRUTURA_PCS", "QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS",
        "TEM_ROTEIRO_FILHO_NO_SOLVER", "FATOR_ATEND_FILHO_TOTAL", "FATOR_FILHO_USADO_PARA_LIMITAR", "NEC_FILHO_ATENDIDA_ESTRUTURA_PCS",
        "QTD_PAI_FINAL_VIAVEL_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "MOTIVO_AJUSTE_PAI_FILHO"
    ]
    cols_resumo = [
        "MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM", "NEC_RAIZ_PCS", "QTD_PAI_SOLVER", "QTD_PAI_FINAL_VIAVEL_EST",
        "QTD_REDUCAO_POS_PAI_FILHO", "FATOR_ATEND_PAI_SOLVER", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "QTD_COMPONENTES_ANALISADOS",
        "QTD_COMPONENTES_LIMITANTES", "QTD_COMPONENTES_SEM_ROTEIRO", "MOTIVO_AJUSTE_PAI_FILHO"
    ]
    if estrutura_explodida is None or estrutura_explodida.empty:
        return pd.DataFrame(columns=cols_ajuste), pd.DataFrame(columns=cols_resumo), pd.DataFrame()

    ajuste = estrutura_explodida.copy()
    for c in ["NEC_RAIZ_PCS", "NEC_GERADA_LIQ_PCS", "QTD_UTIL_PCS", "ESTOQUE_CONSUMIDO_PCS"]:
        if c not in ajuste.columns:
            ajuste[c] = 0.0
        ajuste[c] = to_num(ajuste[c])

    cols_atend_base = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "UNID_PROD", "UNID_FAT", "QTD_ATENDIDA_SOLVER", "NEC_PCS", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA"]
    cols_atend_base = [c for c in cols_atend_base if c in nao_atend_item.columns]
    atend = nao_atend_item[cols_atend_base].copy()
    for c in ["QTD_ATENDIDA_SOLVER", "NEC_PCS", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA"]:
        if c not in atend.columns:
            atend[c] = 0.0
        atend[c] = to_num(atend[c])

    if nec_consolidada_solver is not None and not nec_consolidada_solver.empty and "TEM_ROTEIRO_NO_SOLVER" in nec_consolidada_solver.columns:
        roteiro_filho = nec_consolidada_solver[["MES_REF", "COD_PROD", "UNID_FAT", "TEM_ROTEIRO_NO_SOLVER"]].copy()
        roteiro_filho = roteiro_filho.rename(columns={"COD_PROD": "COD_INSUMO", "UNID_FAT": "UNID_PROD", "TEM_ROTEIRO_NO_SOLVER": "TEM_ROTEIRO_FILHO_NO_SOLVER"})
    else:
        roteiro_filho = pd.DataFrame(columns=["MES_REF", "COD_INSUMO", "UNID_PROD", "TEM_ROTEIRO_FILHO_NO_SOLVER"])

    atend_pai = atend.rename(columns={
        "ID_PROD_UNID_FAT": "ID_PROD_UNID_FAT_ORIGEM",
        "QTD_ATENDIDA_SOLVER": "QTD_PAI_SOLVER",
        "NEC_PCS": "NEC_PAI_SOLVER_PCS"
    })[["MES_REF", "ID_PROD_UNID_FAT_ORIGEM", "QTD_PAI_SOLVER", "NEC_PAI_SOLVER_PCS"]]
    ajuste = ajuste.merge(atend_pai, on=["MES_REF", "ID_PROD_UNID_FAT_ORIGEM"], how="left")
    ajuste["QTD_PAI_SOLVER"] = ajuste["QTD_PAI_SOLVER"].fillna(0)
    ajuste["FATOR_ATEND_PAI_SOLVER"] = np.where(ajuste["NEC_RAIZ_PCS"] > TOL, (ajuste["QTD_PAI_SOLVER"] / ajuste["NEC_RAIZ_PCS"]).clip(0, 1), 0)
    ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] = ajuste["NEC_GERADA_LIQ_PCS"] * ajuste["FATOR_ATEND_PAI_SOLVER"]

    atend_filho = atend.rename(columns={
        "COD_PROD": "COD_INSUMO",
        "UNID_FAT": "UNID_PROD_ESTRUTURA",
        "QTD_ATENDIDA_SOLVER": "QTD_FILHO_SOLVER_TOTAL",
        "NEC_PCS": "NEC_TOTAL_FILHO_SOLVER_PCS",
        "NEC_DIRETA_PCS": "NEC_DIRETA_FILHO_PCS"
    })[["MES_REF", "UNID_PROD_ESTRUTURA", "COD_INSUMO", "QTD_FILHO_SOLVER_TOTAL", "NEC_TOTAL_FILHO_SOLVER_PCS", "NEC_DIRETA_FILHO_PCS"]]
    atend_filho = atend_filho.rename(columns={"UNID_PROD_ESTRUTURA": "UNID_PROD"})
    ajuste = ajuste.merge(atend_filho, on=["MES_REF", "UNID_PROD", "COD_INSUMO"], how="left")
    ajuste = ajuste.merge(roteiro_filho, on=["MES_REF", "UNID_PROD", "COD_INSUMO"], how="left")
    for c in ["QTD_FILHO_SOLVER_TOTAL", "NEC_TOTAL_FILHO_SOLVER_PCS", "NEC_DIRETA_FILHO_PCS"]:
        ajuste[c] = ajuste[c].fillna(0)
    ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"] = ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"].fillna(False).astype(bool)

    # Atendimento total do filho no solver e parcela da produção do filho atribuída à necessidade estrutural.
    ajuste["FATOR_ATEND_FILHO_TOTAL"] = np.where(
        ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"] > TOL,
        (ajuste["QTD_FILHO_SOLVER_TOTAL"] / ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"]).clip(0, 1),
        1.0
    )
    ajuste["PESO_ESTRUTURA_NO_FILHO"] = np.where(
        ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"] > TOL,
        (ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] / ajuste["NEC_TOTAL_FILHO_SOLVER_PCS"]).clip(0, 1),
        0.0
    )
    ajuste["QTD_FILHO_SOLVER_ESTRUTURA_PCS"] = ajuste["QTD_FILHO_SOLVER_TOTAL"] * ajuste["PESO_ESTRUTURA_NO_FILHO"]

    # Regra: componente sustenta o pai com estoque já consumido na explosão + produção do filho alocada à estrutura.
    ajuste["QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS"] = ajuste["ESTOQUE_CONSUMIDO_PCS"].fillna(0) + ajuste["QTD_FILHO_SOLVER_ESTRUTURA_PCS"].fillna(0)
    ajuste["NEC_FILHO_ATENDIDA_ESTRUTURA_PCS"] = np.minimum(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"], ajuste["QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS"])

    ajuste["QTD_PAI_SUSTENTADA_PELO_FILHO"] = np.where(
        ajuste["QTD_UTIL_PCS"] > TOL,
        ajuste["QTD_FILHO_DISPONIVEL_ESTRUTURA_PCS"] / ajuste["QTD_UTIL_PCS"],
        ajuste["QTD_PAI_SOLVER"]
    )
    ajuste["QTD_PAI_FINAL_LINHA"] = ajuste["QTD_PAI_SOLVER"]
    mask_limita = (ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"])
    ajuste.loc[mask_limita, "QTD_PAI_FINAL_LINHA"] = np.minimum(
        ajuste.loc[mask_limita, "QTD_PAI_SOLVER"],
        ajuste.loc[mask_limita, "QTD_PAI_SUSTENTADA_PELO_FILHO"]
    )
    ajuste["FATOR_FILHO_USADO_PARA_LIMITAR"] = np.where(
        ajuste["QTD_PAI_SOLVER"] > TOL,
        (ajuste["QTD_PAI_FINAL_LINHA"] / ajuste["QTD_PAI_SOLVER"]).clip(0, 1),
        1.0
    )
    ajuste.loc[ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] <= TOL, "FATOR_FILHO_USADO_PARA_LIMITAR"] = 1.0
    ajuste.loc[~ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"], "FATOR_FILHO_USADO_PARA_LIMITAR"] = 1.0

    ch_origem = ["MES_REF", "UNID_PROD", "ID_PROD_UNID_FAT_ORIGEM", "COD_PROD_ORIGEM"]
    resumo = ajuste.groupby(ch_origem, as_index=False).agg(
        NEC_RAIZ_PCS=("NEC_RAIZ_PCS", "max"),
        QTD_PAI_SOLVER=("QTD_PAI_SOLVER", "max"),
        FATOR_ATEND_PAI_SOLVER=("FATOR_ATEND_PAI_SOLVER", "max"),
        QTD_PAI_FINAL_VIAVEL_EST=("QTD_PAI_FINAL_LINHA", "min"),
        QTD_COMPONENTES_ANALISADOS=("COD_INSUMO", "nunique")
    )
    resumo["QTD_PAI_FINAL_VIAVEL_EST"] = np.minimum(resumo["QTD_PAI_FINAL_VIAVEL_EST"], resumo["QTD_PAI_SOLVER"])
    resumo["QTD_REDUCAO_POS_PAI_FILHO"] = (resumo["QTD_PAI_SOLVER"] - resumo["QTD_PAI_FINAL_VIAVEL_EST"]).clip(lower=0)
    resumo["FATOR_FINAL_VIAVEL_ORIGEM"] = np.where(resumo["NEC_RAIZ_PCS"] > TOL, (resumo["QTD_PAI_FINAL_VIAVEL_EST"] / resumo["NEC_RAIZ_PCS"]).clip(0, 1), 0)
    resumo["FATOR_AJUSTE_SOBRE_SOLVER"] = np.where(resumo["QTD_PAI_SOLVER"] > TOL, (resumo["QTD_PAI_FINAL_VIAVEL_EST"] / resumo["QTD_PAI_SOLVER"]).clip(0, 1), 0)

    limitantes = ajuste[(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"]) & (ajuste["QTD_PAI_FINAL_LINHA"] < ajuste["QTD_PAI_SOLVER"] - TOL)]
    qtd_limitantes = limitantes.groupby(ch_origem)["COD_INSUMO"].nunique().reset_index(name="QTD_COMPONENTES_LIMITANTES") if not limitantes.empty else pd.DataFrame(columns=ch_origem + ["QTD_COMPONENTES_LIMITANTES"])
    sem_roteiro = ajuste[(ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] > TOL) & (~ajuste["TEM_ROTEIRO_FILHO_NO_SOLVER"])]
    qtd_sem_roteiro = sem_roteiro.groupby(ch_origem)["COD_INSUMO"].nunique().reset_index(name="QTD_COMPONENTES_SEM_ROTEIRO") if not sem_roteiro.empty else pd.DataFrame(columns=ch_origem + ["QTD_COMPONENTES_SEM_ROTEIRO"])
    resumo = resumo.merge(qtd_limitantes, on=ch_origem, how="left").merge(qtd_sem_roteiro, on=ch_origem, how="left")
    resumo["QTD_COMPONENTES_LIMITANTES"] = resumo["QTD_COMPONENTES_LIMITANTES"].fillna(0).astype(int)
    resumo["QTD_COMPONENTES_SEM_ROTEIRO"] = pd.to_numeric(resumo["QTD_COMPONENTES_SEM_ROTEIRO"], errors="coerce").fillna(0).astype("int64")
    resumo["MOTIVO_AJUSTE_PAI_FILHO"] = np.select(
        [resumo["QTD_PAI_SOLVER"] <= TOL, resumo["FATOR_ATEND_PAI_SOLVER"] < 1 - TOL, resumo["QTD_COMPONENTES_LIMITANTES"] > 0, resumo["QTD_COMPONENTES_SEM_ROTEIRO"] > 0],
        ["PAI_NAO_PRODUZIDO_PELO_SOLVER", "PAI_CORTADO_PELO_SOLVER", "FILHO_COM_ROTEIRO_LIMITOU_PAI", "FILHO_SEM_ROTEIRO_EXPOSTO_NAO_LIMITA"],
        default="SEM_AJUSTE_POS_SOLVER"
    )

    ajuste = ajuste.merge(resumo[ch_origem + ["QTD_PAI_FINAL_VIAVEL_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "MOTIVO_AJUSTE_PAI_FILHO"]], on=ch_origem, how="left")
    ajuste.loc[ajuste["NEC_FILHO_POS_CORTE_PAI_PCS"] <= TOL, "MOTIVO_AJUSTE_PAI_FILHO"] = "SEM_NECESSIDADE_FILHO_POS_CORTE_PAI"

    ajuste = ajuste[[c for c in cols_ajuste if c in ajuste.columns]].copy()
    resumo = resumo[[c for c in cols_resumo if c in resumo.columns]].copy()

    fatores = resumo.rename(columns={"ID_PROD_UNID_FAT_ORIGEM": "ID_PROD_UNID_FAT"})[["MES_REF", "ID_PROD_UNID_FAT", "FATOR_AJUSTE_SOBRE_SOLVER", "FATOR_FINAL_VIAVEL_ORIGEM"]]
    if plano_producao is not None and not plano_producao.empty:
        cap_lib = plano_producao.merge(fatores, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
        cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"] = cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"].fillna(1.0)
        cap_lib["FATOR_FINAL_VIAVEL_ORIGEM"] = cap_lib["FATOR_FINAL_VIAVEL_ORIGEM"].fillna(1.0)
        for c in ["QTD_PRODUZIR_SOLVER", "PCS_HORA", "HR_PRODUZIR_SOLVER"]:
            cap_lib[c] = to_num(cap_lib[c]) if c in cap_lib.columns else 0.0
        cap_lib["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = cap_lib["QTD_PRODUZIR_SOLVER"] * cap_lib["FATOR_AJUSTE_SOBRE_SOLVER"]
        cap_lib["QTD_REDUCAO_POS_PAI_FILHO"] = (cap_lib["QTD_PRODUZIR_SOLVER"] - cap_lib["QTD_PRODUZIR_FINAL_VIAVEL_EST"]).clip(lower=0)
        cap_lib["HR_LIBERADA_EST"] = np.where(cap_lib["PCS_HORA"] > TOL, cap_lib["QTD_REDUCAO_POS_PAI_FILHO"] / cap_lib["PCS_HORA"], 0)
        cap_lib = cap_lib[cap_lib["HR_LIBERADA_EST"] > TOL].copy()
        cap_lib = cap_lib[[c for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "QTD_PRODUZIR_SOLVER", "QTD_PRODUZIR_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "HR_PRODUZIR_SOLVER", "HR_LIBERADA_EST", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER"] if c in cap_lib.columns]].copy()
    else:
        cap_lib = pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "ALOC_REC", "COD_FER_UNID", "HR_LIBERADA_EST"])

    return ajuste, resumo, cap_lib

_finalizar_etapa_v58("PREPARACAO_BASES")
print("       OK - Bases preparadas")


# 03. Solver, regras operacionais e encadeamento


In [ ]:
print("[02/04] Processando Solver, regras operacionais e encadeamento...")
_iniciar_etapa_v58("SOLVER_ENCADEAMENTO")

# ============================================================
# 02-06E. MOTOR MENSAL REUTILIZAVEL
# ============================================================

PRIORIDADE_COMPONENTES_V6629 = [
    ('NAO', 'LTP_CART_ARR_MES_ANT', 'NAO|C_ARR'),
    ('SIM', 'LTP_CART_ARR_MES_ANT', 'SIM|C_ARR'),
    ('NAO', 'LTP_CART_MES_ATUAL', 'NAO|C_AT'),
    ('SIM', 'LTP_CART_MES_ATUAL', 'SIM|C_AT'),
    ('NAO', 'LTP_SALDO_PREV_PCS', 'NAO|PREV_VEN'),
    ('SIM', 'LTP_SALDO_PREV_PCS', 'SIM|PREV_VEN'),
    ('NAO', 'LTP_EST_SEG_PCS', 'NAO|EST_SEG'),
    ('NAO', 'LTP_SALDO_PREV_PROX_MES_PCS', 'NAO|PREV_PROX'),
    ('SIM', 'LTP_EST_SEG_PCS', 'SIM|EST_SEG'),
]

def _executar_motor_periodo_v58(bd_periodo_entrada, aplicar_regras_industriais=True):

    # ============================================================
    # 02. LEITURA DA BASE + EXPLOSÃO INICIAL LÍQUIDA DA ESTRUTURA
    # ============================================================

    bd = bd_periodo_entrada.copy()

    bd["COD_PROD"] = normalizar_codigo_serie(bd["COD_PROD"])
    bd["UNID_PROD"] = bd["UNID_PROD"].astype("string").fillna("").str.strip()
    if "UNID_FAT" in bd.columns:
        bd["UNID_FAT"] = bd["UNID_FAT"].astype("string").fillna("").str.strip()

    bd = calcular_nec_pcs_se_necessario(bd, aplicar_regras_industriais=aplicar_regras_industriais)
    bd_mes = preparar_base(bd)

    estrutura_produto = ler_estrutura_produto(ESTRUTURA_FILE)
    estoque_componentes = calcular_estoque_componentes_pos_demanda_direta(bd_mes, aplicar_regras_industriais=aplicar_regras_industriais)

    # rotina: a estrutura é localizada pela unidade produtiva do pai.
    # A UNID_FAT da origem é preservada separadamente para devolver a necessidade do componente ao plano.
    bd_mes["UNID_ESTRUTURA"] = bd_mes["UNID_PROD"].astype("string").fillna("").str.strip()
    bd_mes["UNID_FAT_ORIGEM"] = bd_mes["UNID_FAT"].where(
        bd_mes["UNID_FAT"].astype("string").fillna("").str.strip().ne(""),
        bd_mes["UNID_PROD"]
    ).astype("string").fillna("").str.strip()

    # rotina: a raiz da explosão também deve respeitar estoque/cobertura antes de gerar filhos.
    # Regra vigente: a explosão usa somente NEC_ESTRUTURA_PAI_PCS líquida.
    # Se o pai raiz tem demanda bruta, mas estoque/trânsito cobre 100%, ele fica no plano com produção zero
    # e MOTIVO_STATUS = DEMANDA_DIRETA_COBERTA_POR_ESTOQUE, mas não gera necessidade produtiva para componentes.
    mask_pai_coberto_sem_explodir = (
        (to_num(bd_mes["NEC_ESTRUTURA_PAI_PCS"]) <= TOL)
        & (to_num(bd_mes.get("NEC_DIRETA_BRUTA_PCS", 0)) > TOL)
    )

    demanda_raiz_estrutura = (
        bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
        .groupby(["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD"], as_index=False)
        .agg(
            UNID_PROD=("UNID_ESTRUTURA", "first"),
            UNID_FAT_ORIGEM=("UNID_FAT_ORIGEM", "first"),
            NEC_PCS=("NEC_ESTRUTURA_PAI_PCS", "max")
        )
    )
    demanda_raiz_estrutura = demanda_raiz_estrutura[demanda_raiz_estrutura["NEC_PCS"] > TOL].copy()

    mapa_componentes_retorno = (
        bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
        .groupby(["COD_PROD", "UNID_FAT"], as_index=False)
        .agg(UNID_PROD=("UNID_PROD", "first"))
    )

    estrutura_explodida, problemas_estrutura = explodir_estrutura_multinivel_com_estoque(
        estrutura_produto,
        demanda_raiz_estrutura,
        estoque_componentes,
        mapa_componentes_retorno,
        max_nivel=30
    )

    # v66_30: arredondamento operacional da NEC ANTES da alocacao, mas DEPOIS
    # da explosao estrutural real. O fechamento industrial final segue ativo.
    bd_mes, nec_consolidada_solver = consolidar_necessidade_com_estrutura(
        bd_mes, estrutura_explodida,
        aplicar_regras_industriais=aplicar_regras_industriais,
        ajustar_nec_inicial=True,
    )

    # Validação direcionada rotina operacional do caso crítico 0544 -> 0544NE.
    try:
        _retorno_0544 = estrutura_explodida[(estrutura_explodida["COD_PROD_ORIGEM"].astype(str) == "0544") & (estrutura_explodida["COD_INSUMO"].astype(str) == "0544NE")].copy()
        _retorno_0544ne = nec_consolidada_solver[nec_consolidada_solver["COD_PROD"].astype(str).eq("0544NE")].copy()
    except Exception as e:
        print("VALIDACAO_AUDITORIA_0544NE_FALHOU:", e)

    nec_explodida_componentes = (
        estrutura_explodida.groupby(["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM", "COD_INSUMO"], as_index=False)
        .agg(
            NEC_EXPLODIDA_BRUTA_PCS=("NEC_GERADA_BRUTA_PCS", "sum"),
            ESTOQUE_CONSUMIDO_EXPLOSAO_PCS=("ESTOQUE_CONSUMIDO_PCS", "sum"),
            NEC_EXPLODIDA_PCS=("NEC_GERADA_LIQ_PCS", "sum"),
            MENOR_NIVEL=("NIVEL", "min")
        )
        if not estrutura_explodida.empty else pd.DataFrame(columns=["MES_REF", "UNID_PROD", "COD_PROD_ORIGEM", "COD_INSUMO", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "MENOR_NIVEL"])
    )

    validacao_explosao = pd.DataFrame({
        "METRICA": [
            "QTD_LINHAS_ESTRUTURA_ORIGINAL",
            "QTD_PRODUTOS_RAIZ_COM_DEMANDA",
            "QTD_LINHAS_ESTRUTURA_EXPLODIDA",
            "QTD_COMPONENTES_COM_NEC_EXPLODIDA_LIQUIDA",
            "MAIOR_NIVEL_EXPLODIDO",
            "QTD_PROBLEMAS_ESTRUTURA",
            "QTD_CICLOS_ESTRUTURA",
            "NEC_DIRETA_TOTAL_PCS",
            "NEC_EXPLODIDA_BRUTA_TOTAL_PCS",
            "ESTOQUE_CONSUMIDO_EXPLOSAO_TOTAL_PCS",
            "NEC_EXPLODIDA_LIQUIDA_TOTAL_PCS",
            "NEC_CONSOLIDADA_TOTAL_PCS",
            "QTD_ITENS_COM_DEMANDA_EXPLODIDA_LIQUIDA",
            "QTD_ITENS_EXPLODIDOS_SEM_ROTEIRO_NO_SOLVER"
        ],
        "VALOR": [
            len(estrutura_produto),
            len(demanda_raiz_estrutura),
            len(estrutura_explodida),
            nec_explodida_componentes.loc[nec_explodida_componentes["NEC_EXPLODIDA_PCS"] > TOL, "COD_INSUMO"].nunique() if not nec_explodida_componentes.empty else 0,
            estrutura_explodida["NIVEL"].max() if not estrutura_explodida.empty else 0,
            len(problemas_estrutura),
            (problemas_estrutura["MOTIVO"].eq("CICLO_ESTRUTURA").sum() if not problemas_estrutura.empty else 0),
            nec_consolidada_solver["NEC_DIRETA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
            nec_consolidada_solver["NEC_EXPLODIDA_BRUTA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
            nec_consolidada_solver["ESTOQUE_CONSUMIDO_EXPLOSAO_PCS"].sum() if not nec_consolidada_solver.empty else 0,
            nec_consolidada_solver["NEC_EXPLODIDA_PCS"].sum() if not nec_consolidada_solver.empty else 0,
            nec_consolidada_solver["NEC_PCS_CONSOLIDADA"].sum() if not nec_consolidada_solver.empty else 0,
            (nec_consolidada_solver["NEC_EXPLODIDA_PCS"] > TOL).sum() if not nec_consolidada_solver.empty else 0,
            ((nec_consolidada_solver["NEC_EXPLODIDA_PCS"] > TOL) & (~nec_consolidada_solver["TEM_ROTEIRO_NO_SOLVER"])).sum() if not nec_consolidada_solver.empty else 0
        ]
    })



    # ============================================================
    # 03. MONTAGEM DA DEMANDA E DAS ALTERNATIVAS
    # ============================================================

    meta_cols = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS", "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA", "LOTE_MIN", "QTD_EMB", "CURVA_ABC", "CURVA_123"]

    bd_demanda_solver = (
        bd_mes.sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT"])
        .groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False)
        .agg({**{c: "first" for c in meta_cols if c not in ["MES_REF", "ID_PROD_UNID_FAT"] and c in bd_mes.columns}, "NEC_PCS": "max"})
    )
    bd_demanda_solver = bd_demanda_solver[bd_demanda_solver["NEC_PCS"] > TOL].copy()

    alt_cols = ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "ID_RECURSO", "ID_FERRAMENTA", "LOTE_MIN", "QTD_EMB", "CURVA_ABC", "CURVA_123"]
    bd_alternativas_item = bd_mes[[c for c in alt_cols if c in bd_mes.columns]].copy()
    bd_alternativas_item = bd_alternativas_item[bd_alternativas_item["ID_PROD_UNID_FAT"].isin(bd_demanda_solver["ID_PROD_UNID_FAT"])].copy()
    bd_alternativas_item = bd_alternativas_item.drop_duplicates([c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in bd_alternativas_item.columns]).reset_index(drop=True)

    bd_alternativas_item["ALTERNATIVA_VALIDA"] = (
        bd_alternativas_item["ALOC_REC"].notna() & bd_alternativas_item["COD_FER_UNID"].notna()
        & (bd_alternativas_item["PCS_HORA"] > TOL) & (bd_alternativas_item["HOR_REC"] > TOL)
        & (bd_alternativas_item["HOR_FER"] > TOL) & (bd_alternativas_item["HOR_CAP"] > TOL)
    )
    bd_alternativas_item["MOTIVO_INVALIDA"] = bd_alternativas_item.apply(classificar_motivo_invalida, axis=1)
    bd_alternativas_validas = bd_alternativas_item[bd_alternativas_item["ALTERNATIVA_VALIDA"]].copy()

    # TESTE v66_27: CLASS_ROT e a contagem de alternativas usada no transbordo antigo.
    # Uma classe invalida ou ausente deve ser informada, nunca interpretada como prioridade 1.
    if "CLASS_ROT" not in bd_alternativas_validas.columns:
        raise ValueError("CLASS_ROT ausente: priorizacao experimental requer a classificacao original dos roteiros.")
    bd_alternativas_validas["CLASS_ROT"] = pd.to_numeric(bd_alternativas_validas["CLASS_ROT"], errors="coerce")
    if (bd_alternativas_validas["CLASS_ROT"].isna().any()
            or (bd_alternativas_validas["CLASS_ROT"] < 1).any()
            or (bd_alternativas_validas["CLASS_ROT"] % 1 != 0).any()):
        raise ValueError("CLASS_ROT invalido em alguma alternativa produtiva valida.")
    bd_alternativas_validas["CLASS_ROT"] = bd_alternativas_validas["CLASS_ROT"].astype(int)
    classes_divergentes = bd_alternativas_validas.groupby("ID_PROD_UNID_FAT")["CLASS_ROT"].nunique()
    if classes_divergentes.gt(1).any():
        raise ValueError("CLASS_ROT inconsistente entre roteiros do mesmo ID_PROD_UNID_FAT.")

    bd_cap_recursos = bd_alternativas_item.dropna(subset=["ALOC_REC"]).groupby("ALOC_REC", as_index=False).agg(HOR_REC=("HOR_REC", "max"))
    bd_cap_ferramentas = bd_alternativas_item.dropna(subset=["COD_FER_UNID"]).groupby("COD_FER_UNID", as_index=False).agg(HOR_FER=("HOR_FER", "max"))



    # ============================================================
    # 04. ALOCACAO POR CLASSE E COMPONENTE (v66_30 experimental)
    # ============================================================
    # 9 criterios: ordem inversa da Matriz de Cortes canonica.
    # A matriz de cortes e a reconciliacao PA->PI continuam inalteradas.
    prioridade_componentes = PRIORIDADE_COMPONENTES_V6629

    # Separa NEC direta de estoque coberto e da parcela estrutural.
    # O estoque cobre primeiro a demanda comercial mais urgente.
    # NEC_COMP_PCS nao tem componente de origem na base: mantem-se em
    # faixa estrutural separada, sem presumir qual carteira a gerou.
    def decompor_demanda_para_rateio(bd_mes, bd_demanda):
        cols = ['MES_REF', 'ID_PROD_UNID_FAT', 'MESMA_REG', 'NEC_DIRETA_PCS',
                'NEC_COMP_PCS', 'NEC_ACRESCIMO_INDUSTRIAL_PRE_ALOCACAO'] + sorted({campo for _, campo, _ in prioridade_componentes})
        src = bd_mes.copy()
        if 'MESMA_REG' not in src:
            src['MESMA_REG'] = 'SIM'
        src = ensure_columns(src, [c for c in cols if c not in src.columns], 0)
        src['MESMA_REG'] = src['MESMA_REG'].astype(str).str.upper().str.strip()
        if src.groupby('ID_PROD_UNID_FAT')['MESMA_REG'].nunique().gt(1).any():
            raise ValueError('MESMA_REG diverge entre roteiros do mesmo produto.')
        src = (src.sort_values(['ID_PROD_UNID_FAT', 'PRIOR_MATPAR', 'PRIOR_ROT'])
               .drop_duplicates(['MES_REF', 'ID_PROD_UNID_FAT'])[cols]
               .set_index('ID_PROD_UNID_FAT'))
        src = src.reindex(bd_demanda['ID_PROD_UNID_FAT']).copy()
        if src['MESMA_REG'].isna().any():
            raise ValueError('Item da NEC sem componentes LTP correspondentes.')
        for campo in set(c for _, c, _ in prioridade_componentes):
            src[campo] = to_num(src[campo]).clip(lower=0)
        direct = to_num(src['NEC_DIRETA_PCS']).clip(lower=0)
        estrutural = to_num(src['NEC_COMP_PCS']).clip(lower=0)
        total = to_num(bd_demanda.set_index('ID_PROD_UNID_FAT')['NEC_PCS']).reindex(src.index).clip(lower=0)
        brutos = [src[campo].where(src['MESMA_REG'].eq(flag), 0.0).to_numpy(float)
                  for flag, campo, _ in prioridade_componentes]
        soma_bruta = np.sum(np.stack(brutos, axis=0), axis=0)
        cobertura = np.maximum(soma_bruta - direct.to_numpy(float), 0.0)
        acum_bruta = np.zeros(len(src), dtype=float)
        faixas = []
        for (flag, campo, nome), bruto in zip(prioridade_componentes, brutos):
            coberto = np.minimum(bruto, np.maximum(cobertura - acum_bruta, 0.0))
            faixa = (bruto - coberto).clip(min=0.0)
            faixas.append((nome, pd.Series(faixa, index=src.index)))
            acum_bruta += bruto
        soma_liquida = np.sum(np.stack([serie.to_numpy(float) for _, serie in faixas]), axis=0)
        restante = total.to_numpy(float) - soma_liquida
        if (restante < -1e-5).any():
            raise ValueError('Decomposicao de demanda excede NEC_PCS: conferir LTP e estoque.')
        # v66_30: NAO confundir arredondamento industrial com demanda estrutural.
        # O acrescimo por LOTE_MIN/QTD_EMB/LIMIT_PCS fica em faixa propria e
        # vem por ultimo: nunca deve furar a prioridade da carteira/previsao.
        operacional = to_num(src['NEC_ACRESCIMO_INDUSTRIAL_PRE_ALOCACAO']).clip(lower=0.0).to_numpy(float)
        if (operacional > restante + 1e-5).any():
            raise ValueError('Acrescimo industrial excede residual da NEC: revisar decomposicao.')
        extra_estrutural = np.maximum(restante - operacional, 0.0)
        estrutural_real = np.maximum(estrutural.to_numpy(float), 0.0)
        if (extra_estrutural + 1e-5 < estrutural_real).any():
            raise ValueError('Parcela estrutural maior que residual da NEC: decomposicao inconsistente.')
        extra = pd.Series(extra_estrutural, index=src.index)
        # O arredondamento nao constitui um 10o criterio de prioridade. Deve
        # acompanhar a faixa mais urgente do proprio item; caso contrario o
        # arredondamento de C_ARR seria atendido DEPOIS do EST_SEG de outro item.
        resto_operacional = operacional.copy()
        faixas_sem_extra = faixas
        if extra.gt(TOL).any():
            mask_estr = extra.to_numpy(float) > TOL
            acrescimo_estr = np.where(mask_estr, resto_operacional, 0.0)
            resto_operacional -= acrescimo_estr
            faixas = [('NEC_ESTRUTURAL_NAO_CLASSIFICADA',
                       extra + pd.Series(acrescimo_estr, index=src.index))] + faixas_sem_extra
        for _, serie in faixas:
            faltante = resto_operacional > TOL
            if not faltante.any():
                break
            presente = serie.to_numpy(float) > TOL
            mask = faltante & presente
            if mask.any():
                serie.iloc[np.flatnonzero(mask)] += resto_operacional[mask]
                resto_operacional[mask] = 0.0
        if (resto_operacional > 1e-5).any():
            raise ValueError('Ajuste industrial sem componente de demanda elegivel.')
        if not np.allclose(sum((x.to_numpy(float) for _, x in faixas)), total.to_numpy(float), rtol=0, atol=1e-5):
            raise ValueError('A soma das faixas difere da necessidade consolidada.')
        return faixas

    faixas_demanda = decompor_demanda_para_rateio(bd_mes, bd_demanda_solver)

    def executar_rateio_proporcional_dupla_restricao(bd_demanda, bd_alternativas_validas,
                                                     bd_cap_recursos, bd_cap_ferramentas):
        alternativas = bd_alternativas_validas.copy()
        cap_restante_rec = bd_cap_recursos.set_index('ALOC_REC')['HOR_REC'].to_dict()
        cap_restante_fer = bd_cap_ferramentas.set_index('COD_FER_UNID')['HOR_FER'].to_dict()
        pendente = bd_demanda.set_index('ID_PROD_UNID_FAT')['NEC_PCS'].to_dict()
        pendentes_faixa = {nome: serie.to_dict() for nome, serie in faixas_demanda}
        producao, rastro, rastro_faixas = [], [], []

        chaves_prioridade = ['PRIOR_MATPAR', 'PRIOR_ROT', 'CLASS_ROT']
        niveis_prioridade = list(alternativas[chaves_prioridade].drop_duplicates()
                                 .sort_values(chaves_prioridade, kind='mergesort')
                                 .itertuples(index=False, name=None))
        for rodada, (prior_matpar, prior_rot, class_rot) in enumerate(niveis_prioridade, start=1):
            nivel_base = alternativas[
                alternativas['PRIOR_MATPAR'].eq(prior_matpar)
                & alternativas['PRIOR_ROT'].eq(prior_rot)
                & alternativas['CLASS_ROT'].eq(class_rot)
            ].copy()
            nivel_base['RODADA'] = rodada
            nivel_base['QTD_PENDENTE_ANTES'] = nivel_base['ID_PROD_UNID_FAT'].map(pendente).fillna(0.0)
            nivel_base = nivel_base[nivel_base['QTD_PENDENTE_ANTES'] > TOL].copy()
            if nivel_base.empty:
                continue
            ids = nivel_base['ID_PROD_UNID_FAT']
            qtd_alt = nivel_base.groupby('ID_PROD_UNID_FAT')['ALOC_REC'].transform('count')
            nivel_base['QTD_ALTERNATIVAS_MESMO_NIVEL'] = qtd_alt
            nivel_base['QTD_SOLICITADA'] = nivel_base['QTD_PENDENTE_ANTES'] / qtd_alt
            nivel_base['HR_SOLICITADA'] = nivel_base['QTD_SOLICITADA'] / nivel_base['PCS_HORA']
            nivel_base['CAP_RESTANTE_ALOC_REC_ANTES'] = nivel_base['ALOC_REC'].map(cap_restante_rec).fillna(0.0)
            nivel_base['CAP_RESTANTE_COD_FER_ANTES'] = nivel_base['COD_FER_UNID'].map(cap_restante_fer).fillna(0.0)
            nivel_base['HR_DEMANDA_TOTAL_ALOC_REC'] = nivel_base.groupby('ALOC_REC')['HR_SOLICITADA'].transform('sum')
            nivel_base['HR_DEMANDA_TOTAL_COD_FER'] = nivel_base.groupby('COD_FER_UNID')['HR_SOLICITADA'].transform('sum')
            nivel_base['FATOR_RATEIO_ALOC_REC'] = np.where(
                nivel_base['HR_DEMANDA_TOTAL_ALOC_REC'] > TOL,
                np.minimum(1.0, nivel_base['CAP_RESTANTE_ALOC_REC_ANTES'] /
                           nivel_base['HR_DEMANDA_TOTAL_ALOC_REC']), 0.0)
            nivel_base['FATOR_RATEIO_COD_FER'] = np.where(
                nivel_base['HR_DEMANDA_TOTAL_COD_FER'] > TOL,
                np.minimum(1.0, nivel_base['CAP_RESTANTE_COD_FER_ANTES'] /
                           nivel_base['HR_DEMANDA_TOTAL_COD_FER']), 0.0)
            nivel_base['CAP_DISPONIVEL_LINHA'] = nivel_base[
                ['CAP_RESTANTE_ALOC_REC_ANTES', 'CAP_RESTANTE_COD_FER_ANTES']].min(axis=1)
            qtd_prod_total = np.zeros(len(nivel_base), dtype=float)
            hr_prod_total = np.zeros(len(nivel_base), dtype=float)

            # Sem disputa fisica nesta classe, uma unica passagem aloca toda a NEC.
            # Assim a execucao normal nao paga o custo das 9 faixas.
            sem_conflito = bool((
                (nivel_base['HR_DEMANDA_TOTAL_ALOC_REC'] <= nivel_base['CAP_RESTANTE_ALOC_REC_ANTES'] + TOL)
                & (nivel_base['HR_DEMANDA_TOTAL_COD_FER'] <= nivel_base['CAP_RESTANTE_COD_FER_ANTES'] + TOL)
            ).all())
            faixas_nivel = [('TOTAL_SEM_DISPUTA', pd.Series(pendente))] if sem_conflito else faixas_demanda
            for nome_faixa, _serie_faixa in faixas_nivel:
                if sem_conflito:
                    qtd_faixa = ids.map(pendente).fillna(0.0)
                else:
                    qtd_faixa = ids.map(pendentes_faixa[nome_faixa]).fillna(0.0)
                nivel = nivel_base.copy()
                nivel['QTD_PENDENTE_FAIXA'] = qtd_faixa.to_numpy(float)
                nivel = nivel[nivel['QTD_PENDENTE_FAIXA'] > TOL].copy()
                if nivel.empty:
                    continue
                qtd_alt_faixa = nivel.groupby('ID_PROD_UNID_FAT')['ALOC_REC'].transform('count')
                nivel['QTD_SOLICITADA'] = nivel['QTD_PENDENTE_FAIXA'] / qtd_alt_faixa
                nivel['HR_SOLICITADA'] = nivel['QTD_SOLICITADA'] / nivel['PCS_HORA']
                nivel['CAP_RESTANTE_ALOC_REC_ANTES'] = nivel['ALOC_REC'].map(cap_restante_rec).fillna(0.0)
                nivel['CAP_RESTANTE_COD_FER_ANTES'] = nivel['COD_FER_UNID'].map(cap_restante_fer).fillna(0.0)
                demanda_rec = nivel.groupby('ALOC_REC')['HR_SOLICITADA'].transform('sum')
                demanda_fer = nivel.groupby('COD_FER_UNID')['HR_SOLICITADA'].transform('sum')
                fator_rec = np.where(demanda_rec > TOL,
                                     np.minimum(1.0, nivel['CAP_RESTANTE_ALOC_REC_ANTES'] / demanda_rec), 0.0)
                fator_fer = np.where(demanda_fer > TOL,
                                     np.minimum(1.0, nivel['CAP_RESTANTE_COD_FER_ANTES'] / demanda_fer), 0.0)
                nivel['FATOR_RATEIO'] = np.minimum(fator_rec, fator_fer).clip(0.0, 1.0)
                nivel['HR_PRODUZIR_SOLVER'] = nivel['HR_SOLICITADA'] * nivel['FATOR_RATEIO']
                nivel['QTD_PRODUZIR_SOLVER'] = nivel['HR_PRODUZIR_SOLVER'] * nivel['PCS_HORA']
                nivel['COMPONENTE_FAIXA_ALOCACAO'] = nome_faixa

                # Saldo REC/FER deve ser compartilhado entre faixas e rotas.
                for rec, consumo in nivel.groupby('ALOC_REC')['HR_PRODUZIR_SOLVER'].sum().items():
                    cap_restante_rec[rec] = max(0.0, cap_restante_rec.get(rec, 0.0) - float(consumo))
                for fer, consumo in nivel.groupby('COD_FER_UNID')['HR_PRODUZIR_SOLVER'].sum().items():
                    cap_restante_fer[fer] = max(0.0, cap_restante_fer.get(fer, 0.0) - float(consumo))
                for item, qtd in nivel.groupby('ID_PROD_UNID_FAT')['QTD_PRODUZIR_SOLVER'].sum().items():
                    pendente[item] = max(0.0, pendente[item] - float(qtd))
                    if not sem_conflito:
                        pendentes_faixa[nome_faixa][item] = max(0.0, pendentes_faixa[nome_faixa][item] - float(qtd))
                    else:
                        # Niveis sem conflito atendem integralmente todos os baldes
                        # ainda pendentes, respeitando o total fisico efetivamente alocado.
                        resto = float(qtd)
                        for outro_nome, saldos in pendentes_faixa.items():
                            usado = min(resto, saldos.get(item, 0.0))
                            saldos[item] = max(0.0, saldos.get(item, 0.0) - usado)
                            resto -= usado
                            if resto <= TOL:
                                break
                rastro_faixas.append(nivel[nivel['QTD_PRODUZIR_SOLVER'] > TOL].copy())
                # Index original da rota permite acumular faixas sem duplicar linha no plano.
                idx = nivel.index.to_numpy(dtype=int)
                qtd_prod_total[nivel_base.index.get_indexer(idx)] += nivel['QTD_PRODUZIR_SOLVER'].to_numpy(float)
                hr_prod_total[nivel_base.index.get_indexer(idx)] += nivel['HR_PRODUZIR_SOLVER'].to_numpy(float)

            nivel_base['QTD_PRODUZIR_SOLVER'] = qtd_prod_total
            nivel_base['HR_PRODUZIR_SOLVER'] = hr_prod_total
            nivel_base['QTD_PENDENTE_DEPOIS'] = nivel_base['ID_PROD_UNID_FAT'].map(pendente).fillna(0.0)
            nivel_base['FATOR_RATEIO'] = np.divide(hr_prod_total,
                                                   nivel_base['HR_SOLICITADA'].to_numpy(float),
                                                   out=np.zeros(len(nivel_base), dtype=float),
                                                   where=nivel_base['HR_SOLICITADA'].to_numpy(float) > TOL).clip(0, 1)
            rec_fim = nivel_base['ALOC_REC'].map(cap_restante_rec).fillna(0.0)
            fer_fim = nivel_base['COD_FER_UNID'].map(cap_restante_fer).fillna(0.0)
            cond_rec = (rec_fim <= TOL) & (nivel_base['QTD_PENDENTE_DEPOIS'] > TOL)
            cond_fer = (fer_fim <= TOL) & (nivel_base['QTD_PENDENTE_DEPOIS'] > TOL)
            nivel_base['MOTIVO_LIMITANTE'] = np.select(
                [cond_rec & cond_fer, cond_rec, cond_fer],
                ['ALOC_REC_E_COD_FER', 'ALOC_REC', 'COD_FER_UNID'], default='SEM_LIMITACAO')
            nivel_base['STATUS_DECISAO'] = np.select(
                [nivel_base['QTD_PRODUZIR_SOLVER'] <= TOL,
                 nivel_base['QTD_PENDENTE_DEPOIS'] <= TOL],
                ['NAO_ALOCADO_RATEIO_ZERO', 'ALOCADO_TOTAL_RODADA'],
                default='ALOCADO_PARCIAL_RATEIO')
            nivel_base['MOTIVO_CORTE'] = np.where(
                nivel_base['QTD_PENDENTE_DEPOIS'] <= TOL, 'SEM_CORTE',
                'CORTE_POR_' + nivel_base['MOTIVO_LIMITANTE'].astype(str))
            producao.append(nivel_base[nivel_base['QTD_PRODUZIR_SOLVER'] > TOL].copy())
            rastro.append(nivel_base)

        bd_producao = pd.concat(producao, ignore_index=True) if producao else pd.DataFrame()
        bd_rastro = pd.concat(rastro, ignore_index=True) if rastro else pd.DataFrame()
        bd_rastro_faixas = pd.concat(rastro_faixas, ignore_index=True) if rastro_faixas else pd.DataFrame()
        return bd_producao, bd_rastro, pendente, cap_restante_rec, cap_restante_fer, bd_rastro_faixas

    (bd_plano_producao, rastro_rateio, pendente_final, cap_restante_rec_final,
     cap_restante_fer_final, rastro_prioridade_componentes) = executar_rateio_proporcional_dupla_restricao(
        bd_demanda_solver, bd_alternativas_validas, bd_cap_recursos, bd_cap_ferramentas)


    # ============================================================
    # 04A. PONTE DE DIAGNOSTICO V66 -> MATRIZ DE CORTES
    # ============================================================
    # Esta etapa NAO corta NEC, NAO redistribui capacidade e NAO altera o plano.
    # Ela apenas traduz o estado final do Solver em um diagnostico seguro para a
    # futura Matriz: o que ficou pendente, quais rotas ainda teriam capacidade
    # simultanea de recurso + ferramenta e quais restricoes aparecem como
    # candidatas a corte quando nao existe mais alternativa livre.

    def gerar_diagnostico_matriz_ponte(
        bd_demanda,
        bd_alternativas,
        bd_rastro,
        pendente,
        cap_restante_rec,
        cap_restante_fer,
    ):
        cols_resumo = [
            'MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD',
            'NEC_PCS','QTD_PENDENTE_FINAL','RODADA_FINAL_ITEM','PRIOR_MATPAR_FINAL','PRIOR_ROT_FINAL',
            'QTD_ROTAS_VALIDAS','QTD_ROTAS_ULTIMA_RODADA','QTD_ROTAS_COM_CAPACIDADE_FINAL',
            'QTD_FALTA_ULTIMA_RODADA','DIF_PENDENCIA_VS_ULTIMA_RODADA',
            'STATUS_DIAGNOSTICO','TIPO_RESTRICAO_CANDIDATA','RESTRICAO_CANDIDATA',
            'CORTE_HR_CANDIDATO','QTD_CORTE_PCS_CANDIDATO','RESTRICOES_BLOQUEANTES'
        ]
        cols_rotas = [
            'MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD',
            'PRIOR_MATPAR','PRIOR_ROT','CLASS_ROT','ALOC_REC','COD_FER_UNID','PCS_HORA',
            'QTD_PENDENTE_FINAL','CAP_RESTANTE_ALOC_REC_FINAL','CAP_RESTANTE_COD_FER_FINAL',
            'HR_DISPONIVEL_FINAL','QTD_POSSIVEL_FINAL','ALTERNATIVA_COM_CAPACIDADE_FINAL',
            'RODADA_FINAL_ITEM','FOI_ULTIMA_RODADA','QTD_SOLICITADA_ULTIMA_RODADA',
            'QTD_PRODUZIR_ULTIMA_RODADA','QTD_FALTA_ROTA','HR_FALTA_ROTA',
            'MOTIVO_LIMITANTE_ULTIMA_RODADA','BLOQUEIOS_FINAIS_ROTA'
        ]

        pend_df = pd.DataFrame([
            {'ID_PROD_UNID_FAT': k, 'QTD_PENDENTE_FINAL': float(v)}
            for k, v in pendente.items() if float(v) > TOL
        ])
        if pend_df.empty:
            return pd.DataFrame(columns=cols_resumo), pd.DataFrame(columns=cols_rotas)

        meta_cols = [
            c for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS']
            if c in bd_demanda.columns
        ]
        meta = bd_demanda[meta_cols].drop_duplicates('ID_PROD_UNID_FAT')
        pend_df = pend_df.merge(meta, on='ID_PROD_UNID_FAT', how='left')

        alts = bd_alternativas[bd_alternativas['ID_PROD_UNID_FAT'].isin(pend_df['ID_PROD_UNID_FAT'])].copy()
        if alts.empty:
            resumo = pend_df.copy()
            resumo['RODADA_FINAL_ITEM'] = np.nan
            resumo['PRIOR_MATPAR_FINAL'] = np.nan
            resumo['PRIOR_ROT_FINAL'] = np.nan
            resumo['QTD_ROTAS_VALIDAS'] = 0
            resumo['QTD_ROTAS_ULTIMA_RODADA'] = 0
            resumo['QTD_ROTAS_COM_CAPACIDADE_FINAL'] = 0
            resumo['QTD_FALTA_ULTIMA_RODADA'] = resumo['QTD_PENDENTE_FINAL']
            resumo['DIF_PENDENCIA_VS_ULTIMA_RODADA'] = 0.0
            resumo['STATUS_DIAGNOSTICO'] = 'SEM_ALTERNATIVA_VALIDA'
            resumo['TIPO_RESTRICAO_CANDIDATA'] = ''
            resumo['RESTRICAO_CANDIDATA'] = ''
            resumo['CORTE_HR_CANDIDATO'] = np.nan
            resumo['QTD_CORTE_PCS_CANDIDATO'] = np.nan
            resumo['RESTRICOES_BLOQUEANTES'] = ''
            return resumo[[c for c in cols_resumo if c in resumo.columns]], pd.DataFrame(columns=cols_rotas)

        for c in ['PCS_HORA','PRIOR_MATPAR','PRIOR_ROT']:
            if c in alts.columns:
                alts[c] = to_num(alts[c])
        alts['QTD_PENDENTE_FINAL'] = alts['ID_PROD_UNID_FAT'].map(
            pend_df.set_index('ID_PROD_UNID_FAT')['QTD_PENDENTE_FINAL']
        ).fillna(0.0)
        alts['CAP_RESTANTE_ALOC_REC_FINAL'] = alts['ALOC_REC'].map(cap_restante_rec).fillna(0.0)
        alts['CAP_RESTANTE_COD_FER_FINAL'] = alts['COD_FER_UNID'].map(cap_restante_fer).fillna(0.0)
        alts['HR_DISPONIVEL_FINAL'] = alts[['CAP_RESTANTE_ALOC_REC_FINAL','CAP_RESTANTE_COD_FER_FINAL']].min(axis=1).clip(lower=0.0)
        alts['QTD_POSSIVEL_FINAL'] = alts['HR_DISPONIVEL_FINAL'] * alts['PCS_HORA']
        alts['ALTERNATIVA_COM_CAPACIDADE_FINAL'] = alts['QTD_POSSIVEL_FINAL'] > TOL

        def _bloqueios_rota(row):
            bloqueios = []
            if float(row['CAP_RESTANTE_ALOC_REC_FINAL']) <= TOL:
                bloqueios.append('REC|' + str(row['ALOC_REC']))
            if float(row['CAP_RESTANTE_COD_FER_FINAL']) <= TOL:
                bloqueios.append('FER|' + str(row['COD_FER_UNID']))
            return bloqueios

        alts['_BLOQUEIOS_SET'] = alts.apply(_bloqueios_rota, axis=1)
        alts['BLOQUEIOS_FINAIS_ROTA'] = alts['_BLOQUEIOS_SET'].apply(lambda x: ';'.join(x))

        rastro = bd_rastro.copy()
        if rastro.empty:
            rodada_final = pd.Series(dtype=float)
            ultimo = pd.DataFrame()
        else:
            rastro = rastro[rastro['ID_PROD_UNID_FAT'].isin(pend_df['ID_PROD_UNID_FAT'])].copy()
            rodada_final = rastro.groupby('ID_PROD_UNID_FAT')['RODADA'].max()
            rastro['_RODADA_FINAL_ITEM'] = rastro['ID_PROD_UNID_FAT'].map(rodada_final)
            ultimo = rastro[rastro['RODADA'].eq(rastro['_RODADA_FINAL_ITEM'])].copy()
            ultimo['QTD_FALTA_ROTA'] = (
                to_num(ultimo['QTD_SOLICITADA']) - to_num(ultimo['QTD_PRODUZIR_SOLVER'])
            ).clip(lower=0.0)
            ultimo['HR_FALTA_ROTA'] = np.divide(
                ultimo['QTD_FALTA_ROTA'],
                to_num(ultimo['PCS_HORA']),
                out=np.zeros(len(ultimo), dtype=float),
                where=to_num(ultimo['PCS_HORA']).to_numpy() > TOL,
            )

        chave_alt = [
            c for c in ['ID_PROD_UNID_FAT','PRIOR_MATPAR','PRIOR_ROT','ALOC_REC','COD_FER_UNID']
            if c in alts.columns
        ]
        if not ultimo.empty:
            ult_cols = chave_alt + [
                'RODADA','QTD_SOLICITADA','QTD_PRODUZIR_SOLVER','QTD_FALTA_ROTA','HR_FALTA_ROTA','MOTIVO_LIMITANTE'
            ]
            ult = ultimo[[c for c in ult_cols if c in ultimo.columns]].copy()
            ren = {
                'RODADA':'RODADA_FINAL_ITEM',
                'QTD_SOLICITADA':'QTD_SOLICITADA_ULTIMA_RODADA',
                'QTD_PRODUZIR_SOLVER':'QTD_PRODUZIR_ULTIMA_RODADA',
                'MOTIVO_LIMITANTE':'MOTIVO_LIMITANTE_ULTIMA_RODADA',
            }
            ult = ult.rename(columns=ren)
            alts = alts.merge(ult, on=chave_alt, how='left')
        else:
            alts['RODADA_FINAL_ITEM'] = np.nan
            alts['QTD_SOLICITADA_ULTIMA_RODADA'] = 0.0
            alts['QTD_PRODUZIR_ULTIMA_RODADA'] = 0.0
            alts['QTD_FALTA_ROTA'] = 0.0
            alts['HR_FALTA_ROTA'] = 0.0
            alts['MOTIVO_LIMITANTE_ULTIMA_RODADA'] = ''

        for c in ['QTD_SOLICITADA_ULTIMA_RODADA','QTD_PRODUZIR_ULTIMA_RODADA','QTD_FALTA_ROTA','HR_FALTA_ROTA']:
            if c not in alts.columns:
                alts[c] = 0.0
            alts[c] = to_num(alts[c])
        alts['FOI_ULTIMA_RODADA'] = alts['RODADA_FINAL_ITEM'].notna()

        # Reaplica metadados do item de forma controlada quando a tabela de alternativas nao os possui.
        meta_map = pend_df.set_index('ID_PROD_UNID_FAT')
        for c in ['MES_REF','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD']:
            if c not in alts.columns and c in meta_map.columns:
                alts[c] = alts['ID_PROD_UNID_FAT'].map(meta_map[c])

        resumos = []
        for item, grp in alts.groupby('ID_PROD_UNID_FAT', sort=False):
            p = pend_df.loc[pend_df['ID_PROD_UNID_FAT'].eq(item)].iloc[0]
            grp_ult = grp[grp['FOI_ULTIMA_RODADA'] & (grp['QTD_FALTA_ROTA'] > TOL)].copy()
            qtd_falta_ult = float(grp_ult['QTD_FALTA_ROTA'].sum()) if not grp_ult.empty else 0.0
            qtd_pend = float(p['QTD_PENDENTE_FINAL'])
            rotas_cap = int(grp['ALTERNATIVA_COM_CAPACIDADE_FINAL'].sum())

            # Uma restricao so pode ser chamada de candidata objetiva se ela bloqueia
            # TODAS as rotas que ainda carregam falta na ultima rodada.
            comum = None
            if not grp_ult.empty:
                sets = [set(x) for x in grp_ult['_BLOQUEIOS_SET']]
                comum = set.intersection(*sets) if sets else set()
            else:
                comum = set()

            todos_bloqueios = sorted(set().union(*[set(x) for x in grp['_BLOQUEIOS_SET']])) if len(grp) else []

            tipo = ''
            restr = ''
            corte_hr = np.nan
            qtd_corte = np.nan

            if rotas_cap > 0:
                status = 'REOTIMIZAR_ANTES_DE_CORTAR'
            elif grp_ult.empty:
                status = 'SEM_RASTRO_FINAL_PARA_PENDENCIA'
            elif len(comum) == 1:
                restr_key = next(iter(comum))
                tipo, restr = restr_key.split('|', 1)
                corte_hr = float(grp_ult['HR_FALTA_ROTA'].sum())
                qtd_corte = qtd_pend
                status = 'CORTE_CANDIDATO_OBJETIVO'
            elif len(comum) == 2 and any(x.startswith('REC|') for x in comum) and any(x.startswith('FER|') for x in comum):
                # Recurso e ferramenta podem estar simultaneamente esgotados na mesma decisao.
                # O volume de demanda deve ser cortado UMA unica vez; esse mesmo corte libera
                # horas nas duas restricoes e nao autoriza dois cortes independentes.
                tipo = 'REC_E_FER'
                restr = ';'.join(sorted(comum))
                corte_hr = float(grp_ult['HR_FALTA_ROTA'].sum())
                qtd_corte = qtd_pend
                status = 'CORTE_CANDIDATO_BLOQUEIO_DUPLO'
            else:
                status = 'AMBIGUO_MULTIPLAS_RESTRICOES'

            rodada_item = grp['RODADA_FINAL_ITEM'].dropna()
            rodada_item = float(rodada_item.iloc[0]) if not rodada_item.empty else np.nan
            pm = grp_ult['PRIOR_MATPAR'].iloc[0] if not grp_ult.empty and 'PRIOR_MATPAR' in grp_ult.columns else np.nan
            pr = grp_ult['PRIOR_ROT'].iloc[0] if not grp_ult.empty and 'PRIOR_ROT' in grp_ult.columns else np.nan

            row = {c: p[c] for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS'] if c in p.index}
            row.update({
                'QTD_PENDENTE_FINAL': qtd_pend,
                'RODADA_FINAL_ITEM': rodada_item,
                'PRIOR_MATPAR_FINAL': pm,
                'PRIOR_ROT_FINAL': pr,
                'QTD_ROTAS_VALIDAS': int(len(grp)),
                'QTD_ROTAS_ULTIMA_RODADA': int(grp['FOI_ULTIMA_RODADA'].sum()),
                'QTD_ROTAS_COM_CAPACIDADE_FINAL': rotas_cap,
                'QTD_FALTA_ULTIMA_RODADA': qtd_falta_ult,
                'DIF_PENDENCIA_VS_ULTIMA_RODADA': qtd_pend - qtd_falta_ult,
                'STATUS_DIAGNOSTICO': status,
                'TIPO_RESTRICAO_CANDIDATA': tipo,
                'RESTRICAO_CANDIDATA': restr,
                'CORTE_HR_CANDIDATO': corte_hr,
                'QTD_CORTE_PCS_CANDIDATO': qtd_corte,
                'RESTRICOES_BLOQUEANTES': ';'.join(todos_bloqueios),
            })
            resumos.append(row)

        resumo = pd.DataFrame(resumos)
        rotas = alts[[c for c in cols_rotas if c in alts.columns]].copy()
        resumo = resumo[[c for c in cols_resumo if c in resumo.columns]].copy()
        return resumo, rotas

    diagnostico_matriz, diagnostico_matriz_rotas = gerar_diagnostico_matriz_ponte(
        bd_demanda_solver,
        bd_alternativas_validas,
        rastro_rateio,
        pendente_final,
        cap_restante_rec_final,
        cap_restante_fer_final,
    )

    # Validacao interna: a ponte e somente observadora; nao pode alterar o plano.
    if not diagnostico_matriz.empty:
        _dif_diag = diagnostico_matriz['DIF_PENDENCIA_VS_ULTIMA_RODADA'].abs()
        diagnostico_matriz['VALIDA_PENDENCIA_ULTIMA_RODADA'] = _dif_diag <= max(TOL * 10, 1e-6)


    # ============================================================
    # 05. SAÍDAS DE NEGÓCIO E RASTREABILIDADE
    # ============================================================


    cols_plano = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS", "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA", "LINHA_PROD", "FAMILIA_PROD", "MO", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "PESO_PROD_KG", "VOL_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO", "FATOR_FINAL_VIAVEL_ORIGEM", "FATOR_AJUSTE_SOBRE_SOLVER", "QTD_PRODUZIR_FINAL_VIAVEL_EST", "QTD_REDUCAO_POS_PAI_FILHO", "HR_LIBERADA_EST"]
    campos_produto_saida = ["COD_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]
    campos_produto_saida = [c for c in campos_produto_saida if c in bd_mes.columns]
    if campos_produto_saida and "COD_PROD" in campos_produto_saida:
        cadastro_produto_saida = bd_mes[campos_produto_saida].drop_duplicates("COD_PROD")
        for c in ["LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG"]:
            if c in bd_plano_producao.columns:
                bd_plano_producao = bd_plano_producao.drop(columns=[c])
        bd_plano_producao = bd_plano_producao.merge(cadastro_produto_saida, on="COD_PROD", how="left")

    bd_plano_producao["PESO_PROD_KG"] = to_num(bd_plano_producao["PESO_PROD_KG"]) if "PESO_PROD_KG" in bd_plano_producao.columns else 0
    bd_plano_producao["VOL_PRODUZIR_SOLVER"] = bd_plano_producao["QTD_PRODUZIR_SOLVER"] * bd_plano_producao["PESO_PROD_KG"] if "QTD_PRODUZIR_SOLVER" in bd_plano_producao.columns else 0
    plano_producao = bd_plano_producao[[c for c in cols_plano if c in bd_plano_producao.columns]].copy()

    atendido_item = plano_producao.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(QTD_ATENDIDA_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "QTD_ATENDIDA_SOLVER"])
    nao_atend_item = bd_demanda_solver.merge(atendido_item, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
    nao_atend_item["QTD_ATENDIDA_SOLVER"] = nao_atend_item["QTD_ATENDIDA_SOLVER"].fillna(0)
    nao_atend_item["QTD_NAO_ATEND_SOLVER"] = (nao_atend_item["NEC_PCS"] - nao_atend_item["QTD_ATENDIDA_SOLVER"]).clip(lower=0)
    nao_atend_item["PERC_ATENDIDO_SOLVER"] = np.where(nao_atend_item["NEC_PCS"] > 0, nao_atend_item["QTD_ATENDIDA_SOLVER"] / nao_atend_item["NEC_PCS"], 0)
    nao_atend_item["PERC_NAO_ATENDIDO_SOLVER", "QTD_FINAL_VIAVEL_PAI_FILHO", "QTD_AJUSTE_PAI_FILHO", "FATOR_FINAL_VIAVEL_PAI_FILHO", "MOTIVO_AJUSTE_PAI_FILHO"] = 1 - nao_atend_item["PERC_ATENDIDO_SOLVER"]
    cols_nao_atend = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "NEC_PCS_ORIGINAL", "NEC_ESTRUTURA_PAI_PCS", "NEC_DIRETA_BRUTA_PCS", "NEC_DIRETA_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_COMP_PCS", "NEC_PCS_CONSOLIDADA", "NEC_PCS", "QTD_ATENDIDA_SOLVER", "QTD_NAO_ATEND_SOLVER", "PERC_ATENDIDO_SOLVER", "PERC_NAO_ATENDIDO_SOLVER"]
    nao_atend_item = nao_atend_item[[c for c in cols_nao_atend if c in nao_atend_item.columns]].copy()

    # Inicializa os campos pós-solver. O fechamento estrutural final é executado depois das auditorias base.
    ajuste_pai_filho = pd.DataFrame()
    resumo_pai_filho = pd.DataFrame()
    capacidade_liberada_pf = pd.DataFrame()
    nao_atend_item["QTD_FINAL_VIAVEL_PAI_FILHO"] = nao_atend_item["QTD_ATENDIDA_SOLVER"]
    nao_atend_item["QTD_AJUSTE_PAI_FILHO"] = 0.0
    nao_atend_item["FATOR_FINAL_VIAVEL_PAI_FILHO"] = 1.0
    nao_atend_item["MOTIVO_AJUSTE_PAI_FILHO"] = "SEM_AJUSTE_POS_SOLVER"
    plano_producao["FATOR_FINAL_VIAVEL_ORIGEM"] = 1.0
    plano_producao["FATOR_AJUSTE_SOBRE_SOLVER"] = 1.0
    plano_producao["QTD_PRODUZIR_FINAL_VIAVEL_EST"] = plano_producao["QTD_PRODUZIR_SOLVER"]
    plano_producao["QTD_REDUCAO_POS_PAI_FILHO"] = 0.0
    plano_producao["HR_LIBERADA_EST"] = 0.0

    # Ajuste operacional — blindagem contra duplicidade na saída do plano.
    # A origem do problema era o merge do plano com fatores pai/filho quando o resumo pai/filho tinha mais de uma linha para a mesma chave MES_REF + ID_PROD_UNID_FAT.
    # O solver/rastro não estava estourado; a duplicidade ocorria na montagem/exportação do plano e contaminava os resumos.
    CHAVE_PLANO_UNICA = [c for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','RODADA','PRIOR_MATPAR','PRIOR_ROT','ALOC_REC','COD_FER_UNID','QTD_PRODUZIR_SOLVER','HR_PRODUZIR_SOLVER'] if c in plano_producao.columns]
    QTD_LINHAS_PLANO_ANTES_DEDUP = len(plano_producao)
    plano_producao = plano_producao.drop_duplicates(subset=CHAVE_PLANO_UNICA, keep='first').copy()
    QTD_LINHAS_PLANO_DEPOIS_DEDUP = len(plano_producao)

    prod_recurso = plano_producao.groupby("ALOC_REC", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ALOC_REC", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"])
    resumo_recursos = bd_cap_recursos.merge(prod_recurso, on="ALOC_REC", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0})
    resumo_recursos["OCUPACAO_RECURSO_PCT"] = np.where(resumo_recursos["HOR_REC"] > 0, resumo_recursos["HR_PRODUZIR_SOLVER"] / resumo_recursos["HOR_REC"], 0)
    resumo_recursos["HR_RECURSO_OCIOSA"] = resumo_recursos["HOR_REC"] - resumo_recursos["HR_PRODUZIR_SOLVER"]
    resumo_recursos["ESTOURO_HR_RECURSO"] = np.where(resumo_recursos["HR_PRODUZIR_SOLVER"] > resumo_recursos["HOR_REC"] + TOL, resumo_recursos["HR_PRODUZIR_SOLVER"] - resumo_recursos["HOR_REC"], 0)
    resumo_recursos = resumo_recursos[["ALOC_REC", "HOR_REC", "HR_PRODUZIR_SOLVER", "OCUPACAO_RECURSO_PCT", "HR_RECURSO_OCIOSA", "ESTOURO_HR_RECURSO", "QTD_PRODUZIR_SOLVER", "QTD_ITENS"]].sort_values(["OCUPACAO_RECURSO_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

    prod_ferramenta = plano_producao.groupby("COD_FER_UNID", as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum"), QTD_ITENS=("ID_PROD_UNID_FAT", "nunique"), QTD_RECURSOS=("ALOC_REC", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["COD_FER_UNID", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"])
    resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on="COD_FER_UNID", how="left").fillna({"HR_PRODUZIR_SOLVER": 0, "QTD_PRODUZIR_SOLVER": 0, "QTD_ITENS": 0, "QTD_RECURSOS": 0})
    resumo_ferramentas["OCUPACAO_FER_PCT"] = np.where(resumo_ferramentas["HOR_FER"] > 0, resumo_ferramentas["HR_PRODUZIR_SOLVER"] / resumo_ferramentas["HOR_FER"], 0)
    resumo_ferramentas["HR_FER_OCIOSA"] = resumo_ferramentas["HOR_FER"] - resumo_ferramentas["HR_PRODUZIR_SOLVER"]
    resumo_ferramentas["ESTOURO_HR_FER"] = np.where(resumo_ferramentas["HR_PRODUZIR_SOLVER"] > resumo_ferramentas["HOR_FER"] + TOL, resumo_ferramentas["HR_PRODUZIR_SOLVER"] - resumo_ferramentas["HOR_FER"], 0)
    resumo_ferramentas = resumo_ferramentas[["COD_FER_UNID", "HOR_FER", "HR_PRODUZIR_SOLVER", "OCUPACAO_FER_PCT", "HR_FER_OCIOSA", "ESTOURO_HR_FER", "QTD_PRODUZIR_SOLVER", "QTD_ITENS", "QTD_RECURSOS"]].sort_values(["OCUPACAO_FER_PCT", "HR_PRODUZIR_SOLVER"], ascending=[False, False])

    cols_auditoria = ["MES_REF", "ALOC_REC", "COD_FER_UNID", "HOR_REC", "HOR_FER", "HOR_CAP"]
    auditoria_capacidade = bd_alternativas_item[[c for c in cols_auditoria if c in bd_alternativas_item.columns]].drop_duplicates().sort_values(["ALOC_REC", "COD_FER_UNID"]).copy()

    cols_alt = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "ALTERNATIVA_VALIDA", "MOTIVO_INVALIDA"]
    alternativas_item = bd_alternativas_item[[c for c in cols_alt if c in bd_alternativas_item.columns]].sort_values(["ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC"]).copy()

    cols_rastro = ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "RODADA", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "QTD_PENDENTE_ANTES", "QTD_SOLICITADA", "HR_SOLICITADA", "HR_DEMANDA_TOTAL_ALOC_REC", "HR_DEMANDA_TOTAL_COD_FER", "CAP_RESTANTE_ALOC_REC_ANTES", "CAP_RESTANTE_COD_FER_ANTES", "CAP_DISPONIVEL_LINHA", "FATOR_RATEIO_ALOC_REC", "FATOR_RATEIO_COD_FER", "FATOR_RATEIO", "MOTIVO_LIMITANTE", "QTD_PRODUZIR_SOLVER", "HR_PRODUZIR_SOLVER", "QTD_PENDENTE_DEPOIS", "MOTIVO_CORTE", "STATUS_DECISAO"]
    rastro_rateio = rastro_rateio[[c for c in cols_rastro if c in rastro_rateio.columns]].copy() if not rastro_rateio.empty else pd.DataFrame(columns=cols_rastro)

    qtd_alt_audit = bd_alternativas_item.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_AUDITADAS=("ALOC_REC", "count"))
    qtd_alt_valid = bd_alternativas_validas.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_VALIDAS=("ALOC_REC", "count"))
    qtd_alt_usadas = plano_producao.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_ALTERNATIVAS_USADAS=("ALOC_REC", "count"), QTD_ALOC_REC_USADOS=("ALOC_REC", "nunique"), QTD_COD_FER_USADOS=("COD_FER_UNID", "nunique")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS"])
    qtd_rodadas = rastro_rateio.groupby("ID_PROD_UNID_FAT", as_index=False).agg(QTD_RODADAS_TENTADAS=("RODADA", "nunique"), QTD_ALOC_REC_TENTADOS=("ALOC_REC", "nunique"), QTD_COD_FER_TENTADOS=("COD_FER_UNID", "nunique")) if not rastro_rateio.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"])

    diagnostico_item = nao_atend_item.copy()
    for df_merge in [qtd_alt_audit, qtd_alt_valid, qtd_alt_usadas, qtd_rodadas]:
        diagnostico_item = diagnostico_item.merge(df_merge, on="ID_PROD_UNID_FAT", how="left")
    for c in ["QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS", "QTD_ALTERNATIVAS_USADAS", "QTD_ALOC_REC_USADOS", "QTD_COD_FER_USADOS", "QTD_RODADAS_TENTADAS", "QTD_ALOC_REC_TENTADOS", "QTD_COD_FER_TENTADOS"]:
        if c in diagnostico_item.columns:
            diagnostico_item[c] = diagnostico_item[c].fillna(0).astype(int)
    diagnostico_item["MOTIVO_SALDO_FINAL"] = np.select([diagnostico_item["QTD_NAO_ATEND_SOLVER"] <= TOL, diagnostico_item.get("QTD_ALTERNATIVAS_AUDITADAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_VALIDAS", 0).eq(0), diagnostico_item.get("QTD_ALTERNATIVAS_USADAS", 0).eq(0)], ["ATENDIDO_TOTAL", "SEM_ALTERNATIVA_CADASTRADA", "SEM_ALTERNATIVA_VALIDA", "SEM_ALOCACAO_COM_ALTERNATIVA_VALIDA"], default="SALDO_APOS_RATEIO_E_LIMITES_DE_CAPACIDADE")

    uso_alt = plano_producao.groupby(["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], as_index=False).agg(HR_PRODUZIR_SOLVER=("HR_PRODUZIR_SOLVER", "sum"), QTD_PRODUZIR_SOLVER=("QTD_PRODUZIR_SOLVER", "sum")) if not plano_producao.empty else pd.DataFrame(columns=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT", "HR_PRODUZIR_SOLVER", "QTD_PRODUZIR_SOLVER"])
    diagnostico_alternativas = alternativas_item.merge(uso_alt, on=["ID_PROD_UNID_FAT", "ALOC_REC", "COD_FER_UNID", "PRIOR_MATPAR", "PRIOR_ROT"], how="left")
    diagnostico_alternativas["HR_PRODUZIR_SOLVER"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"].fillna(0)
    diagnostico_alternativas["QTD_PRODUZIR_SOLVER"] = diagnostico_alternativas["QTD_PRODUZIR_SOLVER"].fillna(0)
    diagnostico_alternativas["ALTERNATIVA_USADA"] = diagnostico_alternativas["HR_PRODUZIR_SOLVER"] > TOL


    # ============================================================
    # 06. REALOCAÇÃO DA CAPACIDADE LIBERADA - rotina operacional
    # ============================================================

    # Para realocar em outro item, o modelo valida a capacidade da máquina e também a capacidade do molde/ferramenta destino (COD_FER_UNID).
    # A realocação não fica mais presa ao mesmo molde que gerou a liberação.

    # ---------- rotina operacional realocacao ----------
    def calcular_realocacao_realoc(nao_atend_item, alternativas_item, plano_producao, capacidade_liberada_pf):
        cols_log = [
            'RODADA_REALOC','MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD',
            'PRIOR_MATPAR','PRIOR_ROT','CLASS_ROT','ALOC_REC','COD_FER_UNID','PCS_HORA',
            'QTD_GAP_ANTES_REALOC','HR_MAQ_LIBERADA_DISP_ANTES','HR_FER_DESTINO_DISP_ANTES','HR_DISPONIVEL_REALOC',
            'QTD_POSSIVEL_REALOCAR','QTD_REALOCADA','HR_CONSUMIDA_REALOC','HR_MAQ_LIBERADA_SALDO_DEPOIS',
            'HR_FER_DESTINO_SALDO_DEPOIS','QTD_GAP_DEPOIS_REALOC','STATUS_REALOCACAO','MOTIVO_NAO_REALOCACAO','MOTIVO_AJUSTE_PAI_FILHO'
        ]
        if capacidade_liberada_pf.empty:
            return pd.DataFrame(columns=cols_log), pd.DataFrame(), pd.DataFrame()
        na = nao_atend_item.copy(); alts = alternativas_item.copy(); plano = plano_producao.copy(); cap = capacidade_liberada_pf.copy()
        for df in [na, alts, plano, cap]:
            if 'MES_REF' in df.columns:
                df['MES_REF'] = pd.to_datetime(df['MES_REF'], errors='coerce')
        for c in ['NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_AJUSTE_PAI_FILHO','QTD_ATENDIDA_SOLVER']:
            if c not in na.columns: na[c] = 0.0
            na[c] = to_num(na[c])
        na['QTD_GAP_APOS_PAI_FILHO'] = (na['NEC_PCS'] - na['QTD_FINAL_VIAVEL_PAI_FILHO']).clip(lower=0)
        # Mantem bloqueio: itens cortados por pai/filho nao recebem realocacao nesta rodada, para nao produzir sem componente.
        na['BLOQUEADO_AJUSTE_PAI_FILHO'] = na['QTD_AJUSTE_PAI_FILHO'] > TOL

        if 'ALTERNATIVA_VALIDA' in alts.columns:
            alts['_ALT_VALIDA'] = is_valid_alt(alts['ALTERNATIVA_VALIDA'])
        else:
            alts['_ALT_VALIDA'] = True
        for c in ['PCS_HORA','PRIOR_MATPAR','PRIOR_ROT','HOR_FER','HOR_REC']:
            if c in alts.columns: alts[c] = to_num(alts[c])
        for c in ['HR_PRODUZIR_SOLVER','HOR_FER','HOR_REC']:
            if c in plano.columns: plano[c] = to_num(plano[c])
        for c in ['HR_LIBERADA_EST']:
            cap[c] = to_num(cap[c])

        # Capacidade liberada volta para a maquina, nao fica presa ao molde original.
        pool_maq = cap.groupby(['MES_REF','ALOC_REC'], as_index=False).agg(
            HR_MAQ_LIBERADA_TOTAL=('HR_LIBERADA_EST','sum'),
            QTD_ORIGENS_LIBERARAM=('ID_PROD_UNID_FAT','nunique'),
            QTD_LINHAS_LIBERARAM=('ID_PROD_UNID_FAT','count')
        )

        # Capacidade disponivel por molde destino depois do ajuste pai/filho.
        # HOR_FER total mensal por ferramenta.
        fer_cap_1 = alts.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HOR_FER=('HOR_FER','max')) if 'HOR_FER' in alts.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HOR_FER'])
        fer_cap_2 = plano.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HOR_FER_PLANO=('HOR_FER','max')) if 'HOR_FER' in plano.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HOR_FER_PLANO'])
        fer_cap = fer_cap_1.merge(fer_cap_2, on=['MES_REF','COD_FER_UNID'], how='outer')
        fer_cap['HOR_FER'] = fer_cap.get('HOR_FER', 0).fillna(0)
        if 'HOR_FER_PLANO' in fer_cap.columns:
            fer_cap['HOR_FER'] = np.maximum(fer_cap['HOR_FER'], fer_cap['HOR_FER_PLANO'].fillna(0))
        fer_cap = fer_cap[['MES_REF','COD_FER_UNID','HOR_FER']]

        fer_usado_solver = plano.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HR_USADA_SOLVER=('HR_PRODUZIR_SOLVER','sum')) if 'HR_PRODUZIR_SOLVER' in plano.columns else pd.DataFrame(columns=['MES_REF','COD_FER_UNID','HR_USADA_SOLVER'])
        fer_liberada = cap.groupby(['MES_REF','COD_FER_UNID'], as_index=False).agg(HR_LIBERADA_FER=('HR_LIBERADA_EST','sum'))
        fer_pool = fer_cap.merge(fer_usado_solver, on=['MES_REF','COD_FER_UNID'], how='left').merge(fer_liberada, on=['MES_REF','COD_FER_UNID'], how='left').fillna({'HR_USADA_SOLVER':0,'HR_LIBERADA_FER':0})
        fer_pool['HR_FER_USADA_POS_PAI_FILHO'] = (fer_pool['HR_USADA_SOLVER'] - fer_pool['HR_LIBERADA_FER']).clip(lower=0)
        fer_pool['HR_FER_DISPONIVEL_POS_PAI_FILHO'] = (fer_pool['HOR_FER'] - fer_pool['HR_FER_USADA_POS_PAI_FILHO']).clip(lower=0)

        # Candidatos: itens com gap na mesma maquina onde houve liberacao; ferramenta destino tem que ter capacidade.
        item_cols = ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_GAP_APOS_PAI_FILHO','BLOQUEADO_AJUSTE_PAI_FILHO','MOTIVO_AJUSTE_PAI_FILHO']
        cand = alts.merge(na[item_cols], on=['MES_REF','ID_PROD_UNID_FAT'], how='inner', suffixes=('', '_ITEM'))
        for c in ['COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD']:
            ci = c + '_ITEM'
            if ci in cand.columns:
                cand[c] = cand[c].fillna(cand[ci]) if c in cand.columns else cand[ci]
        cand = cand[(cand['_ALT_VALIDA']) & (cand['PCS_HORA'] > TOL) & (cand['QTD_GAP_APOS_PAI_FILHO'] > TOL) & (~cand['BLOQUEADO_AJUSTE_PAI_FILHO'])].copy()
        cand = cand.merge(pool_maq[['MES_REF','ALOC_REC','HR_MAQ_LIBERADA_TOTAL']], on=['MES_REF','ALOC_REC'], how='inner')
        cand = cand.merge(fer_pool[['MES_REF','COD_FER_UNID','HR_FER_DISPONIVEL_POS_PAI_FILHO']], on=['MES_REF','COD_FER_UNID'], how='left')
        cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'] = cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'].fillna(0)
        cand = cand[cand['HR_FER_DISPONIVEL_POS_PAI_FILHO'] > TOL].copy()

        gap_rem = {tuple(k): float(v) for k, v in na.set_index(['MES_REF','ID_PROD_UNID_FAT'])['QTD_GAP_APOS_PAI_FILHO'].items()}
        bloqueado = {tuple(k): bool(v) for k, v in na.set_index(['MES_REF','ID_PROD_UNID_FAT'])['BLOQUEADO_AJUSTE_PAI_FILHO'].items()}
        hr_maq_rem = {tuple(row[['MES_REF','ALOC_REC']]): float(row['HR_MAQ_LIBERADA_TOTAL']) for _, row in pool_maq.iterrows()}
        hr_fer_rem = {tuple(row[['MES_REF','COD_FER_UNID']]): float(row['HR_FER_DISPONIVEL_POS_PAI_FILHO']) for _, row in fer_pool.iterrows()}

        logs = []
        if not cand.empty:
            cand['_GAP_SORT'] = cand['QTD_GAP_APOS_PAI_FILHO']
            cand = cand.sort_values(['MES_REF','ALOC_REC','PRIOR_MATPAR','PRIOR_ROT','_GAP_SORT','ID_PROD_UNID_FAT'], ascending=[True, True, True, True, False, True])
            rodada = 1
            for _, row in cand.iterrows():
                item_key = (row['MES_REF'], row['ID_PROD_UNID_FAT'])
                maq_key = (row['MES_REF'], row['ALOC_REC'])
                fer_key = (row['MES_REF'], row['COD_FER_UNID'])
                gap_antes = gap_rem.get(item_key, 0.0)
                hr_maq_antes = hr_maq_rem.get(maq_key, 0.0)
                hr_fer_antes = hr_fer_rem.get(fer_key, 0.0)
                pcs_hora = float(row['PCS_HORA'])
                hr_disp = min(hr_maq_antes, hr_fer_antes)
                if gap_antes <= TOL:
                    status='NAO_REALOCADO_SEM_GAP_REMANESCENTE'; qtd_possivel=qtd_realoc=hr_consumida=0.0
                elif hr_maq_antes <= TOL:
                    status='NAO_REALOCADO_SEM_HORA_MAQUINA_LIBERADA'; qtd_possivel=qtd_realoc=hr_consumida=0.0
                elif hr_fer_antes <= TOL:
                    status='NAO_REALOCADO_SEM_HORA_MOLDE_DESTINO'; qtd_possivel=qtd_realoc=hr_consumida=0.0
                elif bloqueado.get(item_key, False):
                    status='NAO_REALOCADO_ITEM_BLOQUEADO_PAI_FILHO'; qtd_possivel=qtd_realoc=hr_consumida=0.0
                else:
                    qtd_possivel = hr_disp * pcs_hora
                    qtd_limite_realoc = min(gap_antes, qtd_possivel)

                    # A capacidade pode ser calculada continuamente, mas a decisao efetiva de realocacao
                    # deve manter a producao final em um bloco operacional valido de QTD_EMB.
                    # Em vez de simplesmente arredondar a parcela adicionada, alinhamos o TOTAL final
                    # do item; isso tambem protege contra pequenos residuos numericos vindos da etapa anterior.
                    qtd_base_item = float(max(0.0, row.get('QTD_FINAL_VIAVEL_PAI_FILHO', 0.0) or 0.0))
                    qtd_emb_item = float(max(0.0, row.get('QTD_EMB', 0.0) or 0.0))
                    if qtd_emb_item > TOL:
                        qtd_final_limite = qtd_base_item + qtd_limite_realoc
                        qtd_final_valida = float(np.floor((qtd_final_limite + TOL) / qtd_emb_item) * qtd_emb_item)
                        qtd_realoc = max(0.0, qtd_final_valida - qtd_base_item)
                        # Protecao contra qualquer efeito de ponto flutuante que pudesse ultrapassar
                        # o gap ou a capacidade efetivamente disponivel.
                        if qtd_realoc > qtd_limite_realoc + TOL:
                            qtd_realoc = max(0.0, float(np.floor((qtd_limite_realoc + TOL) / qtd_emb_item) * qtd_emb_item))
                    else:
                        qtd_realoc = qtd_limite_realoc

                    hr_consumida = qtd_realoc / pcs_hora if pcs_hora > TOL else 0.0
                    if qtd_realoc > TOL:
                        status = 'REALOCADO_TOTAL_GAP' if qtd_realoc >= gap_antes - TOL else 'REALOCADO_PARCIAL_POR_HORA_DISPONIVEL'
                        gap_rem[item_key] = max(0.0, gap_antes - qtd_realoc)
                        hr_maq_rem[maq_key] = max(0.0, hr_maq_antes - hr_consumida)
                        hr_fer_rem[fer_key] = max(0.0, hr_fer_antes - hr_consumida)
                    else:
                        status='NAO_REALOCADO_QTD_ZERO'
                logs.append({
                    'RODADA_REALOC': rodada,
                    'MES_REF': row['MES_REF'],
                    'ID_PROD_UNID_FAT': row['ID_PROD_UNID_FAT'],
                    'COD_PROD': row.get('COD_PROD'),
                    'DESC_PROD': row.get('DESC_PROD'),
                    'UNID_PROD': row.get('UNID_PROD'),
                    'UNID_FAT': row.get('UNID_FAT'),
                    'TIPO_PROD': row.get('TIPO_PROD'),
                    'PRIOR_MATPAR': row.get('PRIOR_MATPAR'),
                    'PRIOR_ROT': row.get('PRIOR_ROT'),
                    'CLASS_ROT': row.get('CLASS_ROT'),
                    'ALOC_REC': row['ALOC_REC'],
                    'COD_FER_UNID': row['COD_FER_UNID'],
                    'PCS_HORA': pcs_hora,
                    'QTD_GAP_ANTES_REALOC': gap_antes,
                    'HR_MAQ_LIBERADA_DISP_ANTES': hr_maq_antes,
                    'HR_FER_DESTINO_DISP_ANTES': hr_fer_antes,
                    'HR_DISPONIVEL_REALOC': hr_disp,
                    'QTD_POSSIVEL_REALOCAR': qtd_possivel,
                    'QTD_REALOCADA': qtd_realoc,
                    'HR_CONSUMIDA_REALOC': hr_consumida,
                    'HR_MAQ_LIBERADA_SALDO_DEPOIS': hr_maq_rem.get(maq_key, 0.0),
                    'HR_FER_DESTINO_SALDO_DEPOIS': hr_fer_rem.get(fer_key, 0.0),
                    'QTD_GAP_DEPOIS_REALOC': gap_rem.get(item_key, 0.0),
                    'STATUS_REALOCACAO': status,
                    'MOTIVO_NAO_REALOCACAO': '' if str(status).startswith('REALOCADO') else status,
                    'MOTIVO_AJUSTE_PAI_FILHO': row.get('MOTIVO_AJUSTE_PAI_FILHO')
                })
                rodada += 1
        # rotina: registrar sobras de horas liberadas que passaram pela realocação e não encontraram uso total.
        for (mes_ref, aloc_rec), hr_saldo in list(hr_maq_rem.items()):
            if hr_saldo > TOL:
                logs.append({
                    'RODADA_REALOC': rodada if 'rodada' in locals() else 1,
                    'MES_REF': mes_ref,
                    'ID_PROD_UNID_FAT': '',
                    'COD_PROD': '',
                    'DESC_PROD': '',
                    'UNID_PROD': '',
                    'UNID_FAT': '',
                    'TIPO_PROD': '',
                    'PRIOR_MATPAR': np.nan,
                    'PRIOR_ROT': np.nan,
                    'CLASS_ROT': '',
                    'ALOC_REC': aloc_rec,
                    'COD_FER_UNID': '',
                    'PCS_HORA': 0.0,
                    'QTD_GAP_ANTES_REALOC': 0.0,
                    'HR_MAQ_LIBERADA_DISP_ANTES': hr_saldo,
                    'HR_FER_DESTINO_DISP_ANTES': 0.0,
                    'HR_DISPONIVEL_REALOC': hr_saldo,
                    'QTD_POSSIVEL_REALOCAR': 0.0,
                    'QTD_REALOCADA': 0.0,
                    'HR_CONSUMIDA_REALOC': 0.0,
                    'HR_MAQ_LIBERADA_SALDO_DEPOIS': hr_saldo,
                    'HR_FER_DESTINO_SALDO_DEPOIS': 0.0,
                    'QTD_GAP_DEPOIS_REALOC': 0.0,
                    'STATUS_REALOCACAO': 'NAO_REALOCADO_HORA_LIBERADA_REMANESCENTE',
                    'MOTIVO_NAO_REALOCACAO': 'SEM_ITEM_COM_GAP_E_ALTERNATIVA_VALIDA_NA_MAQUINA_OU_SEM_FERRAMENTA_DESTINO',
                    'MOTIVO_AJUSTE_PAI_FILHO': ''
                })
                rodada = (rodada + 1) if 'rodada' in locals() else 2
        realoc = pd.DataFrame(logs, columns=cols_log)
        realoc_pos = realoc[realoc['QTD_REALOCADA'] > TOL].copy() if not realoc.empty else realoc.copy()

        realoc_item = realoc_pos.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(
            QTD_REALOCADA_CAP_LIBERADA=('QTD_REALOCADA','sum'),
            HR_CONSUMIDA_REALOC=('HR_CONSUMIDA_REALOC','sum')
        ) if not realoc_pos.empty else pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','QTD_REALOCADA_CAP_LIBERADA','HR_CONSUMIDA_REALOC'])
        resumo_item = na[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','DESC_PROD','UNID_PROD','UNID_FAT','TIPO_PROD','NEC_PCS','QTD_FINAL_VIAVEL_PAI_FILHO','QTD_GAP_APOS_PAI_FILHO','QTD_AJUSTE_PAI_FILHO','MOTIVO_AJUSTE_PAI_FILHO']].copy()
        resumo_item = resumo_item.merge(realoc_item, on=['MES_REF','ID_PROD_UNID_FAT'], how='left').fillna({'QTD_REALOCADA_CAP_LIBERADA':0,'HR_CONSUMIDA_REALOC':0})
        resumo_item['QTD_FINAL_APOS_REALOCACAO'] = resumo_item['QTD_FINAL_VIAVEL_PAI_FILHO'] + resumo_item['QTD_REALOCADA_CAP_LIBERADA']
        resumo_item['QTD_GAP_FINAL_APOS_REALOCACAO'] = (resumo_item['NEC_PCS'] - resumo_item['QTD_FINAL_APOS_REALOCACAO']).clip(lower=0)
        resumo_item['PERC_GAP_RECUPERADO'] = np.where(resumo_item['QTD_GAP_APOS_PAI_FILHO'] > TOL, resumo_item['QTD_REALOCADA_CAP_LIBERADA'] / resumo_item['QTD_GAP_APOS_PAI_FILHO'], 0)
        resumo_item = resumo_item[resumo_item['QTD_GAP_APOS_PAI_FILHO'] > TOL].sort_values(['QTD_REALOCADA_CAP_LIBERADA','QTD_GAP_APOS_PAI_FILHO'], ascending=[False, False])

        # Resumo interno para validacao (nao vai para Excel final como guia 17)
        resumo_maq = pool_maq.copy()
        uso_maq = realoc_pos.groupby(['MES_REF','ALOC_REC'], as_index=False).agg(
            HR_REAPROVEITADA=('HR_CONSUMIDA_REALOC','sum'),
            QTD_REALOCADA=('QTD_REALOCADA','sum'),
            QTD_ITENS_REALOCADOS=('ID_PROD_UNID_FAT','nunique')
        ) if not realoc_pos.empty else pd.DataFrame(columns=['MES_REF','ALOC_REC','HR_REAPROVEITADA','QTD_REALOCADA','QTD_ITENS_REALOCADOS'])
        resumo_maq = resumo_maq.merge(uso_maq, on=['MES_REF','ALOC_REC'], how='left').fillna({'HR_REAPROVEITADA':0,'QTD_REALOCADA':0,'QTD_ITENS_REALOCADOS':0})
        resumo_maq['HR_MAQ_NAO_REAPROVEITADA'] = (resumo_maq['HR_MAQ_LIBERADA_TOTAL'] - resumo_maq['HR_REAPROVEITADA']).clip(lower=0)
        resumo_maq['PERC_REAPROVEITAMENTO_HR'] = np.where(resumo_maq['HR_MAQ_LIBERADA_TOTAL'] > TOL, resumo_maq['HR_REAPROVEITADA'] / resumo_maq['HR_MAQ_LIBERADA_TOTAL'], 0)
        return realoc, resumo_maq, resumo_item

    realocacao_cap_liberada, resumo_realocacao_maquina, resumo_item_realocacao = calcular_realocacao_realoc(
        nao_atend_item,
        alternativas_item,
        plano_producao,
        capacidade_liberada_pf
    )



    # ============================================================
    # 06. VALIDAÇÕES
    # ============================================================

    nec_total = nao_atend_item["NEC_PCS"].sum()
    atend_total = nao_atend_item["QTD_ATENDIDA_SOLVER"].sum()
    nao_atend_total = nao_atend_item["QTD_NAO_ATEND_SOLVER"].sum()
    dif_fechamento = nec_total - atend_total - nao_atend_total
    qtd_rec_estouro = (resumo_recursos["ESTOURO_HR_RECURSO"] > TOL).sum()
    qtd_fer_estouro = (resumo_ferramentas["ESTOURO_HR_FER"] > TOL).sum()

    validacao = pd.DataFrame({
        "METRICA": ["NEC_PCS_TOTAL", "QTD_ATENDIDA_SOLVER_TOTAL", "QTD_NAO_ATEND_SOLVER_TOTAL", "DIF_FECHAMENTO_DEMANDA", "QTD_ITENS_DEMANDA", "QTD_ITENS_COM_PRODUCAO", "QTD_ITENS_SEM_PRODUCAO", "QTD_ITENS_COM_NAO_ATENDIMENTO", "QTD_RECURSOS", "QTD_RECURSOS_ESTOURO", "ESTOURO_HR_RECURSO_TOTAL", "HOR_REC_TOTAL", "HR_PRODUZIR_SOLVER_RECURSO_TOTAL", "OCUPACAO_GLOBAL_RECURSOS", "QTD_FERRAMENTAS", "QTD_FERRAMENTAS_ESTOURO", "ESTOURO_HR_FER_TOTAL", "HOR_FER_TOTAL", "HR_PRODUZIR_SOLVER_FER_TOTAL", "OCUPACAO_GLOBAL_FERRAMENTAS", "QTD_ALTERNATIVAS_AUDITADAS", "QTD_ALTERNATIVAS_VALIDAS_PRODUCAO", "QTD_ALTERNATIVAS_HOR_CAP_ZERO", "QTD_LINHAS_RASTRO_RATEIO", "QTD_ITENS_DIAGNOSTICO"],
        "VALOR": [nec_total, atend_total, nao_atend_total, dif_fechamento, len(nao_atend_item), (nao_atend_item["QTD_ATENDIDA_SOLVER"] > TOL).sum(), (nao_atend_item["QTD_ATENDIDA_SOLVER"] <= TOL).sum(), (nao_atend_item["QTD_NAO_ATEND_SOLVER"] > TOL).sum(), len(resumo_recursos), qtd_rec_estouro, resumo_recursos["ESTOURO_HR_RECURSO"].sum(), resumo_recursos["HOR_REC"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum(), resumo_recursos["HR_PRODUZIR_SOLVER"].sum() / resumo_recursos["HOR_REC"].sum() if resumo_recursos["HOR_REC"].sum() > 0 else 0, len(resumo_ferramentas), qtd_fer_estouro, resumo_ferramentas["ESTOURO_HR_FER"].sum(), resumo_ferramentas["HOR_FER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum(), resumo_ferramentas["HR_PRODUZIR_SOLVER"].sum() / resumo_ferramentas["HOR_FER"].sum() if resumo_ferramentas["HOR_FER"].sum() > 0 else 0, len(bd_alternativas_item), len(bd_alternativas_validas), (bd_alternativas_item["HOR_CAP"] <= TOL).sum(), len(rastro_rateio), len(diagnostico_item)]
    })
    validacao_checks = pd.DataFrame({"METRICA": ["FECHAMENTO_DEMANDA_OK", "CAPACIDADE_RECURSOS_OK", "CAPACIDADE_FERRAMENTAS_OK"], "VALOR": [abs(dif_fechamento) <= 1e-4, qtd_rec_estouro == 0, qtd_fer_estouro == 0]})
    validacao = pd.concat([validacao, pd.DataFrame({"METRICA": ["---"], "VALOR": [""]}), validacao_checks], ignore_index=True)


    # Adiciona bloco de validação da explosão de estrutura dentro da validação geral
    validacao = pd.concat([
        validacao,
        pd.DataFrame({"METRICA": ["--- EXPLOSAO_ESTRUTURA ---"], "VALOR": [""]}),
        validacao_explosao
    ], ignore_index=True)


    # Validação específica rotina operacional - componente 0544NE recebendo demanda estrutural do pai 0544
    try:
        validacao_0544ne = bd_mes.loc[bd_mes["COD_PROD"].astype(str).eq("0544NE"), [c for c in ["MES_REF","ID_PROD_UNID_FAT","COD_PROD","UNID_PROD","UNID_FAT","NEC_DIRETA_PCS","NEC_COMP_PCS","NEC_PCS"] if c in bd_mes.columns]].drop_duplicates().copy()
    except Exception as e:
        print("Aviso validação 0544NE não executada:", e)



    # ============================================================
    # 06B. AUDITORIA COMPLETA DO PLANO - rotina operacional
    # ============================================================
    # AUDITORIA_PLANO_COMPLETO_STATUS_ZERO_PLANO
    # Regra: toda combinação MES_REF + ID_PROD_UNID_FAT existente no bd_LTP_NEC_SOLVER
    # deve aparecer no 03_PLANO_PRODUCAO.
    # Se QTD_PRODUZIR_SOLVER = 0, MOTIVO_STATUS deve explicar o motivo correto:
    # - DEMANDA_DIRETA_COBERTA_POR_ESTOQUE
    # - DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE
    # - ITEM_SEM_DEMANDA_NO_MES
    # - NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE
    # - NAO_ALOCADO_RECURSO_SEM_CAPACIDADE
    # - NAO_ALOCADO_PCS_HORA_INVALIDO
    # - NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA
    # - FALHA_RETORNO_DEMANDA_COMPONENTE_LTP

    try:
        def _key_mes_id(df):
            return pd.to_datetime(df["MES_REF"], errors="coerce").dt.strftime("%Y-%m-%d") + "|" + df["ID_PROD_UNID_FAT"].astype(str)

        def _calc_demanda_bruta_direta_ref(df):
            cols = ["LTP_CART_ARR_MES_ANT", "LTP_CART_MES_ATUAL", "LTP_SALDO_PREV_PCS", "LTP_EST_SEG_PCS"]
            bruta = sum([to_num(df[c]) if c in df.columns else 0 for c in cols])
            if "MESMA_REG" in df.columns and "LTP_SALDO_PREV_PROX_MES_PCS" in df.columns:
                bruta = bruta + np.where(df["MESMA_REG"].astype(str).str.upper().eq("NAO"), to_num(df["LTP_SALDO_PREV_PROX_MES_PCS"]), 0)
            return pd.Series(bruta, index=df.index)

        def _calc_cobertura_ref(df):
            cols = ["LTP_EST_INI_PCS", "LTP_EST_TRANS_PCS", "ORI_TOT_PCS", "TRIANG_TOT_PCS"]
            return sum([to_num(df[c]) if c in df.columns else 0 for c in cols])

        def _map_status_decisao_auditoria(motivo):
            if motivo in ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE"]:
                return "NAO_PLANEJADO_DEMANDA_COBERTA_ESTOQUE"
            if motivo == "ITEM_SEM_DEMANDA_NO_MES":
                return "NAO_PLANEJADO_SEM_DEMANDA"
            if motivo == "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE":
                return "NAO_ALOCADO_CAP_FERRAMENTA_ZERO"
            if motivo == "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE":
                return "NAO_ALOCADO_CAP_RECURSO_ZERO"
            if motivo == "NAO_ALOCADO_PCS_HORA_INVALIDO":
                return "NAO_ALOCADO_ALTERNATIVA_INVALIDA"
            if motivo == "NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA":
                return "NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA"
            if motivo == "FALHA_RETORNO_DEMANDA_COMPONENTE_LTP":
                return "NAO_PLANEJADO_ALERTA_COMPONENTE"
            return "NAO_PLANEJADO_SEM_DEMANDA"

        def _motivo_limitante_auditoria(motivo):
            if motivo == "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE":
                return "COD_FER_UNID"
            if motivo == "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE":
                return "ALOC_REC"
            if motivo == "NAO_ALOCADO_PCS_HORA_INVALIDO":
                return "PCS_HORA_INVALIDO"
            if motivo in ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE", "ITEM_SEM_DEMANDA_NO_MES"]:
                return "NAO_APLICAVEL_SEM_DEMANDA_LIQUIDA"
            return "NAO_APLICAVEL"

        def _motivo_corte_auditoria(motivo):
            if motivo == "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE":
                return "CORTE_POR_COD_FER_UNID"
            if motivo == "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE":
                return "CORTE_POR_ALOC_REC"
            if motivo == "NAO_ALOCADO_PCS_HORA_INVALIDO":
                return "CORTE_POR_ALTERNATIVA_INVALIDA"
            if motivo in ["DEMANDA_DIRETA_COBERTA_POR_ESTOQUE", "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE", "ITEM_SEM_DEMANDA_NO_MES"]:
                return "SEM_CORTE_SEM_DEMANDA_LIQUIDA"
            return "SEM_CORTE"

        base_ltp_auditoria = bd_mes.copy()
        base_ltp_auditoria["MES_REF"] = pd.to_datetime(base_ltp_auditoria["MES_REF"], errors="coerce")
        base_ltp_auditoria["DEMANDA_BRUTA_DIRETA_REF_PCS"] = _calc_demanda_bruta_direta_ref(base_ltp_auditoria)
        base_ltp_auditoria["COBERTURA_ESTOQUE_REF_PCS"] = _calc_cobertura_ref(base_ltp_auditoria)
        base_ltp_auditoria["NEC_DIRETA_CALC_REF_PCS"] = (base_ltp_auditoria["DEMANDA_BRUTA_DIRETA_REF_PCS"] - base_ltp_auditoria["COBERTURA_ESTOQUE_REF_PCS"]).clip(lower=0)

        base_cols = [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP", "DEMANDA_BRUTA_DIRETA_REF_PCS", "COBERTURA_ESTOQUE_REF_PCS", "NEC_DIRETA_CALC_REF_PCS"] if c in base_ltp_auditoria.columns]
        base_status = base_ltp_auditoria.sort_values([c for c in ["MES_REF", "ID_PROD_UNID_FAT", "PRIOR_MATPAR", "PRIOR_ROT", "ALOC_REC", "COD_FER_UNID"] if c in base_ltp_auditoria.columns])[base_cols].drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT"], keep="first").copy()

        nec_cols = [c for c in ["MES_REF", "ID_PROD_UNID_FAT", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA"] if c in nec_consolidada_solver.columns]
        if nec_cols:
            base_status = base_status.merge(nec_consolidada_solver[nec_cols].drop_duplicates(["MES_REF", "ID_PROD_UNID_FAT"]), on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")

        # Motivo de alternativa inválida para itens com necessidade líquida, mas sem produção.
        motivos_alt = pd.DataFrame(columns=["MES_REF", "ID_PROD_UNID_FAT", "MOTIVOS_INVALIDA"])
        if "MOTIVO_INVALIDA" in bd_alternativas_item.columns:
            motivos_alt = bd_alternativas_item.groupby(["MES_REF", "ID_PROD_UNID_FAT"], as_index=False).agg(MOTIVOS_INVALIDA=("MOTIVO_INVALIDA", lambda x: " | ".join(sorted(set(map(str, x))))))
            base_status = base_status.merge(motivos_alt, on=["MES_REF", "ID_PROD_UNID_FAT"], how="left")
        else:
            base_status["MOTIVOS_INVALIDA"] = ""

        def _classificar_status_auditoria(row):
            nec_total = float(row.get("NEC_PCS_CONSOLIDADA", 0) or 0)
            direta = float(row.get("NEC_DIRETA_PCS", 0) or 0)
            comp = float(row.get("NEC_COMP_PCS", 0) or 0)
            exp_bruta = float(row.get("NEC_EXPLODIDA_BRUTA_PCS", 0) or 0)
            exp_liq = float(row.get("NEC_EXPLODIDA_PCS", 0) or 0)
            bruta_dir = float(row.get("DEMANDA_BRUTA_DIRETA_REF_PCS", 0) or 0)
            cobertura = float(row.get("COBERTURA_ESTOQUE_REF_PCS", 0) or 0)
            motivos = str(row.get("MOTIVOS_INVALIDA", ""))
            if nec_total <= TOL:
                if bruta_dir > TOL and cobertura + TOL >= bruta_dir:
                    return "DEMANDA_DIRETA_COBERTA_POR_ESTOQUE"
                if exp_bruta > TOL and exp_liq <= TOL:
                    return "DEMANDA_ESTRUTURAL_COBERTA_POR_ESTOQUE"
                if exp_bruta > TOL and exp_liq > TOL and comp <= TOL:
                    return "FALHA_RETORNO_DEMANDA_COMPONENTE_LTP"
                return "ITEM_SEM_DEMANDA_NO_MES"
            if "HOR_FER_ZERO_OU_NEGATIVO" in motivos:
                return "NAO_ALOCADO_FERRAMENTA_SEM_CAPACIDADE"
            if "HOR_REC_ZERO_OU_NEGATIVO" in motivos:
                return "NAO_ALOCADO_RECURSO_SEM_CAPACIDADE"
            if "PCS_HORA_ZERO_OU_INVALIDO" in motivos:
                return "NAO_ALOCADO_PCS_HORA_INVALIDO"
            return "NAO_ALOCADO_SEM_ALTERNATIVA_VALIDA"

        base_status["STATUS_SUGERIDO_PLANO"] = base_status.apply(_classificar_status_auditoria, axis=1)
        base_status["_KEY_PLANO"] = _key_mes_id(base_status)
        plano_producao["_KEY_PLANO"] = _key_mes_id(plano_producao)

        diag_cols_auditoria = ["DEMANDA_BRUTA_DIRETA_REF_PCS", "COBERTURA_ESTOQUE_REF_PCS", "NEC_DIRETA_CALC_REF_PCS", "NEC_DIRETA_PCS", "NEC_COMP_PCS", "NEC_EXPLODIDA_BRUTA_PCS", "ESTOQUE_CONSUMIDO_EXPLOSAO_PCS", "NEC_EXPLODIDA_PCS", "NEC_PCS_CONSOLIDADA"]
        for c in diag_cols_auditoria:
            if c not in plano_producao.columns:
                plano_producao[c] = np.nan

        base_merge_cols = ["_KEY_PLANO", "STATUS_SUGERIDO_PLANO"] + [c for c in diag_cols_auditoria if c in base_status.columns]
        plano_producao = plano_producao.merge(base_status[base_merge_cols].drop_duplicates("_KEY_PLANO"), on="_KEY_PLANO", how="left", suffixes=("", "_AUD"))
        for c in diag_cols_auditoria:
            ca = c + "_AUD"
            if ca in plano_producao.columns:
                plano_producao[c] = plano_producao[c].where(plano_producao[c].notna(), plano_producao[ca])
                plano_producao = plano_producao.drop(columns=[ca])

        if "MOTIVO_STATUS" not in plano_producao.columns:
            plano_producao["MOTIVO_STATUS"] = ""
        qtd_plan = to_num(plano_producao["QTD_PRODUZIR_SOLVER"])
        plano_producao.loc[qtd_plan > TOL, "MOTIVO_STATUS"] = "PRODUCAO_ALOCADA_PELO_SOLVER"
        mask_zero = qtd_plan <= TOL
        plano_producao.loc[mask_zero & plano_producao["STATUS_SUGERIDO_PLANO"].notna(), "MOTIVO_STATUS"] = plano_producao.loc[mask_zero & plano_producao["STATUS_SUGERIDO_PLANO"].notna(), "STATUS_SUGERIDO_PLANO"]
        for idx in plano_producao.index[mask_zero]:
            motivo = str(plano_producao.at[idx, "MOTIVO_STATUS"])
            plano_producao.at[idx, "STATUS_DECISAO"] = _map_status_decisao_auditoria(motivo)
            if "MOTIVO_LIMITANTE" in plano_producao.columns:
                plano_producao.at[idx, "MOTIVO_LIMITANTE"] = _motivo_limitante_auditoria(motivo)
            if "MOTIVO_CORTE" in plano_producao.columns:
                plano_producao.at[idx, "MOTIVO_CORTE"] = _motivo_corte_auditoria(motivo)

        keys_plano = set(plano_producao["_KEY_PLANO"].astype(str))
        faltantes_auditoria = base_status[~base_status["_KEY_PLANO"].astype(str).isin(keys_plano)].copy()
        linhas_zero = []
        for _, r in faltantes_auditoria.iterrows():
            row = {c: 0 for c in plano_producao.columns if c != "STATUS_SUGERIDO_PLANO"}
            for c in ["MES_REF", "ID_PROD_UNID_FAT", "COD_PROD", "DESC_PROD", "UNID_PROD", "UNID_FAT", "TIPO_PROD", "LINHA_PROD", "FAMILIA_PROD", "MO", "PESO_PROD_KG", "PRIOR_MATPAR", "PRIOR_ROT", "CLASS_ROT", "ALOC_REC", "COD_FER_UNID", "PCS_HORA", "HOR_REC", "HOR_FER", "HOR_CAP"] + diag_cols_auditoria:
                if c in row and c in r.index:
                    row[c] = r[c]
            motivo = str(r.get("STATUS_SUGERIDO_PLANO", "ITEM_SEM_DEMANDA_NO_MES"))
            row["STATUS_DECISAO"] = _map_status_decisao_auditoria(motivo)
            row["MOTIVO_STATUS"] = motivo
            if "MOTIVO_LIMITANTE" in row:
                row["MOTIVO_LIMITANTE"] = _motivo_limitante_auditoria(motivo)
            if "MOTIVO_CORTE" in row:
                row["MOTIVO_CORTE"] = _motivo_corte_auditoria(motivo)
            if "RODADA" in row:
                row["RODADA"] = 0
            row["_KEY_PLANO"] = r["_KEY_PLANO"]
            linhas_zero.append(row)

        if linhas_zero:
            plano_producao = pd.concat([plano_producao, pd.DataFrame(linhas_zero).reindex(columns=plano_producao.columns)], ignore_index=True)

        plano_producao = plano_producao.drop(columns=[c for c in ["STATUS_SUGERIDO_PLANO", "_KEY_PLANO"] if c in plano_producao.columns])
    except Exception as e:
        print("Aviso auditoria operacional plano completo:", e)



    # ============================================================
    # 06C. CORREÇÃO rotina operacional - PLANO FINAL SEM ALTERAR LAYOUT
    # ============================================================
    # Regra:
    # - não criar coluna nova para o cliente;
    # - não renomear coluna;
    # - não remover coluna;
    # - manter QTD_PRODUZIR_SOLVER como coluna oficial da 03_PLANO_PRODUCAO;
    # - atualizar apenas o valor de QTD_PRODUZIR_SOLVER para a produção final válida,
    #   usando as colunas já existentes desde a rotina operacional.

    pass

    _keys_roteamento = ['MES_REF', 'ID_PROD_UNID_FAT']
    if 'MES_REF' in plano_producao.columns:
        plano_producao['MES_REF'] = pd.to_datetime(plano_producao['MES_REF'], errors='coerce')
    if 'MES_REF' in nao_atend_item.columns:
        nao_atend_item['MES_REF'] = pd.to_datetime(nao_atend_item['MES_REF'], errors='coerce')

    # Restaura colunas já validadas na rotina operacional quando a execução local ainda não as tiver montado.
    for _c in ['MOTIVO_AJUSTE_PAI_FILHO', 'QTD_REALOCADA_CAP_LIBERADA', 'QTD_FINAL_APOS_REALOCACAO', 'QTD_GAP_FINAL_APOS_REALOCACAO']:
        if _c not in plano_producao.columns:
            plano_producao[_c] = '' if _c == 'MOTIVO_AJUSTE_PAI_FILHO' else 0.0

    if 'MOTIVO_AJUSTE_PAI_FILHO' in nao_atend_item.columns:
        _mot_pf_roteamento = nao_atend_item[_keys_roteamento + ['MOTIVO_AJUSTE_PAI_FILHO']].drop_duplicates(_keys_roteamento)
        plano_producao = plano_producao.merge(_mot_pf_roteamento, on=_keys_roteamento, how='left', suffixes=('', '_NA_ROTEAMENTO'))
        if 'MOTIVO_AJUSTE_PAI_FILHO_NA_ROTEAMENTO' in plano_producao.columns:
            plano_producao['MOTIVO_AJUSTE_PAI_FILHO'] = np.where(
                plano_producao['MOTIVO_AJUSTE_PAI_FILHO'].astype(str).str.len() > 0,
                plano_producao['MOTIVO_AJUSTE_PAI_FILHO'],
                plano_producao['MOTIVO_AJUSTE_PAI_FILHO_NA_ROTEAMENTO'].fillna('')
            )
            plano_producao = plano_producao.drop(columns=['MOTIVO_AJUSTE_PAI_FILHO_NA_ROTEAMENTO'])

    # Restaura colunas da guia 04 que já existiam na rotina operacional.
    if 'FATOR_AJUSTE_SOBRE_SOLVER' not in nao_atend_item.columns:
        nao_atend_item['FATOR_AJUSTE_SOBRE_SOLVER'] = nao_atend_item['FATOR_FINAL_VIAVEL_PAI_FILHO'] if 'FATOR_FINAL_VIAVEL_PAI_FILHO' in nao_atend_item.columns else 1.0
    if 'QTD_GAP_APOS_PAI_FILHO' not in nao_atend_item.columns:
        _nec_na_roteamento = to_num(nao_atend_item['NEC_PCS']) if 'NEC_PCS' in nao_atend_item.columns else 0.0
        _final_na_roteamento = to_num(nao_atend_item['QTD_FINAL_VIAVEL_PAI_FILHO']) if 'QTD_FINAL_VIAVEL_PAI_FILHO' in nao_atend_item.columns else to_num(nao_atend_item['QTD_ATENDIDA_SOLVER'])
        nao_atend_item['QTD_GAP_APOS_PAI_FILHO'] = np.maximum(_nec_na_roteamento - _final_na_roteamento, 0)
    if 'BLOQUEADO_AJUSTE_PAI_FILHO' not in nao_atend_item.columns:
        _mot_na_roteamento = nao_atend_item['MOTIVO_AJUSTE_PAI_FILHO'].astype(str) if 'MOTIVO_AJUSTE_PAI_FILHO' in nao_atend_item.columns else ''
        nao_atend_item['BLOQUEADO_AJUSTE_PAI_FILHO'] = _mot_na_roteamento.eq('FILHO_COM_ROTEIRO_LIMITOU_PAI')

    # Base de realocação por linha produtiva.
    # rotina: a quantidade realocada deve ser somada na alternativa em que a realocação ocorreu.
    # Não distribuir a realocação proporcionalmente por todas as linhas do item, para não criar estouro artificial de capacidade.
    _realoc_linha_roteamento = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT','QTD_REALOCADA_CAP_LIBERADA'])
    try:
        if 'realocacao_cap_liberada' in locals() and isinstance(realocacao_cap_liberada, pd.DataFrame) and not realocacao_cap_liberada.empty:
            _keys_realoc_linha_roteamento = ['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT']
            _cols_realoc_linha_roteamento = [c for c in _keys_realoc_linha_roteamento + ['QTD_REALOCADA'] if c in realocacao_cap_liberada.columns]
            if set(_keys_realoc_linha_roteamento).issubset(_cols_realoc_linha_roteamento) and 'QTD_REALOCADA' in _cols_realoc_linha_roteamento:
                _realoc_linha_roteamento = realocacao_cap_liberada[_cols_realoc_linha_roteamento].copy()
                _realoc_linha_roteamento['MES_REF'] = pd.to_datetime(_realoc_linha_roteamento['MES_REF'], errors='coerce')
                _realoc_linha_roteamento['QTD_REALOCADA'] = to_num(_realoc_linha_roteamento['QTD_REALOCADA'])
                _realoc_linha_roteamento = _realoc_linha_roteamento[_realoc_linha_roteamento['QTD_REALOCADA'] > TOL].copy()
                _realoc_linha_roteamento = _realoc_linha_roteamento.groupby(_keys_realoc_linha_roteamento, as_index=False).agg(QTD_REALOCADA_CAP_LIBERADA=('QTD_REALOCADA', 'sum'))
    except Exception as e:
        print('Aviso rotina: não foi possível usar realocacao_cap_liberada por linha:', e)

    plano_producao['_QTD_BASE_FINAL_FINAL'] = to_num(plano_producao['QTD_PRODUZIR_FINAL_VIAVEL_EST']) if 'QTD_PRODUZIR_FINAL_VIAVEL_EST' in plano_producao.columns else to_num(plano_producao['QTD_PRODUZIR_SOLVER'])
    if not _realoc_linha_roteamento.empty:
        _keys_merge_realoc_roteamento = ['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT']
        plano_producao = plano_producao.merge(_realoc_linha_roteamento, on=_keys_merge_realoc_roteamento, how='left', suffixes=('', '_LINHA_ROTEAMENTO'))
        if 'QTD_REALOCADA_CAP_LIBERADA_LINHA_ROTEAMENTO' in plano_producao.columns:
            plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = to_num(plano_producao['QTD_REALOCADA_CAP_LIBERADA_LINHA_ROTEAMENTO'])
            plano_producao = plano_producao.drop(columns=['QTD_REALOCADA_CAP_LIBERADA_LINHA_ROTEAMENTO'])
        else:
            plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = 0.0
    else:
        plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = 0.0

    plano_producao['QTD_FINAL_APOS_REALOCACAO'] = plano_producao['_QTD_BASE_FINAL_FINAL'] + to_num(plano_producao['QTD_REALOCADA_CAP_LIBERADA'])

    if 'NEC_PCS_CONSOLIDADA' in plano_producao.columns:
        _nec_ref_final = to_num(plano_producao['NEC_PCS_CONSOLIDADA'])
    elif 'NEC_PCS' in plano_producao.columns:
        _nec_ref_final = to_num(plano_producao['NEC_PCS'])
    else:
        _nec_ref_final = 0.0
    plano_producao['QTD_GAP_FINAL_APOS_REALOCACAO'] = np.maximum(_nec_ref_final - to_num(plano_producao['QTD_FINAL_APOS_REALOCACAO']), 0)

    # Atualiza a coluna oficial sem mudar o nome.
    plano_producao['QTD_PRODUZIR_SOLVER'] = to_num(plano_producao['QTD_FINAL_APOS_REALOCACAO'])
    plano_producao['HR_PRODUZIR_SOLVER'] = np.where(to_num(plano_producao['PCS_HORA']) > TOL, to_num(plano_producao['QTD_PRODUZIR_SOLVER']) / to_num(plano_producao['PCS_HORA']), 0)
    if 'PESO_PROD_KG' in plano_producao.columns and 'VOL_PRODUZIR_SOLVER' in plano_producao.columns:
        plano_producao['VOL_PRODUZIR_SOLVER'] = to_num(plano_producao['QTD_PRODUZIR_SOLVER']) * to_num(plano_producao['PESO_PROD_KG'])
    plano_producao = plano_producao.drop(columns=['_QTD_BASE_FINAL_FINAL'])

    # Recalcula resumos que dependem diretamente da coluna oficial, sem mudar layout.
    prod_recurso = plano_producao.groupby('ALOC_REC', as_index=False).agg(
        HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
        QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
        QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique')
    ) if not plano_producao.empty else pd.DataFrame(columns=['ALOC_REC', 'HR_PRODUZIR_SOLVER', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS'])
    resumo_recursos = bd_cap_recursos.merge(prod_recurso, on='ALOC_REC', how='left').fillna({'HR_PRODUZIR_SOLVER': 0, 'QTD_PRODUZIR_SOLVER': 0, 'QTD_ITENS': 0})
    resumo_recursos['OCUPACAO_RECURSO_PCT'] = np.where(resumo_recursos['HOR_REC'] > 0, resumo_recursos['HR_PRODUZIR_SOLVER'] / resumo_recursos['HOR_REC'], 0)
    resumo_recursos['HR_RECURSO_OCIOSA'] = resumo_recursos['HOR_REC'] - resumo_recursos['HR_PRODUZIR_SOLVER']
    resumo_recursos['ESTOURO_HR_RECURSO'] = np.where(resumo_recursos['HR_PRODUZIR_SOLVER'] > resumo_recursos['HOR_REC'] + TOL, resumo_recursos['HR_PRODUZIR_SOLVER'] - resumo_recursos['HOR_REC'], 0)
    resumo_recursos = resumo_recursos[['ALOC_REC', 'HOR_REC', 'HR_PRODUZIR_SOLVER', 'OCUPACAO_RECURSO_PCT', 'HR_RECURSO_OCIOSA', 'ESTOURO_HR_RECURSO', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS']].sort_values(['OCUPACAO_RECURSO_PCT', 'HR_PRODUZIR_SOLVER'], ascending=[False, False])

    prod_ferramenta = plano_producao.groupby('COD_FER_UNID', as_index=False).agg(
        HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
        QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
        QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique'),
        QTD_RECURSOS=('ALOC_REC', 'nunique')
    ) if not plano_producao.empty else pd.DataFrame(columns=['COD_FER_UNID', 'HR_PRODUZIR_SOLVER', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS', 'QTD_RECURSOS'])
    resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on='COD_FER_UNID', how='left').fillna({'HR_PRODUZIR_SOLVER': 0, 'QTD_PRODUZIR_SOLVER': 0, 'QTD_ITENS': 0, 'QTD_RECURSOS': 0})
    resumo_ferramentas['OCUPACAO_FER_PCT'] = np.where(resumo_ferramentas['HOR_FER'] > 0, resumo_ferramentas['HR_PRODUZIR_SOLVER'] / resumo_ferramentas['HOR_FER'], 0)
    resumo_ferramentas['HR_FER_OCIOSA'] = resumo_ferramentas['HOR_FER'] - resumo_ferramentas['HR_PRODUZIR_SOLVER']
    resumo_ferramentas['ESTOURO_HR_FER'] = np.where(resumo_ferramentas['HR_PRODUZIR_SOLVER'] > resumo_ferramentas['HOR_FER'] + TOL, resumo_ferramentas['HR_PRODUZIR_SOLVER'] - resumo_ferramentas['HOR_FER'], 0)
    resumo_ferramentas = resumo_ferramentas[['COD_FER_UNID', 'HOR_FER', 'HR_PRODUZIR_SOLVER', 'OCUPACAO_FER_PCT', 'HR_FER_OCIOSA', 'ESTOURO_HR_FER', 'QTD_PRODUZIR_SOLVER', 'QTD_ITENS', 'QTD_RECURSOS']].sort_values(['OCUPACAO_FER_PCT', 'HR_PRODUZIR_SOLVER'], ascending=[False, False])

    # Recalcula diagnóstico de alternativas para manter consistência com a coluna oficial.
    try:
        uso_alt = plano_producao.groupby(['ID_PROD_UNID_FAT', 'ALOC_REC', 'COD_FER_UNID', 'PRIOR_MATPAR', 'PRIOR_ROT'], as_index=False).agg(
            HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
            QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum')
        ) if not plano_producao.empty else pd.DataFrame(columns=['ID_PROD_UNID_FAT', 'ALOC_REC', 'COD_FER_UNID', 'PRIOR_MATPAR', 'PRIOR_ROT', 'HR_PRODUZIR_SOLVER', 'QTD_PRODUZIR_SOLVER'])
        diagnostico_alternativas = alternativas_item.merge(uso_alt, on=['ID_PROD_UNID_FAT', 'ALOC_REC', 'COD_FER_UNID', 'PRIOR_MATPAR', 'PRIOR_ROT'], how='left')
        diagnostico_alternativas['HR_PRODUZIR_SOLVER'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'].fillna(0)
        diagnostico_alternativas['QTD_PRODUZIR_SOLVER'] = diagnostico_alternativas['QTD_PRODUZIR_SOLVER'].fillna(0)
        diagnostico_alternativas['ALTERNATIVA_USADA'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'] > TOL
    except Exception as e:
        print('Aviso rotina: diagnóstico de alternativas não recalculado:', e)


    # Preserva exatamente a ordem de colunas já validada na rotina operacional para as guias afetadas.
    _ORDEM_03_PLANO_FINAL = ['MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD', 'LINHA_PROD', 'FAMILIA_PROD', 'MO', 'RODADA', 'PRIOR_MATPAR', 'PRIOR_ROT', 'CLASS_ROT', 'ALOC_REC', 'COD_FER_UNID', 'PCS_HORA', 'HOR_REC', 'HOR_FER', 'HOR_CAP', 'QTD_PENDENTE_ANTES', 'QTD_SOLICITADA', 'HR_SOLICITADA', 'CAP_RESTANTE_ALOC_REC_ANTES', 'CAP_RESTANTE_COD_FER_ANTES', 'CAP_DISPONIVEL_LINHA', 'HR_DEMANDA_TOTAL_ALOC_REC', 'HR_DEMANDA_TOTAL_COD_FER', 'FATOR_RATEIO_ALOC_REC', 'FATOR_RATEIO_COD_FER', 'FATOR_RATEIO', 'MOTIVO_LIMITANTE', 'QTD_PRODUZIR_SOLVER', 'PESO_PROD_KG', 'VOL_PRODUZIR_SOLVER', 'HR_PRODUZIR_SOLVER', 'QTD_PENDENTE_DEPOIS', 'MOTIVO_CORTE', 'STATUS_DECISAO', 'DEMANDA_BRUTA_DIRETA_REF_PCS', 'COBERTURA_ESTOQUE_REF_PCS', 'NEC_DIRETA_CALC_REF_PCS', 'NEC_DIRETA_PCS', 'NEC_COMP_PCS', 'NEC_EXPLODIDA_BRUTA_PCS', 'ESTOQUE_CONSUMIDO_EXPLOSAO_PCS', 'NEC_EXPLODIDA_PCS', 'NEC_PCS_CONSOLIDADA', 'MOTIVO_STATUS', 'QTD_REDUCAO_POS_PAI_FILHO', 'FATOR_FINAL_VIAVEL_ORIGEM', 'FATOR_AJUSTE_SOBRE_SOLVER', 'MOTIVO_AJUSTE_PAI_FILHO', 'QTD_PRODUZIR_FINAL_VIAVEL_EST', 'HR_LIBERADA_EST', 'QTD_REALOCADA_CAP_LIBERADA', 'QTD_FINAL_APOS_REALOCACAO', 'QTD_GAP_FINAL_APOS_REALOCACAO']
    _ORDEM_04_NAO_ATEND_FINAL = ['MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD', 'NEC_PCS_ORIGINAL', 'NEC_ESTRUTURA_PAI_PCS', 'NEC_DIRETA_BRUTA_PCS', 'NEC_DIRETA_PCS', 'NEC_EXPLODIDA_BRUTA_PCS', 'ESTOQUE_CONSUMIDO_EXPLOSAO_PCS', 'NEC_EXPLODIDA_PCS', 'NEC_COMP_PCS', 'NEC_PCS_CONSOLIDADA', 'NEC_PCS', 'QTD_ATENDIDA_SOLVER', 'QTD_NAO_ATEND_SOLVER', 'PERC_ATENDIDO_SOLVER', 'QTD_FINAL_VIAVEL_PAI_FILHO', 'QTD_AJUSTE_PAI_FILHO', 'FATOR_FINAL_VIAVEL_PAI_FILHO', 'FATOR_AJUSTE_SOBRE_SOLVER', 'MOTIVO_AJUSTE_PAI_FILHO', 'QTD_GAP_APOS_PAI_FILHO', 'BLOQUEADO_AJUSTE_PAI_FILHO']
    plano_producao = plano_producao[[c for c in _ORDEM_03_PLANO_FINAL if c in plano_producao.columns] + [c for c in plano_producao.columns if c not in _ORDEM_03_PLANO_FINAL]].copy()
    nao_atend_item = nao_atend_item[[c for c in _ORDEM_04_NAO_ATEND_FINAL if c in nao_atend_item.columns] + [c for c in nao_atend_item.columns if c not in _ORDEM_04_NAO_ATEND_FINAL]].copy()

    pass



    # ============================================================
    # 06D. MATERIALIZACAO TECNICA CONTINUA - V66_25
    # ============================================================
    # Regra canonica: a fase tecnica permanece continua. LIMIT_PCS, LOTE_MIN e QTD_EMB
    # so podem atuar depois do fechamento tecnico/estrutural, na etapa industrial dedicada.
    # Portanto, quando aplicar_regras_industriais=False, nao existe arredondamento legado.
    if not plano_producao.empty:
        _qtd_final_v66 = pd.to_numeric(plano_producao['QTD_PRODUZIR_SOLVER'], errors='coerce').fillna(0.0).clip(lower=0.0)
        if aplicar_regras_industriais:
            plano_producao['QTD_PRODUZIR_SOLVER'] = np.floor(_qtd_final_v66 + 0.5)
        else:
            plano_producao['QTD_PRODUZIR_SOLVER'] = _qtd_final_v66

        _pcs_hora_v66 = pd.to_numeric(plano_producao['PCS_HORA'], errors='coerce').fillna(0.0)
        plano_producao['HR_PRODUZIR_SOLVER'] = np.divide(
            plano_producao['QTD_PRODUZIR_SOLVER'].to_numpy(dtype=float),
            _pcs_hora_v66.to_numpy(dtype=float),
            out=np.zeros(len(plano_producao), dtype=float),
            where=_pcs_hora_v66.to_numpy(dtype=float) > TOL
        )

        if 'PESO_PROD_KG' in plano_producao.columns and 'VOL_PRODUZIR_SOLVER' in plano_producao.columns:
            plano_producao['VOL_PRODUZIR_SOLVER'] = plano_producao['QTD_PRODUZIR_SOLVER'] * pd.to_numeric(plano_producao['PESO_PROD_KG'], errors='coerce').fillna(0.0)

        if 'QTD_PRODUZIR_FINAL_VIAVEL_EST' in plano_producao.columns:
            plano_producao['QTD_PRODUZIR_FINAL_VIAVEL_EST'] = plano_producao['QTD_PRODUZIR_SOLVER']
        if 'QTD_FINAL_APOS_REALOCACAO' in plano_producao.columns:
            plano_producao['QTD_FINAL_APOS_REALOCACAO'] = plano_producao['QTD_PRODUZIR_SOLVER']
        if 'QTD_REDUCAO_POS_PAI_FILHO' in plano_producao.columns:
            plano_producao['QTD_REDUCAO_POS_PAI_FILHO'] = 0.0
        if 'HR_LIBERADA_EST' in plano_producao.columns:
            plano_producao['HR_LIBERADA_EST'] = 0.0
        if 'QTD_REALOCADA_CAP_LIBERADA' in plano_producao.columns:
            plano_producao['QTD_REALOCADA_CAP_LIBERADA'] = 0.0
        if 'MOTIVO_AJUSTE_PAI_FILHO' in plano_producao.columns:
            plano_producao['MOTIVO_AJUSTE_PAI_FILHO'] = 'SEM_AJUSTE_POS_SOLVER'

    # Nao existe mais fechamento pai/filho ou propagacao pos-Solver nesta versao.
    ajuste_pai_filho = pd.DataFrame()
    resumo_pai_filho = pd.DataFrame()
    capacidade_liberada_pf = pd.DataFrame()
    realocacao_cap_liberada = pd.DataFrame()
    resumo_item_realocacao = pd.DataFrame()

    # Mantem somente a demanda estrutural ja calculada antes do Solver para o encadeamento mensal.
    _cols_demanda_final_v66 = [c for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_PROD','UNID_FAT','NEC_EXPLODIDA_BRUTA_PCS'] if c in nec_consolidada_solver.columns]
    demanda_final_m1 = nec_consolidada_solver[_cols_demanda_final_v66].copy() if _cols_demanda_final_v66 else pd.DataFrame()
    if not demanda_final_m1.empty:
        if 'NEC_EXPLODIDA_BRUTA_PCS' in demanda_final_m1.columns:
            demanda_final_m1 = demanda_final_m1.rename(columns={'NEC_EXPLODIDA_BRUTA_PCS':'NEC_BRUTA_FINAL'})
        elif 'NEC_BRUTA_FINAL' not in demanda_final_m1.columns:
            demanda_final_m1['NEC_BRUTA_FINAL'] = 0.0

    # Recalcula os resumos diretamente da quantidade final inteira.
    prod_recurso = plano_producao.groupby('ALOC_REC', as_index=False).agg(
        HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
        QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
        QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique')
    ) if not plano_producao.empty else pd.DataFrame(columns=['ALOC_REC','HR_PRODUZIR_SOLVER','QTD_PRODUZIR_SOLVER','QTD_ITENS'])
    resumo_recursos = bd_cap_recursos.merge(prod_recurso, on='ALOC_REC', how='left').fillna({'HR_PRODUZIR_SOLVER':0,'QTD_PRODUZIR_SOLVER':0,'QTD_ITENS':0})
    resumo_recursos['OCUPACAO_RECURSO_PCT'] = np.where(resumo_recursos['HOR_REC'] > 0, resumo_recursos['HR_PRODUZIR_SOLVER'] / resumo_recursos['HOR_REC'], 0)
    resumo_recursos['HR_RECURSO_OCIOSA'] = resumo_recursos['HOR_REC'] - resumo_recursos['HR_PRODUZIR_SOLVER']
    resumo_recursos['ESTOURO_HR_RECURSO'] = np.where(resumo_recursos['HR_PRODUZIR_SOLVER'] > resumo_recursos['HOR_REC'] + TOL, resumo_recursos['HR_PRODUZIR_SOLVER'] - resumo_recursos['HOR_REC'], 0)
    resumo_recursos = resumo_recursos[['ALOC_REC','HOR_REC','HR_PRODUZIR_SOLVER','OCUPACAO_RECURSO_PCT','HR_RECURSO_OCIOSA','ESTOURO_HR_RECURSO','QTD_PRODUZIR_SOLVER','QTD_ITENS']].sort_values(['OCUPACAO_RECURSO_PCT','HR_PRODUZIR_SOLVER'], ascending=[False,False])

    prod_ferramenta = plano_producao.groupby('COD_FER_UNID', as_index=False).agg(
        HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER', 'sum'),
        QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER', 'sum'),
        QTD_ITENS=('ID_PROD_UNID_FAT', 'nunique'),
        QTD_RECURSOS=('ALOC_REC', 'nunique')
    ) if not plano_producao.empty else pd.DataFrame(columns=['COD_FER_UNID','HR_PRODUZIR_SOLVER','QTD_PRODUZIR_SOLVER','QTD_ITENS','QTD_RECURSOS'])
    resumo_ferramentas = bd_cap_ferramentas.merge(prod_ferramenta, on='COD_FER_UNID', how='left').fillna({'HR_PRODUZIR_SOLVER':0,'QTD_PRODUZIR_SOLVER':0,'QTD_ITENS':0,'QTD_RECURSOS':0})
    resumo_ferramentas['OCUPACAO_FER_PCT'] = np.where(resumo_ferramentas['HOR_FER'] > 0, resumo_ferramentas['HR_PRODUZIR_SOLVER'] / resumo_ferramentas['HOR_FER'], 0)
    resumo_ferramentas['HR_FER_OCIOSA'] = resumo_ferramentas['HOR_FER'] - resumo_ferramentas['HR_PRODUZIR_SOLVER']
    resumo_ferramentas['ESTOURO_HR_FER'] = np.where(resumo_ferramentas['HR_PRODUZIR_SOLVER'] > resumo_ferramentas['HOR_FER'] + TOL, resumo_ferramentas['HR_PRODUZIR_SOLVER'] - resumo_ferramentas['HOR_FER'], 0)
    resumo_ferramentas = resumo_ferramentas[['COD_FER_UNID','HOR_FER','HR_PRODUZIR_SOLVER','OCUPACAO_FER_PCT','HR_FER_OCIOSA','ESTOURO_HR_FER','QTD_PRODUZIR_SOLVER','QTD_ITENS','QTD_RECURSOS']].sort_values(['OCUPACAO_FER_PCT','HR_PRODUZIR_SOLVER'], ascending=[False,False])

    # Atualiza atendimento/gap apos o arredondamento final, sem outra regra industrial.
    if not nao_atend_item.empty:
        _atendido_final_v66 = plano_producao.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(QTD_ATENDIDA_SOLVER=('QTD_PRODUZIR_SOLVER','sum')) if not plano_producao.empty else pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','QTD_ATENDIDA_SOLVER'])
        nao_atend_item = nao_atend_item.drop(columns=['QTD_ATENDIDA_SOLVER'], errors='ignore').merge(_atendido_final_v66, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
        nao_atend_item['QTD_ATENDIDA_SOLVER'] = pd.to_numeric(nao_atend_item['QTD_ATENDIDA_SOLVER'], errors='coerce').fillna(0.0)
        nao_atend_item['QTD_NAO_ATEND_SOLVER'] = (pd.to_numeric(nao_atend_item['NEC_PCS'], errors='coerce').fillna(0.0) - nao_atend_item['QTD_ATENDIDA_SOLVER']).clip(lower=0.0)
        nao_atend_item['PERC_ATENDIDO_SOLVER'] = np.where(pd.to_numeric(nao_atend_item['NEC_PCS'], errors='coerce').fillna(0.0) > 0, nao_atend_item['QTD_ATENDIDA_SOLVER'] / pd.to_numeric(nao_atend_item['NEC_PCS'], errors='coerce').fillna(0.0), 0.0)
        nao_atend_item['PERC_NAO_ATENDIDO_SOLVER'] = 1.0 - nao_atend_item['PERC_ATENDIDO_SOLVER']

    try:
        _uso_alt_final_v66 = plano_producao.groupby(['ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT'], as_index=False).agg(HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER','sum'), QTD_PRODUZIR_SOLVER=('QTD_PRODUZIR_SOLVER','sum'))
        diagnostico_alternativas = alternativas_item.merge(_uso_alt_final_v66, on=['ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT'], how='left')
        diagnostico_alternativas['HR_PRODUZIR_SOLVER'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'].fillna(0.0)
        diagnostico_alternativas['QTD_PRODUZIR_SOLVER'] = diagnostico_alternativas['QTD_PRODUZIR_SOLVER'].fillna(0.0)
        diagnostico_alternativas['ALTERNATIVA_USADA'] = diagnostico_alternativas['HR_PRODUZIR_SOLVER'] > TOL
    except Exception as exc:
        print('Diagnóstico de alternativas não recalculado:', exc)


    # ============================================================
    # 06E.1. RASTREABILIDADE DA FORMACAO DA NEC NO PLANO
    # ============================================================
    # Exibe no 03_PLANO_PRODUCAO os mesmos campos de necessidade usados pelo motor.
    # M1: os valores permanecem os dados originais do LTP.
    # M2 em diante: LTP_EST_INI_PCS e LTP_CART_ARR_MES_ANT ja chegam em bd_mes
    # atualizados pelo encadeamento mensal; portanto nao sao criadas colunas paralelas.
    _cols_formacao_nec_v58 = [
        'LTP_CART_ARR_MES_ANT',
        'LTP_CART_MES_ATUAL',
        'LTP_SALDO_PREV_PCS',
        'LTP_EST_SEG_PCS',
        'LTP_EST_INI_PCS',
        'LTP_EST_TRANS_PCS',
        'ORI_TOT_PCS',
        'TRIANG_TOT_PCS',
        'NEC_PCS',
    ]
    _cols_formacao_nec_v58 = [c for c in _cols_formacao_nec_v58 if c in bd_mes.columns]

    if not plano_producao.empty and _cols_formacao_nec_v58:
        _base_formacao_nec_v58 = bd_mes.copy()
        _base_formacao_nec_v58['MES_REF'] = pd.to_datetime(_base_formacao_nec_v58['MES_REF'], errors='coerce')
        _base_formacao_nec_v58['ID_PROD_UNID_FAT'] = _base_formacao_nec_v58['ID_PROD_UNID_FAT'].astype(str)
        for _c_v58 in _cols_formacao_nec_v58:
            _base_formacao_nec_v58[_c_v58] = to_num(_base_formacao_nec_v58[_c_v58])

        _base_formacao_nec_v58 = (
            _base_formacao_nec_v58
            .groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)
            .agg({_c_v58: 'max' for _c_v58 in _cols_formacao_nec_v58})
        )

        plano_producao['MES_REF'] = pd.to_datetime(plano_producao['MES_REF'], errors='coerce')
        plano_producao['ID_PROD_UNID_FAT'] = plano_producao['ID_PROD_UNID_FAT'].astype(str)
        plano_producao = plano_producao.drop(columns=[c for c in _cols_formacao_nec_v58 if c in plano_producao.columns], errors='ignore')
        plano_producao = plano_producao.merge(
            _base_formacao_nec_v58,
            on=['MES_REF', 'ID_PROD_UNID_FAT'],
            how='left',
            validate='many_to_one',
        )
        for _c_v58 in _cols_formacao_nec_v58:
            plano_producao[_c_v58] = to_num(plano_producao[_c_v58])

        # Traz regras operacionais e classificacoes para a mesma tabela de validação do plano.
        _cols_validacao_v58 = ['QTD_EMB','LOTE_MIN','CURVA_ABC','CURVA_123']
        _base_validacao_v58 = bd_mes[['MES_REF','ID_PROD_UNID_FAT'] + [c for c in _cols_validacao_v58 if c in bd_mes.columns]].copy()
        for _c_v58 in ['QTD_EMB','LOTE_MIN']:
            if _c_v58 in _base_validacao_v58.columns:
                _base_validacao_v58[_c_v58] = to_num(_base_validacao_v58[_c_v58])
        _agg_validacao_v58 = {c:'first' for c in _base_validacao_v58.columns if c not in ['MES_REF','ID_PROD_UNID_FAT']}
        _base_validacao_v58 = _base_validacao_v58.groupby(['MES_REF','ID_PROD_UNID_FAT'],as_index=False).agg(_agg_validacao_v58)
        plano_producao = plano_producao.drop(columns=[c for c in _cols_validacao_v58 if c in plano_producao.columns], errors='ignore')
        plano_producao = plano_producao.merge(_base_validacao_v58,on=['MES_REF','ID_PROD_UNID_FAT'],how='left',validate='many_to_one')
        for _c_v58 in _cols_validacao_v58:
            if _c_v58 not in plano_producao.columns:
                plano_producao[_c_v58] = pd.NA

        # Mantem os campos de formacao da necessidade juntos e no inicio do bloco de negocio.
        _prefixo_plano_v58 = [
            'MES_REF', 'ID_PROD_UNID_FAT', 'COD_PROD', 'DESC_PROD', 'UNID_PROD', 'UNID_FAT', 'TIPO_PROD',
            'QTD_EMB', 'LOTE_MIN', 'CURVA_ABC', 'CURVA_123',
            'LTP_CART_ARR_MES_ANT', 'LTP_CART_MES_ATUAL', 'LTP_SALDO_PREV_PCS', 'LTP_EST_SEG_PCS',
            'LTP_EST_INI_PCS', 'LTP_EST_TRANS_PCS', 'ORI_TOT_PCS', 'TRIANG_TOT_PCS', 'NEC_PCS',
        ]
        plano_producao = plano_producao[
            [c for c in _prefixo_plano_v58 if c in plano_producao.columns]
            + [c for c in plano_producao.columns if c not in _prefixo_plano_v58]
        ].copy()

    return {
        'resumo_recursos': resumo_recursos,
        'resumo_ferramentas': resumo_ferramentas,
        'plano_producao': plano_producao,
        'nao_atend_item': nao_atend_item,
        'auditoria_capacidade': auditoria_capacidade,
        'alternativas_item': alternativas_item,
        'rastro_rateio': rastro_rateio,
        'rastro_prioridade_componentes': rastro_prioridade_componentes,
        'diagnostico_matriz': diagnostico_matriz,
        'diagnostico_matriz_rotas': diagnostico_matriz_rotas,
        'diagnostico_item': diagnostico_item,
        'diagnostico_alternativas': diagnostico_alternativas,
        'estrutura_explodida': estrutura_explodida,
        'nec_explodida_componentes': nec_explodida_componentes,
        'nec_consolidada_solver': nec_consolidada_solver,
        'ajuste_pai_filho': ajuste_pai_filho,
        'resumo_pai_filho': resumo_pai_filho,
        'capacidade_liberada_pf': capacidade_liberada_pf,
        'realocacao_cap_liberada': realocacao_cap_liberada,
        'resumo_item_realocacao': resumo_item_realocacao,
        'demanda_final_m1': demanda_final_m1,
    }

# ============================================================
# 06F. PROCESSAMENTO SEQUENCIAL DE TODOS OS PERIODOS
# ============================================================

def _chave_estado_v58(cod_prod, unid_fat):
    # Chave oficial alinhada com o cliente: estoque/carteira fecham e reabrem por COD_PROD + UNID_FAT.
    return (str(cod_prod).strip(), str(unid_fat).strip())


def _base_item_periodo_v58(bd_periodo):
    cols_valor = [
        'LTP_EST_INI_PCS','LTP_EST_TRANS_PCS','ORI_TOT_PCS','TRIANG_TOT_PCS',
        'LTP_SALDO_PREV_PCS','LTP_CART_MES_ATUAL','LTP_CART_ARR_MES_ANT'
    ]
    base = bd_periodo.copy()
    base['MES_REF'] = pd.to_datetime(base['MES_REF'], errors='coerce')
    base['COD_PROD'] = normalizar_codigo_serie(base['COD_PROD'])
    base['UNID_FAT'] = base['UNID_FAT'].astype('string').fillna('').str.strip()
    for c in cols_valor:
        if c not in base.columns:
            base[c] = 0.0
        base[c] = to_num(base[c])
    chave = ['MES_REF','ID_PROD_UNID_FAT']
    agg = {c:'max' for c in cols_valor}
    agg.update({c:'first' for c in ['COD_PROD','UNID_PROD','UNID_FAT'] if c in base.columns})
    return base.groupby(chave, as_index=False).agg(agg)


def _calcular_estado_proximo_periodo_v58(bd_periodo, resultado_periodo):
    base = _base_item_periodo_v58(bd_periodo)
    plano = resultado_periodo['plano_producao'].copy()
    if plano.empty:
        prod = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','PRODUCAO_MES'])
    else:
        plano['MES_REF'] = pd.to_datetime(plano['MES_REF'], errors='coerce')
        plano['QTD_PRODUZIR_SOLVER'] = to_num(plano['QTD_PRODUZIR_SOLVER'])
        prod = plano.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(PRODUCAO_MES=('QTD_PRODUZIR_SOLVER','sum'))

    dem = resultado_periodo.get('demanda_final_m1', pd.DataFrame()).copy()
    if dem.empty or 'NEC_BRUTA_FINAL' not in dem.columns:
        estrut = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','DEMANDA_ESTRUTURAL_MES'])
    else:
        dem['MES_REF'] = pd.to_datetime(dem['MES_REF'], errors='coerce')
        dem['NEC_BRUTA_FINAL'] = to_num(dem['NEC_BRUTA_FINAL'])
        estrut = dem.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(DEMANDA_ESTRUTURAL_MES=('NEC_BRUTA_FINAL','max'))

    saldo = base.merge(prod, on=['MES_REF','ID_PROD_UNID_FAT'], how='left').merge(estrut, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
    saldo[['PRODUCAO_MES','DEMANDA_ESTRUTURAL_MES']] = saldo[['PRODUCAO_MES','DEMANDA_ESTRUTURAL_MES']].fillna(0.0)
    saldo['SALDO_PROX_PERIODO'] = (
        saldo['LTP_EST_INI_PCS'] + saldo['LTP_EST_TRANS_PCS'] + saldo['ORI_TOT_PCS'] + saldo['TRIANG_TOT_PCS']
        + saldo['PRODUCAO_MES'] - saldo['LTP_SALDO_PREV_PCS'] - saldo['LTP_CART_MES_ATUAL']
        - saldo['LTP_CART_ARR_MES_ANT'] - saldo['DEMANDA_ESTRUTURAL_MES']
    )
    saldo['ESTOQUE_INICIAL_PROX_PERIODO'] = saldo['SALDO_PROX_PERIODO'].clip(lower=0.0)
    saldo['CARTEIRA_ARRASTE_PROX_PERIODO'] = (-saldo['SALDO_PROX_PERIODO']).clip(lower=0.0)
    saldo['CHAVE_ESTADO'] = [_chave_estado_v58(c, u) for c, u in zip(saldo['COD_PROD'], saldo['UNID_FAT'])]
    return saldo


def _aplicar_estado_anterior_v58(bd_periodo, estado_itens):
    out = bd_periodo.copy()
    if 'LTP_EST_INI_PCS' not in out.columns:
        out['LTP_EST_INI_PCS'] = 0.0
    if 'LTP_CART_ARR_MES_ANT' not in out.columns:
        out['LTP_CART_ARR_MES_ANT'] = 0.0
    out['COD_PROD'] = normalizar_codigo_serie(out['COD_PROD'])
    out['UNID_FAT'] = out['UNID_FAT'].astype('string').fillna('').str.strip()
    chaves = [_chave_estado_v58(c, u) for c, u in zip(out['COD_PROD'], out['UNID_FAT'])]
    # Itens novos no horizonte, sem estado anterior, entram com zero projetado; itens existentes usam o fechamento anterior.
    out['LTP_EST_INI_PCS'] = [float(estado_itens[k]['ESTOQUE']) if k in estado_itens else 0.0 for k in chaves]
    out['LTP_CART_ARR_MES_ANT'] = [float(estado_itens[k]['ARRASTE']) if k in estado_itens else 0.0 for k in chaves]
    out['_CHAVE_ESTADO_V58'] = chaves
    # NEC e campos derivados precisam ser recalculados com o novo estoque inicial e novo arraste.
    out = out.drop(columns=[c for c in ['NEC_PCS','NEC_ESTRUTURA_PAI_PCS','NEC_DIRETA_BRUTA_PCS'] if c in out.columns], errors='ignore')
    return out


def _auditar_transporte_estado_v58(bd_periodo, estado_anterior, mes_ref):
    # Compara, antes do Solver do mês, o fechamento anterior com o valor efetivamente injetado no mês corrente.
    if not estado_anterior:
        return pd.DataFrame()
    atual = bd_periodo.copy()
    atual['COD_PROD'] = normalizar_codigo_serie(atual['COD_PROD'])
    atual['UNID_FAT'] = atual['UNID_FAT'].astype('string').fillna('').str.strip()
    atual['CHAVE_ESTADO_V58'] = [_chave_estado_v58(c, u) for c, u in zip(atual['COD_PROD'], atual['UNID_FAT'])]
    atual = atual.groupby('CHAVE_ESTADO_V58', as_index=False).agg(
        COD_PROD=('COD_PROD','first'), UNID_FAT=('UNID_FAT','first'),
        ESTOQUE_INICIAL_ATUAL=('LTP_EST_INI_PCS','max'), CARTEIRA_ARR_ATUAL=('LTP_CART_ARR_MES_ANT','max')
    )
    registros=[]
    for r in atual.itertuples(index=False):
        k=r.CHAVE_ESTADO_V58
        if k not in estado_anterior:
            continue
        esp=estado_anterior[k]
        dif_est=float(r.ESTOQUE_INICIAL_ATUAL)-float(esp['ESTOQUE'])
        dif_arr=float(r.CARTEIRA_ARR_ATUAL)-float(esp['ARRASTE'])
        registros.append({
            'MES_REF':pd.Timestamp(mes_ref),'COD_PROD':r.COD_PROD,'UNID_FAT':r.UNID_FAT,
            'ESTOQUE_FINAL_MES_ANT':float(esp['ESTOQUE']),'ESTOQUE_INICIAL_MES_ATUAL':float(r.ESTOQUE_INICIAL_ATUAL),
            'DIF_ESTOQUE':dif_est,'CARTEIRA_FINAL_MES_ANT':float(esp['ARRASTE']),
            'CARTEIRA_ARR_MES_ATUAL':float(r.CARTEIRA_ARR_ATUAL),'DIF_CARTEIRA':dif_arr,
            'STATUS':'OK' if abs(dif_est)<=TOL and abs(dif_arr)<=TOL else 'DIVERGENTE'
        })
    audit=pd.DataFrame(registros)
    if not audit.empty and audit['STATUS'].ne('OK').any():
        qtd=int(audit['STATUS'].ne('OK').sum())
        raise RuntimeError(f'ENCADEAMENTO_MENSAL_DIVERGENTE: {qtd} chaves COD_PROD + UNID_FAT nao reconciliaram em {pd.Timestamp(mes_ref):%Y-%m}.')
    return audit



bd_entrada_completa = pd.read_excel(INPUT_FILE, dtype={'COD_PROD': str, 'ID_PROD_UNID_FAT': str, 'UNID_PROD': str, 'UNID_FAT': str})
bd_entrada_completa['MES_REF'] = pd.to_datetime(bd_entrada_completa['MES_REF'], errors='coerce').dt.normalize()
periodos_v58 = [pd.Timestamp(x) for x in sorted(bd_entrada_completa['MES_REF'].dropna().unique().tolist())]
if not periodos_v58:
    raise RuntimeError('PROCESSAMENTO_MENSAL_SEM_PERIODOS_VALIDOS')
_estado_itens_v58 = {}
_saldos_periodos_v58 = []
_tempos_periodos_v58 = []
_planos_finais_periodos = []
_auditorias_finais_periodos = []
_validacoes_finais_periodos = []
_log_tempo_periodos = []
print(f'       CICLO SEQUENCIAL - {len(periodos_v58)} periodo(s)')

for _idx_periodo_v58, _mes_ref_v58 in enumerate(periodos_v58):
    _bd_periodo_v58 = bd_entrada_completa.loc[bd_entrada_completa['MES_REF'].eq(_mes_ref_v58)].copy().reset_index(drop=True)
    _inicio_periodo_v58 = time.perf_counter()
    if _idx_periodo_v58 > 0:
        _bd_periodo_v58 = _aplicar_estado_anterior_v58(_bd_periodo_v58, _estado_itens_v58)
        _auditar_transporte_estado_v58(_bd_periodo_v58, _estado_itens_v58, _mes_ref_v58)
    _resultado_v58 = _executar_motor_periodo_v58(_bd_periodo_v58, aplicar_regras_industriais=False)
    # Every period has an independent initial global Solver pass, followed by the same local and industrial closure.
    # Fotografia unica do motor tecnico M1
    print("       FOTOGRAFIA TECNICA M1 congelada - nenhuma rerodagem global apos este ponto")
    PLANO_FOTOGRAFIA_TECNICA_M1 = _resultado_v58.get('plano_producao', pd.DataFrame()).copy()
    NEC_FOTOGRAFIA_TECNICA_M1 = _resultado_v58.get('nec_consolidada_solver', pd.DataFrame()).copy()
    ESTRUTURA_FOTOGRAFIA_TECNICA_M1 = _resultado_v58.get('estrutura_explodida', pd.DataFrame()).copy()


    CRITERIOS_MATRIZ_CORTE = [
        (1, 'SIM', 'EST_SEG', 'SIM|EST_SEG'),
        (2, 'NAO', 'PREV_PROX', 'NAO|PREV_PROX'),
        (3, 'NAO', 'EST_SEG', 'NAO|EST_SEG'),
        (4, 'SIM', 'PREV_VEN', 'SIM|PREV_VEN'),
        (5, 'NAO', 'PREV_VEN', 'NAO|PREV_VEN'),
        (6, 'SIM', 'C_AT', 'SIM|C_AT'),
        (7, 'NAO', 'C_AT', 'NAO|C_AT'),
        (8, 'SIM', 'C_ARR', 'SIM|C_ARR'),
        (9, 'NAO', 'C_ARR', 'NAO|C_ARR'),
    ]
    print('       MATRIZ LOCAL - 9 criterios canonicos carregados')

    # A hierarquia de alocacao experimental e o inverso exato da matriz canonica.
    assert [(f,c) for _,f,c,_ in reversed(CRITERIOS_MATRIZ_CORTE)] == [(f,n.split("|")[1]) for f,_,n in PRIORIDADE_COMPONENTES_V6629], "HIERARQUIA_ALOCACAO_DIVERGE_DA_MATRIZ"


    _ordem_criterios=[x[3] for x in CRITERIOS_MATRIZ_CORTE]
    _esperado=['SIM|EST_SEG','NAO|PREV_PROX','NAO|EST_SEG','SIM|PREV_VEN','NAO|PREV_VEN','SIM|C_AT','NAO|C_AT','SIM|C_ARR','NAO|C_ARR']
    if _ordem_criterios != _esperado:
        raise RuntimeError('REGRESSAO_CRITERIOS_MATRIZ')
    if 'SIM|PREV_PROX' in _ordem_criterios:
        raise RuntimeError('REGRESSAO_SIM_PREV_PROX')
    print('       BLINDAGEM CANONICA - criterios e componentes conferidos')


    # Celula preservada como marco de arquitetura.
    # A aplicacao real dos cortes esta concentrada na etapa 04G para manter um unico estado local.
    print('       PREPARACAO LOCAL - sem chamada a _executar_motor_periodo_v58')


    print('       ESTABILIDADE LOCAL - fabrica global permanece congelada apos a fotografia tecnica')


    # ============================================================
    # 04G. FECHAMENTO TECNICO LOCAL M1 - v66_26
    # ============================================================
    # Contrato canonico aplicado aqui:
    # - Solver global roda UMA unica vez antes desta etapa;
    # - REC primeiro, FER depois; se FER expuser REC, volta para REC;
    # - GAP multirota pertence ao produto e usa primeira rota apenas como ancora contabil;
    # - corte fisico unico nos 5 componentes permitidos;
    # - corte recuperavel pertence ao produto;
    # - nenhum corte/recuperacao chama novamente _executar_motor_periodo_v58;
    # - reconciliacao trabalha apenas com deltas de NEC, plano, REC, FER e PA->PI afetados.

    print('[03/06] Fechamento tecnico local do M1 - REC -> FER -> recuperacao...')
    _iniciar_etapa_v58('FECHAMENTO_TECNICO_LOCAL_M1')
    _T0_FECH_TEC_LOCAL = time.perf_counter()

    TOL_LOCAL_HR = max(TOL * 100, 1e-6)
    MAX_GIROS_LOCAL_M1 = 100  # blindagem, nunca criterio normal de convergencia
    MAX_BISSECAO_RECUP_M1 = 14

    CRITERIOS_MATRIZ_LOCAL_M1 = [
        (1, 'SIM', 'EST_SEG', 'SIM|EST_SEG'),
        (2, 'NAO', 'PREV_PROX', 'NAO|PREV_PROX'),
        (3, 'NAO', 'EST_SEG', 'NAO|EST_SEG'),
        (4, 'SIM', 'PREV_VEN', 'SIM|PREV_VEN'),
        (5, 'NAO', 'PREV_VEN', 'NAO|PREV_VEN'),
        (6, 'SIM', 'C_AT', 'SIM|C_AT'),
        (7, 'NAO', 'C_AT', 'NAO|C_AT'),
        (8, 'SIM', 'C_ARR', 'SIM|C_ARR'),
        (9, 'NAO', 'C_ARR', 'NAO|C_ARR'),
    ]

    MAPA_COMPONENTE_LTP_LOCAL_M1 = {
        'C_ARR': 'LTP_CART_ARR_MES_ANT',
        'C_AT': 'LTP_CART_MES_ATUAL',
        'PREV_VEN': 'LTP_SALDO_PREV_PCS',
        'PREV_PROX': 'LTP_SALDO_PREV_PROX_MES_PCS',
        'EST_SEG': 'LTP_EST_SEG_PCS',
    }
    MAPA_ORI_LOCAL_M1 = {
        'C_ARR': 'ORI_C_ARR',
        'C_AT': 'ORI_C_AT',
        'PREV_VEN': 'ORI_PREV_VEN',
        'PREV_PROX': 'ORI_PREV_PROX',
        'EST_SEG': 'ORI_EST_SEG',
    }

    # Fotografia unica do motor global. Nenhuma chamada global depois deste ponto.
    RESULTADO_BASE_TECNICO_M1 = dict(_resultado_v58)
    PLANO_TECNICO_BASE_M1 = RESULTADO_BASE_TECNICO_M1.get('plano_producao', pd.DataFrame()).copy()
    ALTERNATIVAS_BASE_M1 = RESULTADO_BASE_TECNICO_M1.get('alternativas_item', pd.DataFrame()).copy()
    NEC_BASE_TECNICA_M1 = RESULTADO_BASE_TECNICO_M1.get('nec_consolidada_solver', pd.DataFrame()).copy()
    ESTRUTURA_BASE_TECNICA_M1 = RESULTADO_BASE_TECNICO_M1.get('estrutura_explodida', pd.DataFrame()).copy()
    RASTRO_RATEIO_BASE_M1 = RESULTADO_BASE_TECNICO_M1.get('rastro_rateio', pd.DataFrame()).copy()
    BD_M1_BASE_LOCAL = _bd_periodo_v58.copy().reset_index(drop=True)

    if PLANO_TECNICO_BASE_M1.empty or NEC_BASE_TECNICA_M1.empty:
        raise RuntimeError('FECHAMENTO_TECNICO_LOCAL_M1_BASE_VAZIA')

    # -----------------------------
    # Normalizacao basica
    # -----------------------------
    def _solver_num(s):
        return pd.to_numeric(s, errors='coerce').fillna(0.0)

    for _df in [PLANO_TECNICO_BASE_M1, ALTERNATIVAS_BASE_M1, NEC_BASE_TECNICA_M1, BD_M1_BASE_LOCAL]:
        if 'MES_REF' in _df.columns:
            _df['MES_REF'] = pd.to_datetime(_df['MES_REF'], errors='coerce').dt.normalize()
        if 'ID_PROD_UNID_FAT' in _df.columns:
            _df['ID_PROD_UNID_FAT'] = _df['ID_PROD_UNID_FAT'].astype(str)
        for _c in ['COD_PROD', 'COD_INSUMO', 'COD_PROD_ORIGEM']:
            if _c in _df.columns:
                _df[_c] = normalizar_codigo_serie(_df[_c])
        for _c in ['UNID_FAT', 'UNID_PROD', 'ALOC_REC', 'COD_FER_UNID', 'MESMA_REG']:
            if _c in _df.columns:
                _df[_c] = _df[_c].astype('string').fillna('').str.strip()

    MES_M1_LOCAL = pd.to_datetime(NEC_BASE_TECNICA_M1['MES_REF'], errors='coerce').min().normalize()
    PLANO_TECNICO_BASE_M1 = PLANO_TECNICO_BASE_M1[PLANO_TECNICO_BASE_M1['MES_REF'].eq(MES_M1_LOCAL)].copy()
    ALTERNATIVAS_BASE_M1 = ALTERNATIVAS_BASE_M1[ALTERNATIVAS_BASE_M1['MES_REF'].eq(MES_M1_LOCAL)].copy()
    NEC_BASE_TECNICA_M1 = NEC_BASE_TECNICA_M1[NEC_BASE_TECNICA_M1['MES_REF'].eq(MES_M1_LOCAL)].copy()
    BD_M1_BASE_LOCAL = BD_M1_BASE_LOCAL[BD_M1_BASE_LOCAL['MES_REF'].eq(MES_M1_LOCAL)].copy()

    # -----------------------------
    # Estado dos cinco componentes cortaveis
    # -----------------------------
    for _c in MAPA_COMPONENTE_LTP_LOCAL_M1.values():
        if _c not in BD_M1_BASE_LOCAL.columns:
            BD_M1_BASE_LOCAL[_c] = 0.0
        BD_M1_BASE_LOCAL[_c] = _solver_num(BD_M1_BASE_LOCAL[_c]).clip(lower=0.0)
    if 'MESMA_REG' not in BD_M1_BASE_LOCAL.columns:
        BD_M1_BASE_LOCAL['MESMA_REG'] = 'SIM'
    BD_M1_BASE_LOCAL['MESMA_REG'] = BD_M1_BASE_LOCAL['MESMA_REG'].astype(str).str.upper().str.strip()

    _meta_item = [c for c in ['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_PROD','UNID_FAT','DESC_PROD','TIPO_PROD','MESMA_REG'] if c in BD_M1_BASE_LOCAL.columns]
    _agg_comp = {v:'max' for v in MAPA_COMPONENTE_LTP_LOCAL_M1.values()}
    for _c in [c for c in _meta_item if c not in ['MES_REF','ID_PROD_UNID_FAT']]:
        _agg_comp[_c] = 'first'
    COMPONENTES_LOCAL_M1 = BD_M1_BASE_LOCAL.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(_agg_comp)
    for _comp, _col in MAPA_COMPONENTE_LTP_LOCAL_M1.items():
        COMPONENTES_LOCAL_M1[_comp] = _solver_num(COMPONENTES_LOCAL_M1[_col]).clip(lower=0.0)
        COMPONENTES_LOCAL_M1[MAPA_ORI_LOCAL_M1[_comp]] = COMPONENTES_LOCAL_M1[_comp]
    COMPONENTES_LOCAL_M1['MESMA_REG'] = COMPONENTES_LOCAL_M1['MESMA_REG'].astype(str).str.upper().str.strip()

    # NEC tecnica continua inicial: esta e a referencia produtiva congelada para medir somente deltas.
    _nec_col = 'NEC_PCS_CONSOLIDADA' if 'NEC_PCS_CONSOLIDADA' in NEC_BASE_TECNICA_M1.columns else 'NEC_PCS'
    NEC_INICIAL_LOCAL_M1 = NEC_BASE_TECNICA_M1[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_FAT',_nec_col]].copy()
    NEC_INICIAL_LOCAL_M1 = NEC_INICIAL_LOCAL_M1.rename(columns={_nec_col:'NEC_TECNICA_INICIAL'})
    NEC_INICIAL_LOCAL_M1['NEC_TECNICA_INICIAL'] = _solver_num(NEC_INICIAL_LOCAL_M1['NEC_TECNICA_INICIAL']).clip(lower=0.0)
    NEC_INICIAL_LOCAL_M1 = NEC_INICIAL_LOCAL_M1.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False).agg(
        COD_PROD=('COD_PROD','first'), UNID_FAT=('UNID_FAT','first'), NEC_TECNICA_INICIAL=('NEC_TECNICA_INICIAL','max')
    )
    COMPONENTES_LOCAL_M1 = COMPONENTES_LOCAL_M1.merge(
        NEC_INICIAL_LOCAL_M1[['MES_REF','ID_PROD_UNID_FAT','NEC_TECNICA_INICIAL']],
        on=['MES_REF','ID_PROD_UNID_FAT'], how='left', validate='one_to_one'
    )
    COMPONENTES_LOCAL_M1['NEC_TECNICA_INICIAL'] = _solver_num(COMPONENTES_LOCAL_M1['NEC_TECNICA_INICIAL']).clip(lower=0.0)
    # v66_31: regras industriais por item tambem devem acompanhar o estado da NEC apos cortes.
    # Elas nao mudam o delta comercial nem a explosao original PA->PI.
    _REGRAS_NEC_POS_CORTE_M1 = ['LIMIT_PCS', 'LOTE_MIN', 'QTD_EMB']
    _regras_src = NEC_BASE_TECNICA_M1[['MES_REF', 'ID_PROD_UNID_FAT'] + [c for c in _REGRAS_NEC_POS_CORTE_M1 if c in NEC_BASE_TECNICA_M1.columns]].copy()
    for _c in _REGRAS_NEC_POS_CORTE_M1:
        if _c not in _regras_src.columns: _regras_src[_c] = 0.0
        _regras_src[_c] = _solver_num(_regras_src[_c]).clip(lower=0.0)
    _regras_src = _regras_src.groupby(['MES_REF', 'ID_PROD_UNID_FAT'], as_index=False)[_REGRAS_NEC_POS_CORTE_M1].max()
    COMPONENTES_LOCAL_M1 = COMPONENTES_LOCAL_M1.merge(_regras_src, on=['MES_REF', 'ID_PROD_UNID_FAT'], how='left', validate='one_to_one')
    for _c in _REGRAS_NEC_POS_CORTE_M1: COMPONENTES_LOCAL_M1[_c] = _solver_num(COMPONENTES_LOCAL_M1[_c]).clip(lower=0.0)
    # v66_30: ORI_NEC e a NEC REAL original, nao o alvo industrial arredondado.
    # NEC_TECNICA_INICIAL ja contem o alvo ajustado para alocacao/cortes.
    if 'NEC_PCS_BASE_CALCULADA_INTERNA' in NEC_BASE_TECNICA_M1.columns:
        _nec_real_ori = (NEC_BASE_TECNICA_M1[
            ['MES_REF', 'ID_PROD_UNID_FAT', 'NEC_PCS_BASE_CALCULADA_INTERNA']
        ].drop_duplicates(['MES_REF', 'ID_PROD_UNID_FAT'])
         .rename(columns={'NEC_PCS_BASE_CALCULADA_INTERNA': 'ORI_NEC'}))
        COMPONENTES_LOCAL_M1 = COMPONENTES_LOCAL_M1.merge(
            _nec_real_ori, on=['MES_REF', 'ID_PROD_UNID_FAT'], how='left', validate='one_to_one')
        COMPONENTES_LOCAL_M1['ORI_NEC'] = _solver_num(COMPONENTES_LOCAL_M1['ORI_NEC'])
    else:
        COMPONENTES_LOCAL_M1['ORI_NEC'] = COMPONENTES_LOCAL_M1['NEC_TECNICA_INICIAL']

    # -----------------------------
    # Mapa PA -> PI acumulado, vetorizado e reutilizavel
    # -----------------------------
    # O proprio resultado da explosao ja traz a relacao raiz -> descendente e a necessidade bruta.
    # O coeficiente acumulado e NEC_GERADA_BRUTA_PCS / NEC_RAIZ_PCS.
    def _solver_mapa_estrutura(estrutura, nec_itens):
        if estrutura is None or estrutura.empty:
            return pd.DataFrame(columns=['MES_REF','ID_RAIZ','ID_FILHO','FATOR_ESTRUTURAL'])
        e = estrutura.copy()
        for c in ['MES_REF','ID_PROD_UNID_FAT_ORIGEM','COD_INSUMO','UNID_PROD','NEC_GERADA_BRUTA_PCS','NEC_RAIZ_PCS']:
            if c not in e.columns:
                e[c] = 0 if c in ['NEC_GERADA_BRUTA_PCS','NEC_RAIZ_PCS'] else ''
        e['MES_REF'] = pd.to_datetime(e['MES_REF'], errors='coerce').dt.normalize()
        e['ID_RAIZ'] = e['ID_PROD_UNID_FAT_ORIGEM'].astype(str)
        e['COD_INSUMO'] = normalizar_codigo_serie(e['COD_INSUMO'])
        e['UNID_RETORNO'] = e['UNID_PROD'].astype('string').fillna('').str.strip()
        e['NEC_GERADA_BRUTA_PCS'] = _solver_num(e['NEC_GERADA_BRUTA_PCS'])
        e['NEC_RAIZ_PCS'] = _solver_num(e['NEC_RAIZ_PCS'])
        e['FATOR_ESTRUTURAL'] = np.divide(
            e['NEC_GERADA_BRUTA_PCS'].to_numpy(float), e['NEC_RAIZ_PCS'].to_numpy(float),
            out=np.zeros(len(e), dtype=float), where=e['NEC_RAIZ_PCS'].to_numpy(float) > TOL
        )
        e = e[e['FATOR_ESTRUTURAL'] > TOL].copy()
        mapa_id = nec_itens[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_FAT']].drop_duplicates().copy()
        mapa_id['COD_PROD'] = normalizar_codigo_serie(mapa_id['COD_PROD'])
        mapa_id['UNID_FAT'] = mapa_id['UNID_FAT'].astype('string').fillna('').str.strip()
        e = e.merge(
            mapa_id.rename(columns={'COD_PROD':'COD_INSUMO','UNID_FAT':'UNID_RETORNO','ID_PROD_UNID_FAT':'ID_FILHO'}),
            on=['MES_REF','COD_INSUMO','UNID_RETORNO'], how='inner'
        )
        e = e[e['ID_RAIZ'].ne(e['ID_FILHO'].astype(str))].copy()
        return e.groupby(['MES_REF','ID_RAIZ','ID_FILHO'], as_index=False)['FATOR_ESTRUTURAL'].sum()

    MAPA_ESTRUTURAL_LOCAL_M1 = _solver_mapa_estrutura(ESTRUTURA_BASE_TECNICA_M1, NEC_INICIAL_LOCAL_M1)

    # -----------------------------
    # Calculo do estado de NEC por DELTA, sem reexplosao global
    # -----------------------------
    def _solver_estado_nec(comp_state):
        x = comp_state[['MES_REF','ID_PROD_UNID_FAT','NEC_TECNICA_INICIAL','ORI_NEC'] + _REGRAS_NEC_POS_CORTE_M1 + list(MAPA_ORI_LOCAL_M1.values()) + list(MAPA_ORI_LOCAL_M1.keys())].copy()
        red = np.zeros(len(x), dtype=float)
        for comp in MAPA_ORI_LOCAL_M1:
            red += (_solver_num(x[MAPA_ORI_LOCAL_M1[comp]]) - _solver_num(x[comp])).clip(lower=0.0).to_numpy(float)
        x['RED_DIRETA_BRUTA'] = red
        # O corte direto so pode reduzir necessidade produtiva ate o NEC atual da propria origem.
        x['RED_DIRETA_EFETIVA'] = np.minimum(x['RED_DIRETA_BRUTA'].to_numpy(float), _solver_num(x['NEC_TECNICA_INICIAL']).to_numpy(float))
        if MAPA_ESTRUTURAL_LOCAL_M1.empty:
            red_estr = pd.DataFrame(columns=['MES_REF','ID_PROD_UNID_FAT','RED_ESTRUTURAL'])
        else:
            f = MAPA_ESTRUTURAL_LOCAL_M1.merge(
                x[['MES_REF','ID_PROD_UNID_FAT','RED_DIRETA_EFETIVA']].rename(columns={'ID_PROD_UNID_FAT':'ID_RAIZ'}),
                on=['MES_REF','ID_RAIZ'], how='left'
            )
            f['RED_DIRETA_EFETIVA'] = _solver_num(f['RED_DIRETA_EFETIVA'])
            f['RED_ESTRUTURAL'] = f['RED_DIRETA_EFETIVA'] * _solver_num(f['FATOR_ESTRUTURAL'])
            red_estr = f.groupby(['MES_REF','ID_FILHO'], as_index=False)['RED_ESTRUTURAL'].sum().rename(columns={'ID_FILHO':'ID_PROD_UNID_FAT'})
        x = x.merge(red_estr, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
        x['RED_ESTRUTURAL'] = _solver_num(x.get('RED_ESTRUTURAL', 0))
        x['RED_TOTAL_NEC'] = x['RED_DIRETA_BRUTA'] + x['RED_ESTRUTURAL']
        # A NEC tecnica inicial pode ter sido ampliada por lote minimo e embalagem.
        # Cortes comerciais/estruturais devem reduzir primeiro a NEC real original;
        # somente o saldo real restante e arredondado pelas regras industriais.
        # Nao transformar a diferenca de arredondamento inicial em demanda residual.
        base_real = np.minimum(
            _solver_num(x['ORI_NEC']).clip(lower=0.0).to_numpy(float),
            _solver_num(x['NEC_TECNICA_INICIAL']).clip(lower=0.0).to_numpy(float),
        )
        x['NEC_POS_CORTE_REAL'] = np.maximum(base_real - _solver_num(x['RED_TOTAL_NEC']).to_numpy(float), 0.0)
        # v66_31: o corte pode desfazer o multiplo que a v66_30 ajustou na NEC inicial.
        # Reajusta o ALVO industrial antes da realocacao, sem recompor demandas comerciais cortadas.
        # Zero permanece zero; a capacidade REC/FER e a viabilidade PA->PI continuam limitando a producao.
        alvo = np.maximum(x['NEC_POS_CORTE_REAL'].to_numpy(float), x['LIMIT_PCS'].to_numpy(float))
        alvo = pd.Series(np.where(x['NEC_POS_CORTE_REAL'].to_numpy(float)>TOL, alvo, 0.0), index=x.index)
        x['NEC_ATUAL'] = ajustar_nec_lote_embalagem(alvo, x['LOTE_MIN'], x['QTD_EMB'])
        return x[['MES_REF','ID_PROD_UNID_FAT','NEC_TECNICA_INICIAL','RED_DIRETA_BRUTA','RED_DIRETA_EFETIVA','RED_ESTRUTURAL','NEC_ATUAL']]

    # -----------------------------
    # Rotas e plano local
    # -----------------------------
    for c in ['QTD_PRODUZIR_SOLVER','PCS_HORA','HR_PRODUZIR_SOLVER','HOR_REC','HOR_FER','PRIOR_MATPAR','PRIOR_ROT']:
        if c not in PLANO_TECNICO_BASE_M1.columns:
            PLANO_TECNICO_BASE_M1[c] = 0.0
        PLANO_TECNICO_BASE_M1[c] = _solver_num(PLANO_TECNICO_BASE_M1[c])
    for c in ['ALOC_REC','COD_FER_UNID']:
        PLANO_TECNICO_BASE_M1[c] = PLANO_TECNICO_BASE_M1[c].astype('string').fillna('').str.strip()
    PLANO_LOCAL_M1 = PLANO_TECNICO_BASE_M1.copy().reset_index(drop=True)
    PLANO_LOCAL_M1['_LINHA_ORIGINAL'] = True

    if 'ALTERNATIVA_VALIDA' in ALTERNATIVAS_BASE_M1.columns:
        ALTS_VALIDAS_LOCAL_M1 = ALTERNATIVAS_BASE_M1[ALTERNATIVAS_BASE_M1['ALTERNATIVA_VALIDA'].fillna(False)].copy()
    else:
        ALTS_VALIDAS_LOCAL_M1 = ALTERNATIVAS_BASE_M1.copy()
    for c in ['PCS_HORA','HOR_REC','HOR_FER','PRIOR_MATPAR','PRIOR_ROT']:
        if c not in ALTS_VALIDAS_LOCAL_M1.columns:
            ALTS_VALIDAS_LOCAL_M1[c] = 0.0
        ALTS_VALIDAS_LOCAL_M1[c] = _solver_num(ALTS_VALIDAS_LOCAL_M1[c])
    for c in ['ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID']:
        ALTS_VALIDAS_LOCAL_M1[c] = ALTS_VALIDAS_LOCAL_M1[c].astype(str).str.strip()
    ALTS_VALIDAS_LOCAL_M1 = ALTS_VALIDAS_LOCAL_M1[
        (ALTS_VALIDAS_LOCAL_M1['PCS_HORA'] > TOL) &
        (ALTS_VALIDAS_LOCAL_M1['HOR_REC'] > TOL) &
        (ALTS_VALIDAS_LOCAL_M1['HOR_FER'] > TOL) &
        ALTS_VALIDAS_LOCAL_M1['ALOC_REC'].ne('') & ALTS_VALIDAS_LOCAL_M1['COD_FER_UNID'].ne('')
    ].copy()
    ALTS_VALIDAS_LOCAL_M1 = ALTS_VALIDAS_LOCAL_M1.sort_values(['ID_PROD_UNID_FAT','PRIOR_MATPAR','PRIOR_ROT','ALOC_REC','COD_FER_UNID']).drop_duplicates(['ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT'])

    # Garante linhas internas para rotas validas ainda nao usadas. Linhas adicionadas so saem no plano se receberem producao.
    _ch_rota = ['ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT']
    _ch_rota = [c for c in _ch_rota if c in PLANO_LOCAL_M1.columns and c in ALTS_VALIDAS_LOCAL_M1.columns]
    _exist = PLANO_LOCAL_M1[_ch_rota].drop_duplicates().assign(_EXISTE=1) if _ch_rota else pd.DataFrame()
    _missing = ALTS_VALIDAS_LOCAL_M1.merge(_exist, on=_ch_rota, how='left') if _ch_rota else pd.DataFrame()
    if not _missing.empty:
        _missing = _missing[_missing['_EXISTE'].isna()].drop(columns='_EXISTE')
        if not _missing.empty:
            _novas = pd.DataFrame(index=np.arange(len(_missing)), columns=PLANO_LOCAL_M1.columns)
            for c in [c for c in PLANO_LOCAL_M1.columns if c in _missing.columns]:
                _novas[c] = _missing[c].to_numpy()
            _novas['MES_REF'] = MES_M1_LOCAL
            _novas['QTD_PRODUZIR_SOLVER'] = 0.0
            _novas['HR_PRODUZIR_SOLVER'] = 0.0
            _novas['_LINHA_ORIGINAL'] = False
            PLANO_LOCAL_M1 = pd.concat([PLANO_LOCAL_M1, _novas], ignore_index=True, sort=False)

    # Fonte unica da capacidade: todas as alternativas validas do recurso/ferramenta.
    # Nao recalcular capacidade com base apenas nas rotas que receberam producao.
    CAP_REC_LOCAL_M1 = ALTS_VALIDAS_LOCAL_M1.groupby('ALOC_REC')['HOR_REC'].max().to_dict()
    CAP_FER_LOCAL_M1 = ALTS_VALIDAS_LOCAL_M1.groupby('COD_FER_UNID')['HOR_FER'].max().to_dict()
    # Padroniza o cadastro de capacidade das linhas do plano. A capacidade pertence
    # ao recurso/mes e nao ao produto/rota; horas produzidas continuam inalteradas.
    PLANO_LOCAL_M1['HOR_REC'] = PLANO_LOCAL_M1['ALOC_REC'].map(CAP_REC_LOCAL_M1).fillna(0.0)
    PLANO_LOCAL_M1['HOR_FER'] = PLANO_LOCAL_M1['COD_FER_UNID'].map(CAP_FER_LOCAL_M1).fillna(0.0)


    def _solver_totais_plano(plano):
        p = plano.copy()
        p['QTD_PRODUZIR_SOLVER'] = _solver_num(p['QTD_PRODUZIR_SOLVER']).clip(lower=0.0)
        p['PCS_HORA'] = _solver_num(p['PCS_HORA'])
        p['HR_PRODUZIR_SOLVER'] = np.divide(
            p['QTD_PRODUZIR_SOLVER'].to_numpy(float), p['PCS_HORA'].to_numpy(float),
            out=np.zeros(len(p), dtype=float), where=p['PCS_HORA'].to_numpy(float) > TOL
        )
        tot = p.groupby(['MES_REF','ID_PROD_UNID_FAT'], as_index=False)['QTD_PRODUZIR_SOLVER'].sum().rename(columns={'QTD_PRODUZIR_SOLVER':'QTD_PLANO'})
        rec = p.groupby('ALOC_REC', as_index=False)['HR_PRODUZIR_SOLVER'].sum().set_index('ALOC_REC')['HR_PRODUZIR_SOLVER'].to_dict()
        fer = p.groupby('COD_FER_UNID', as_index=False)['HR_PRODUZIR_SOLVER'].sum().set_index('COD_FER_UNID')['HR_PRODUZIR_SOLVER'].to_dict()
        return p, tot, rec, fer


    def _solver_reduzir_plano_ao_nec(plano, estado):
        p, tot, _, _ = _solver_totais_plano(plano)
        e = estado[['MES_REF','ID_PROD_UNID_FAT','NEC_ATUAL']].copy()
        z = tot.merge(e, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
        z['NEC_ATUAL'] = _solver_num(z['NEC_ATUAL'])
        z['EXCESSO'] = (z['QTD_PLANO'] - z['NEC_ATUAL']).clip(lower=0.0)
        afet = z[z['EXCESSO'] > TOL]
        if afet.empty:
            return p, 0.0
        reduzido = 0.0
        for r in afet.itertuples(index=False):
            saldo = float(r.EXCESSO)
            idx = p.index[p['ID_PROD_UNID_FAT'].astype(str).eq(str(r.ID_PROD_UNID_FAT)) & (_solver_num(p['QTD_PRODUZIR_SOLVER']) > TOL)].tolist()
            idx = sorted(idx, key=lambda i: (float(p.at[i,'PRIOR_MATPAR'] or 0), float(p.at[i,'PRIOR_ROT'] or 0), i), reverse=True)
            for i in idx:
                if saldo <= TOL:
                    break
                q = float(p.at[i,'QTD_PRODUZIR_SOLVER'] or 0.0)
                d = min(q, saldo)
                p.at[i,'QTD_PRODUZIR_SOLVER'] = q - d
                saldo -= d
                reduzido += d
        p, _, _, _ = _solver_totais_plano(p)
        return p, reduzido


    def _solver_alocar_gaps(plano, estado, ids_permitidos=None):
        p, tot, rec_used, fer_used = _solver_totais_plano(plano)
        e = estado[['MES_REF','ID_PROD_UNID_FAT','NEC_ATUAL']].merge(tot, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
        e['QTD_PLANO'] = _solver_num(e['QTD_PLANO'])
        e['GAP'] = (e['NEC_ATUAL'] - e['QTD_PLANO']).clip(lower=0.0)
        if ids_permitidos is not None:
            e = e[e['ID_PROD_UNID_FAT'].astype(str).isin(set(map(str, ids_permitidos)))]
        e = e[e['GAP'] > TOL].sort_values(['ID_PROD_UNID_FAT'])
        alocado = 0.0
        logs = []
        for item in e.itertuples(index=False):
            saldo = float(item.GAP)
            alts = ALTS_VALIDAS_LOCAL_M1[ALTS_VALIDAS_LOCAL_M1['ID_PROD_UNID_FAT'].astype(str).eq(str(item.ID_PROD_UNID_FAT))]
            if alts.empty:
                continue
            for a in alts.itertuples(index=False):
                if saldo <= TOL:
                    break
                pcs_h = float(a.PCS_HORA or 0.0)
                rec = str(a.ALOC_REC); fer = str(a.COD_FER_UNID)
                if pcs_h <= TOL:
                    continue
                hr_rec = max(float(CAP_REC_LOCAL_M1.get(rec, 0.0)) - float(rec_used.get(rec, 0.0)), 0.0)
                hr_fer = max(float(CAP_FER_LOCAL_M1.get(fer, 0.0)) - float(fer_used.get(fer, 0.0)), 0.0)
                qtd = min(saldo, hr_rec * pcs_h, hr_fer * pcs_h)
                if qtd <= TOL:
                    continue
                mask = (
                    p['ID_PROD_UNID_FAT'].astype(str).eq(str(item.ID_PROD_UNID_FAT)) &
                    p['ALOC_REC'].astype(str).eq(rec) & p['COD_FER_UNID'].astype(str).eq(fer) &
                    np.isclose(_solver_num(p['PRIOR_MATPAR']), float(a.PRIOR_MATPAR or 0.0)) &
                    np.isclose(_solver_num(p['PRIOR_ROT']), float(a.PRIOR_ROT or 0.0))
                )
                inds = p.index[mask].tolist()
                if not inds:
                    continue
                i = inds[0]
                p.at[i,'QTD_PRODUZIR_SOLVER'] = float(p.at[i,'QTD_PRODUZIR_SOLVER'] or 0.0) + qtd
                hr = qtd / pcs_h
                rec_used[rec] = float(rec_used.get(rec, 0.0)) + hr
                fer_used[fer] = float(fer_used.get(fer, 0.0)) + hr
                saldo -= qtd; alocado += qtd
                logs.append({'ID_PROD_UNID_FAT':str(item.ID_PROD_UNID_FAT),'ALOC_REC':rec,'COD_FER_UNID':fer,'QTD_ALOCADA':qtd,'HR_ALOCADA':hr})
        p, _, _, _ = _solver_totais_plano(p)
        return p, alocado, pd.DataFrame(logs)


    def _solver_classificar_alvos(plano, estado):
        p, tot, rec_used, fer_used = _solver_totais_plano(plano)
        e = estado[['MES_REF','ID_PROD_UNID_FAT','NEC_ATUAL']].merge(tot, on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
        e['QTD_PLANO'] = _solver_num(e['QTD_PLANO'])
        e['GAP'] = (e['NEC_ATUAL'] - e['QTD_PLANO']).clip(lower=0.0)
        alvos = []
        for item in e[e['GAP'] > TOL].itertuples(index=False):
            alts = ALTS_VALIDAS_LOCAL_M1[ALTS_VALIDAS_LOCAL_M1['ID_PROD_UNID_FAT'].astype(str).eq(str(item.ID_PROD_UNID_FAT))]
            if alts.empty:
                continue
            alts = alts.sort_values(['PRIOR_MATPAR','PRIOR_ROT','ALOC_REC','COD_FER_UNID'])
            a0 = alts.iloc[0]
            rec0 = str(a0['ALOC_REC']); fer0 = str(a0['COD_FER_UNID']); pcs0 = float(a0['PCS_HORA'])
            rec_free0 = max(float(CAP_REC_LOCAL_M1.get(rec0,0.0)) - float(rec_used.get(rec0,0.0)), 0.0)
            fers = alts['COD_FER_UNID'].astype(str).unique().tolist()
            fer_comum = len(fers) == 1
            fer_real = fer_comum and max(float(CAP_FER_LOCAL_M1.get(fer0,0.0)) - float(fer_used.get(fer0,0.0)), 0.0) <= TOL_LOCAL_HR
            # REC sempre tem precedencia. Somente um gargalo FER realmente comum e com a REC-ancora livre vai direto para FER.
            if fer_real and rec_free0 > TOL_LOCAL_HR:
                tipo='FER'; restr=fer0
            else:
                tipo='REC'; restr=rec0
            alvos.append({
                'MES_REF':MES_M1_LOCAL, 'ID_PROD_UNID_FAT':str(item.ID_PROD_UNID_FAT),
                'TIPO_RESTRICAO':tipo, 'RESTRICAO':restr, 'ALOC_REC_ANCORA':rec0,
                'COD_FER_ANCORA':fer0, 'PCS_HORA_ANCORA':pcs0, 'GAP_PCS':float(item.GAP),
                'CORTE_HR':float(item.GAP)/pcs0 if pcs0>TOL else 0.0,
            })
        return pd.DataFrame(alvos)


    def _solver_impacto_restricao(tipo, restricao, ids_alvo=None):
        if tipo == 'REC':
            # GAP multirota pertence ao PRODUTO. A REC e somente ancora contabil de fechamento.
            # Portanto, o pool de corte REC contem exclusivamente os produtos com GAP ancorado
            # nesta REC e os PAs que geram demanda estrutural para esses produtos.
            ids_alvo = set(map(str, ids_alvo or []))
            if not ids_alvo:
                return pd.DataFrame(columns=['ID_RAIZ','HR_POR_PCA','TIPO_IMPACTO','PI_IMPACTADO'])
            rotas = ALTS_VALIDAS_LOCAL_M1[
                ALTS_VALIDAS_LOCAL_M1['ALOC_REC'].astype(str).eq(str(restricao))
                & ALTS_VALIDAS_LOCAL_M1['ID_PROD_UNID_FAT'].astype(str).isin(ids_alvo)
            ].copy()
        else:
            # FER e restricao fisica global: todos os itens que disputam a ferramenta participam.
            rotas = ALTS_VALIDAS_LOCAL_M1[ALTS_VALIDAS_LOCAL_M1['COD_FER_UNID'].astype(str).eq(str(restricao))].copy()
        if rotas.empty:
            return pd.DataFrame(columns=['ID_RAIZ','HR_POR_PCA','TIPO_IMPACTO','PI_IMPACTADO'])
        rotas = rotas.sort_values(['ID_PROD_UNID_FAT','PRIOR_MATPAR','PRIOR_ROT']).drop_duplicates('ID_PROD_UNID_FAT', keep='first')
        rotas = rotas[['MES_REF','ID_PROD_UNID_FAT','PCS_HORA']].rename(columns={'ID_PROD_UNID_FAT':'ID_PI','PCS_HORA':'PCS_HORA_PI'})
        rotas['HR_DIRETA'] = np.divide(1.0, _solver_num(rotas['PCS_HORA_PI']).to_numpy(float), out=np.zeros(len(rotas)), where=_solver_num(rotas['PCS_HORA_PI']).to_numpy(float)>TOL)
        direto = rotas.rename(columns={'ID_PI':'ID_RAIZ'})[['ID_RAIZ','HR_DIRETA']].rename(columns={'HR_DIRETA':'HR_POR_PCA'})
        direto['TIPO_IMPACTO']='DIRETO'; direto['PI_IMPACTADO']=direto['ID_RAIZ']
        partes=[direto]
        if not MAPA_ESTRUTURAL_LOCAL_M1.empty:
            ind = MAPA_ESTRUTURAL_LOCAL_M1.merge(rotas[['MES_REF','ID_PI','PCS_HORA_PI']], left_on=['MES_REF','ID_FILHO'], right_on=['MES_REF','ID_PI'], how='inner')
            if not ind.empty:
                ind['HR_POR_PCA'] = _solver_num(ind['FATOR_ESTRUTURAL']) / _solver_num(ind['PCS_HORA_PI']).replace(0,np.nan)
                ind['HR_POR_PCA'] = _solver_num(ind['HR_POR_PCA'])
                ind = ind[['ID_RAIZ','HR_POR_PCA','ID_PI']].rename(columns={'ID_PI':'PI_IMPACTADO'})
                ind['TIPO_IMPACTO']='INDIRETO'
                partes.append(ind)
        imp=pd.concat(partes,ignore_index=True,sort=False)
        if imp.empty:
            return imp
        # Agrega coeficiente; preserva indicacao se a origem tem efeito direto, indireto ou ambos.
        tipo_map = imp.groupby('ID_RAIZ')['TIPO_IMPACTO'].agg(lambda x: 'DIRETO_E_INDIRETO' if len(set(x))>1 else list(x)[0])
        pi_map = imp.groupby('ID_RAIZ')['PI_IMPACTADO'].agg(lambda x: '|'.join(sorted(set(map(str,x)))))
        hr = imp.groupby('ID_RAIZ',as_index=False)['HR_POR_PCA'].sum()
        hr['TIPO_IMPACTO']=hr['ID_RAIZ'].map(tipo_map)
        hr['PI_IMPACTADO']=hr['ID_RAIZ'].map(pi_map)
        return hr[hr['HR_POR_PCA']>TOL].reset_index(drop=True)

    SEQUENCIA_CORTE_LOCAL_M1 = 0
    EVENTOS_CORTE_LOCAL_M1 = []
    LOG_GIROS_LOCAL_M1 = []

    # ------------------------------------------------------------
    # V66_29: congelar a decisao de alocacao por componente ANTES
    # da reconciliacao local. Cortes iniciais seguem recuperaveis
    # se surgir capacidade factivel; a matriz continua livre para
    # resolver deltas estruturais PA->PI e gargalos que restarem.
    # ------------------------------------------------------------
    # O motor ja consumiu as horas REC/FER por classe e faixa.
    # Nao se pode deixar a reconciliacao redistribuir entre produtos
    # as faixas comerciais que a alocacao prioritaria ja definiu.
    def _v6629_registrar_priorizacao_inicial(comp_state, plano):
        global SEQUENCIA_CORTE_LOCAL_M1
        atendimento = plano.groupby('ID_PROD_UNID_FAT')['QTD_PRODUZIR_SOLVER'].sum().to_dict()
        nec_direta = (NEC_BASE_TECNICA_M1.groupby('ID_PROD_UNID_FAT')['NEC_DIRETA_PCS']
                       .max().to_dict())
        original = comp_state[['ID_PROD_UNID_FAT','NEC_TECNICA_INICIAL']].copy()
        original['ATENDIMENTO'] = original['ID_PROD_UNID_FAT'].map(atendimento).fillna(0.0)
        original['GAP'] = (original['NEC_TECNICA_INICIAL'] - original['ATENDIMENTO']).clip(lower=0.0)
        original['NEC_DIRETA'] = original['ID_PROD_UNID_FAT'].map(nec_direta).fillna(0.0)
        total_pre_corte = 0.0
        for row in original.itertuples(index=False):
            # O excedente de NEC estrutural so pode ser resolvido na propria
            # Matriz, propagando o corte aos pais geradores do insumo.
            saldo = min(float(row.GAP), float(row.NEC_DIRETA))
            if saldo <= TOL:
                continue
            idx = comp_state.index[comp_state['ID_PROD_UNID_FAT'].eq(row.ID_PROD_UNID_FAT)]
            if len(idx) != 1:
                raise ValueError('COMPONENTE_V6629_SEM_CHAVE_UNICA')
            i = idx[0]
            flag = str(comp_state.at[i,'MESMA_REG']).upper()
            for ordem, flag_criterio, componente, nome in CRITERIOS_MATRIZ_LOCAL_M1:
                if saldo <= TOL:
                    break
                if flag != flag_criterio:
                    continue
                antes = float(comp_state.at[i,componente] or 0.0)
                pcs = min(saldo, antes)
                if pcs <= TOL:
                    continue
                comp_state.at[i,componente] = max(0.0, antes - pcs)
                saldo -= pcs
                total_pre_corte += pcs
                SEQUENCIA_CORTE_LOCAL_M1 += 1
                EVENTOS_CORTE_LOCAL_M1.append({
                    'SEQ': SEQUENCIA_CORTE_LOCAL_M1, 'MES_REF': MES_M1_LOCAL,
                    'RODADA': 'ALO_PRIORIZADA', 'ID_PROD_UNID_FAT': str(row.ID_PROD_UNID_FAT),
                    'COD_PROD': str(comp_state.at[i,'COD_PROD']),
                    'UNID_FAT': str(comp_state.at[i,'UNID_FAT']),
                    'ORIGEM_RESTRICAO': 'ALOCACAO_PRIORIZADA', 'ALOC_REC': '',
                    'COD_FER_UNID': '', 'CRITERIO_ORDEM': ordem,
                    'CRITERIO_MATRIZ': nome, 'COMPONENTE_CORTE': componente,
                    'VALOR_ANTES_PCS': antes, 'PCS_CORTE_APLICADO': pcs,
                    'PCS_RECUPERADO': 0.0, 'PCS_CORTE_RECUPERAVEL': pcs,
                    'CORTE_HR': 0.0, 'TIPO_IMPACTO': 'DIRETO',
                    'PA_GERADOR': '', 'PI_IMPACTADO': ''
                })
            if saldo > 1e-5:
                raise RuntimeError('ALOCACAO_PRIORIZADA_SEM_COMPONENTE_PARA_FECHAR_GAP: '
                                   + str(row.ID_PROD_UNID_FAT) + '=' + str(saldo))
        return comp_state, total_pre_corte

    # As cinco parcelas sao floats porque o corte tecnico e continuo.
    for _col in MAPA_ORI_LOCAL_M1.keys():
        COMPONENTES_LOCAL_M1[_col] = _solver_num(COMPONENTES_LOCAL_M1[_col]).astype(float)
    COMPONENTES_LOCAL_M1, TOTAL_PRECORTE_ALOCACAO_V6629 = _v6629_registrar_priorizacao_inicial(
        COMPONENTES_LOCAL_M1, PLANO_LOCAL_M1)
    print(f'       ALOCACAO PRIORIZADA CONGELADA: {TOTAL_PRECORTE_ALOCACAO_V6629:.3f} pcs nao atendidas no M1')




    def _solver_cortar_restricao(comp_state, plano, estado, tipo, restricao, corte_hr_alvo, rodada_label, ids_alvo=None):
        global SEQUENCIA_CORTE_LOCAL_M1
        imp = _solver_impacto_restricao(tipo, restricao, ids_alvo=ids_alvo)
        if imp.empty:
            return comp_state, plano, estado, 0.0, corte_hr_alvo
        saldo = float(corte_hr_alvo)
        hr_aplicado_total = 0.0
        for ordem, mesma_reg, componente, criterio in CRITERIOS_MATRIZ_LOCAL_M1:
            if saldo <= TOL_LOCAL_HR:
                break
            base = comp_state[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_FAT','MESMA_REG',componente]].copy()
            base = base[base['MESMA_REG'].astype(str).str.upper().eq(mesma_reg)]
            base = base.merge(imp, left_on='ID_PROD_UNID_FAT', right_on='ID_RAIZ', how='inner')
            base = base.merge(estado[['MES_REF','ID_PROD_UNID_FAT','NEC_ATUAL']], on=['MES_REF','ID_PROD_UNID_FAT'], how='left')
            base['PCS_DISP'] = np.minimum(_solver_num(base[componente]).to_numpy(float), _solver_num(base['NEC_ATUAL']).to_numpy(float))
            base['HR_DISP'] = base['PCS_DISP'] * _solver_num(base['HR_POR_PCA'])
            base = base[(base['PCS_DISP']>TOL) & (base['HR_DISP']>TOL)].copy()
            if base.empty:
                continue
            hr_criterio = float(base['HR_DISP'].sum())
            fator = 1.0 if hr_criterio <= saldo + TOL_LOCAL_HR else saldo / hr_criterio
            base['PCS_CORTE'] = base['PCS_DISP'] * fator
            base['HR_CORTE'] = base['HR_DISP'] * fator
            # Aplica de forma vetorizada pela chave item; cada item aparece uma vez no alvo agregado.
            cortes_item = base.groupby('ID_PROD_UNID_FAT',as_index=False).agg(PCS_CORTE=('PCS_CORTE','sum'),HR_CORTE=('HR_CORTE','sum'))
            pos = comp_state.reset_index().merge(cortes_item,on='ID_PROD_UNID_FAT',how='inner')[['index','ID_PROD_UNID_FAT','PCS_CORTE','HR_CORTE']]
            for r in pos.itertuples(index=False):
                i=int(r.index); antes=float(comp_state.at[i,componente] or 0.0); pcs=min(float(r.PCS_CORTE),antes)
                if pcs<=TOL: continue
                # Reconverte HR conforme o corte realmente limitado.
                hr_coef=float(imp.loc[imp['ID_RAIZ'].astype(str).eq(str(r.ID_PROD_UNID_FAT)),'HR_POR_PCA'].sum())
                hr=pcs*hr_coef
                comp_state.at[i,componente]=max(antes-pcs,0.0)
                meta=imp[imp['ID_RAIZ'].astype(str).eq(str(r.ID_PROD_UNID_FAT))].iloc[0]
                SEQUENCIA_CORTE_LOCAL_M1 += 1
                EVENTOS_CORTE_LOCAL_M1.append({
                    'SEQ':SEQUENCIA_CORTE_LOCAL_M1,'MES_REF':MES_M1_LOCAL,'RODADA':rodada_label,
                    'ID_PROD_UNID_FAT':str(r.ID_PROD_UNID_FAT),'COD_PROD':str(comp_state.at[i,'COD_PROD']),
                    'UNID_FAT':str(comp_state.at[i,'UNID_FAT']),'ORIGEM_RESTRICAO':'ALOC_REC' if tipo=='REC' else 'COD_FER_UNID',
                    'ALOC_REC':str(restricao) if tipo=='REC' else '', 'COD_FER_UNID':str(restricao) if tipo=='FER' else '',
                    'CRITERIO_ORDEM':ordem,'CRITERIO_MATRIZ':criterio,'COMPONENTE_CORTE':componente,
                    'VALOR_ANTES_PCS':antes,'PCS_CORTE_APLICADO':pcs,'PCS_RECUPERADO':0.0,
                    'PCS_CORTE_RECUPERAVEL':pcs,'CORTE_HR':hr,'TIPO_IMPACTO':str(meta['TIPO_IMPACTO']),
                    'PA_GERADOR':str(comp_state.at[i,'COD_PROD']) if 'INDIRETO' in str(meta['TIPO_IMPACTO']) else '',
                    'PI_IMPACTADO':str(meta['PI_IMPACTADO']),
                })
                hr_aplicado_total += hr
                saldo=max(saldo-hr,0.0)
            estado=_solver_estado_nec(comp_state)
            plano,_=_solver_reduzir_plano_ao_nec(plano,estado)
        return comp_state, plano, estado, hr_aplicado_total, saldo


    # -----------------------------
    # Loop local REC -> FER, sem rerodagem global
    # -----------------------------
    ESTADO_NEC_LOCAL_M1 = _solver_estado_nec(COMPONENTES_LOCAL_M1)
    PLANO_LOCAL_M1, _ = _solver_reduzir_plano_ao_nec(PLANO_LOCAL_M1, ESTADO_NEC_LOCAL_M1)

    for giro in range(1, MAX_GIROS_LOCAL_M1 + 1):
        t_giro=time.perf_counter()
        # Primeiro reaproveita capacidade ja livre sem alterar demanda.
        PLANO_LOCAL_M1, qtd_realoc, _log_realoc = _solver_alocar_gaps(PLANO_LOCAL_M1, ESTADO_NEC_LOCAL_M1)
        alvos = _solver_classificar_alvos(PLANO_LOCAL_M1, ESTADO_NEC_LOCAL_M1)
        if alvos.empty:
            LOG_GIROS_LOCAL_M1.append({'GIRO':giro,'FASE':'FECHOU','QTD_RESTRICOES':0,'CORTE_HR_ALVO':0.0,'CORTE_HR_APLICADO':0.0,'QTD_REALOCADA':qtd_realoc,'TEMPO_SEG':time.perf_counter()-t_giro})
            break

        # Regra de precedencia: se existir qualquer REC, FER aguarda.
        fase = 'REC' if (alvos['TIPO_RESTRICAO']=='REC').any() else 'FER'
        alvo_fase = alvos[alvos['TIPO_RESTRICAO'].eq(fase)].copy()
        resumo = alvo_fase.groupby('RESTRICAO',as_index=False)['CORTE_HR'].sum().sort_values('CORTE_HR',ascending=False)
        total_alvo=float(resumo['CORTE_HR'].sum())
        total_aplic=0.0
        total_saldo=0.0
        for n, rr in enumerate(resumo.itertuples(index=False), start=1):
            ids_alvo_restr = (
                alvo_fase.loc[
                    alvo_fase['RESTRICAO'].astype(str).eq(str(rr.RESTRICAO)),
                    'ID_PROD_UNID_FAT'
                ]
                .astype(str)
                .drop_duplicates()
                .tolist()
            )
            COMPONENTES_LOCAL_M1, PLANO_LOCAL_M1, ESTADO_NEC_LOCAL_M1, hr_apl, saldo = _solver_cortar_restricao(
                COMPONENTES_LOCAL_M1, PLANO_LOCAL_M1, ESTADO_NEC_LOCAL_M1,
                fase, str(rr.RESTRICAO), float(rr.CORTE_HR), f'{giro}_{fase}_{n}',
                ids_alvo=ids_alvo_restr
            )
            total_aplic += hr_apl; total_saldo += saldo

        LOG_GIROS_LOCAL_M1.append({'GIRO':giro,'FASE':fase,'QTD_RESTRICOES':len(resumo),'QTD_ITENS_ALVO':int(alvo_fase['ID_PROD_UNID_FAT'].nunique()),'CORTE_HR_ALVO':total_alvo,'CORTE_HR_APLICADO':total_aplic,'SALDO_HR_PLANEJADO':total_saldo,'QTD_REALOCADA':qtd_realoc,'TEMPO_SEG':time.perf_counter()-t_giro})
        if total_aplic <= TOL_LOCAL_HR:
            detalhe = alvo_fase[['ID_PROD_UNID_FAT','RESTRICAO','GAP_PCS','CORTE_HR']].to_dict('records')
            raise RuntimeError(f'RECONCILIACAO_LOCAL_SEM_PROGRESSO: mes={MES_M1_LOCAL}; giro={giro}; fase={fase}; alvo={total_alvo:.9f}h; itens={detalhe}')
    else:
        raise RuntimeError('RECONCILIACAO_LOCAL_M1_MAX_ITER: blindagem atingida sem convergencia valida')

    # Valida que o estado tecnico cortado cabe integralmente no plano atual.
    PLANO_LOCAL_M1, _, _ = _solver_alocar_gaps(PLANO_LOCAL_M1, ESTADO_NEC_LOCAL_M1)
    _alvos_pos_corte = _solver_classificar_alvos(PLANO_LOCAL_M1, ESTADO_NEC_LOCAL_M1)
    if not _alvos_pos_corte.empty:
        raise RuntimeError(f'FECHAMENTO_TECNICO_LOCAL_M1_AINDA_COM_GAP: itens={len(_alvos_pos_corte)}; hr={float(_alvos_pos_corte.CORTE_HR.sum()):.9f}')

    # -----------------------------
    # Recuperacao local: ultimo corte primeiro, sem reabrir Solver/rotas globalmente
    # -----------------------------
    def _solver_testar_restauracao(comp_atual, plano_atual, estado_atual, evento, qtd_restaurar):
        if qtd_restaurar <= TOL:
            return False, comp_atual, plano_atual, estado_atual
        c=comp_atual.copy(); p=plano_atual.copy()
        mask=c['ID_PROD_UNID_FAT'].astype(str).eq(str(evento['ID_PROD_UNID_FAT']))
        inds=c.index[mask].tolist()
        if not inds:
            return False, comp_atual, plano_atual, estado_atual
        i=inds[0]; comp=evento['COMPONENTE_CORTE']
        limite=float(c.at[i,MAPA_ORI_LOCAL_M1[comp]])
        c.at[i,comp]=min(float(c.at[i,comp])+qtd_restaurar,limite)
        e=_solver_estado_nec(c)
        p,_=_solver_reduzir_plano_ao_nec(p,e)
        p,_,_=_solver_alocar_gaps(p,e)
        gaps=_solver_classificar_alvos(p,e)
        if not gaps.empty:
            return False, comp_atual, plano_atual, estado_atual
        # capacidade dura conferida diretamente
        pp,_,ru,fu=_solver_totais_plano(p)
        exc_rec=max([float(ru.get(k,0.0))-float(v) for k,v in CAP_REC_LOCAL_M1.items()] + [0.0])
        exc_fer=max([float(fu.get(k,0.0))-float(v) for k,v in CAP_FER_LOCAL_M1.items()] + [0.0])
        if exc_rec > TOL_LOCAL_HR or exc_fer > TOL_LOCAL_HR:
            return False, comp_atual, plano_atual, estado_atual
        return True,c,pp,e

    # eventos permanecem em lista de dicts; restaura em LIFO real.
    # v66_26: antes das 14 bissecoes, testa a menor quantidade que a propria
    # v66_25 alcancaria no caminho em que todas as bissecoes falham: saldo / 2^N.
    # Se essa quantidade (> TOL) tambem nao cabe, quantidades maiores nao podem
    # ser viaveis no problema continuo; portanto a bissecao terminaria em zero.
    PCS_RECUPERADO_LOCAL_M1 = 0.0
    _T0_RECUPERACAO_V6626 = time.perf_counter()
    QTD_TESTES_RESTAURACAO_M1 = 0
    QTD_EVENTOS_BISSECAO_M1 = 0
    QTD_EVENTOS_BISSECAO_PULADA_ZERO_M1 = 0

    # Otimizacao conservadora: sem REC e FER livres na mesma rota
    # nenhuma restauracao inicial de demanda pode ser atendida pelo
    # reconciliador local. Aplica-se apenas aos pre-cortes v66_29.
    _, _, _USO_REC_RESTAURACAO, _USO_FER_RESTAURACAO = _solver_totais_plano(PLANO_LOCAL_M1)
    _ROTAS_RESTAURACAO = {
        item: g[['ALOC_REC','COD_FER_UNID']].drop_duplicates().itertuples(index=False,name=None)
        for item,g in ALTS_VALIDAS_LOCAL_M1.groupby('ID_PROD_UNID_FAT')
    }
    # Iteradores nao devem ser consumidos: materializar cada lista uma unica vez.
    _ROTAS_RESTAURACAO = {item:list(alts) for item,alts in _ROTAS_RESTAURACAO.items()}

    for idx_evento in range(len(EVENTOS_CORTE_LOCAL_M1)-1,-1,-1):
        ev=EVENTOS_CORTE_LOCAL_M1[idx_evento]
        saldo=float(ev['PCS_CORTE_RECUPERAVEL'])
        if saldo<=TOL:
            continue
        if ev.get('ORIGEM_RESTRICAO') == 'ALOCACAO_PRIORIZADA':
            rotas_validas = _ROTAS_RESTAURACAO.get(str(ev['ID_PROD_UNID_FAT']), [])
            tem_capacidade = any(
                (float(CAP_REC_LOCAL_M1.get(str(rec), 0.0)) - float(_USO_REC_RESTAURACAO.get(str(rec), 0.0)) > TOL_LOCAL_HR)
                and (float(CAP_FER_LOCAL_M1.get(str(fer), 0.0)) - float(_USO_FER_RESTAURACAO.get(str(fer), 0.0)) > TOL_LOCAL_HR)
                for rec, fer in rotas_validas
            )
            if not tem_capacidade:
                continue

        QTD_TESTES_RESTAURACAO_M1 += 1
        ok,c2,p2,e2=_solver_testar_restauracao(
            COMPONENTES_LOCAL_M1,PLANO_LOCAL_M1,ESTADO_NEC_LOCAL_M1,ev,saldo
        )
        melhor=saldo if ok else 0.0
        melhor_estado=(c2,p2,e2) if ok else None

        if not ok:
            # Probe conservador: exatamente o menor mid que a bissecao original
            # alcancaria apos MAX_BISSECAO_RECUP_M1 falhas consecutivas.
            qtd_probe_min = saldo / (2.0 ** MAX_BISSECAO_RECUP_M1)
            pode_pular_bissecao = False

            if qtd_probe_min > TOL:
                QTD_TESTES_RESTAURACAO_M1 += 1
                ok_probe,_,_,_=_solver_testar_restauracao(
                    COMPONENTES_LOCAL_M1,PLANO_LOCAL_M1,ESTADO_NEC_LOCAL_M1,ev,qtd_probe_min
                )
                if not ok_probe:
                    pode_pular_bissecao = True
                    QTD_EVENTOS_BISSECAO_PULADA_ZERO_M1 += 1

            if not pode_pular_bissecao:
                QTD_EVENTOS_BISSECAO_M1 += 1
                lo=0.0; hi=saldo
                for _ in range(MAX_BISSECAO_RECUP_M1):
                    mid=(lo+hi)/2.0
                    QTD_TESTES_RESTAURACAO_M1 += 1
                    okm,cm,pm,em=_solver_testar_restauracao(
                        COMPONENTES_LOCAL_M1,PLANO_LOCAL_M1,ESTADO_NEC_LOCAL_M1,ev,mid
                    )
                    if okm:
                        lo=mid; melhor=mid; melhor_estado=(cm,pm,em)
                    else:
                        hi=mid

        if melhor_estado is not None and melhor>TOL:
            COMPONENTES_LOCAL_M1,PLANO_LOCAL_M1,ESTADO_NEC_LOCAL_M1=melhor_estado
            ev['PCS_RECUPERADO']=melhor
            ev['PCS_CORTE_RECUPERAVEL']=max(float(ev['PCS_CORTE_APLICADO'])-melhor,0.0)
            PCS_RECUPERADO_LOCAL_M1 += melhor
            _plano_rest, _, _USO_REC_RESTAURACAO, _USO_FER_RESTAURACAO = _solver_totais_plano(PLANO_LOCAL_M1)

    TEMPO_RECUPERACAO_LOCAL_M1 = time.perf_counter() - _T0_RECUPERACAO_V6626
    LOG_RECUPERACAO_PERFORMANCE_M1 = {
        'EVENTOS_CORTE': int(len(EVENTOS_CORTE_LOCAL_M1)),
        'TESTES_RESTAURACAO': int(QTD_TESTES_RESTAURACAO_M1),
        'EVENTOS_COM_BISSECAO': int(QTD_EVENTOS_BISSECAO_M1),
        'EVENTOS_BISSECAO_PULADA_ZERO': int(QTD_EVENTOS_BISSECAO_PULADA_ZERO_M1),
        'TEMPO_RECUPERACAO_SEG': float(TEMPO_RECUPERACAO_LOCAL_M1),
        'PCS_RECUPERADO': float(PCS_RECUPERADO_LOCAL_M1),
    }

    # Saldo nao recuperado vira CORTE FINAL. Nao existe saldo recuperavel depois deste marco.
    for ev in EVENTOS_CORTE_LOCAL_M1:
        ev['PCS_CORTE_FINAL']=max(float(ev['PCS_CORTE_APLICADO'])-float(ev['PCS_RECUPERADO']),0.0)
        ev['STATUS_CORTE']='CORTE_FINAL' if ev['PCS_CORTE_FINAL']>TOL else 'RECUPERADO_TOTAL'
        ev['PCS_CORTE_RECUPERAVEL']=0.0

    REGISTRO_CORTE_RECUPERAVEL_M1=pd.DataFrame(EVENTOS_CORTE_LOCAL_M1)

    # -----------------------------
    # Materializa estado final tecnico SEM nova chamada ao motor global
    # -----------------------------
    # Atualiza os cinco componentes em todas as linhas do LTP do item.
    BD_M1_FINAL_MATRIZ=BD_M1_BASE_LOCAL.copy()
    _comp_final=COMPONENTES_LOCAL_M1[['MES_REF','ID_PROD_UNID_FAT']+list(MAPA_COMPONENTE_LTP_LOCAL_M1.keys())+list(MAPA_ORI_LOCAL_M1.values())+['ORI_NEC']].copy()
    BD_M1_FINAL_MATRIZ=BD_M1_FINAL_MATRIZ.merge(_comp_final,on=['MES_REF','ID_PROD_UNID_FAT'],how='left',suffixes=('','_LOCAL'))
    for comp,col in MAPA_COMPONENTE_LTP_LOCAL_M1.items():
        BD_M1_FINAL_MATRIZ[col]=_solver_num(BD_M1_FINAL_MATRIZ[comp])
    for comp,ori in MAPA_ORI_LOCAL_M1.items():
        BD_M1_FINAL_MATRIZ[ori]=_solver_num(BD_M1_FINAL_MATRIZ[ori])
    BD_M1_FINAL_MATRIZ['ORI_NEC']=_solver_num(BD_M1_FINAL_MATRIZ['ORI_NEC'])
    BD_M1_FINAL_MATRIZ=BD_M1_FINAL_MATRIZ.drop(columns=[c for c in MAPA_COMPONENTE_LTP_LOCAL_M1 if c in BD_M1_FINAL_MATRIZ.columns],errors='ignore')

    # NEC final vetorizada por item.
    ESTADO_NEC_LOCAL_M1=_solver_estado_nec(COMPONENTES_LOCAL_M1)
    BD_M1_FINAL_MATRIZ=BD_M1_FINAL_MATRIZ.merge(ESTADO_NEC_LOCAL_M1[['MES_REF','ID_PROD_UNID_FAT','NEC_ATUAL']],on=['MES_REF','ID_PROD_UNID_FAT'],how='left')
    BD_M1_FINAL_MATRIZ['NEC_PCS']=_solver_num(BD_M1_FINAL_MATRIZ['NEC_ATUAL'])
    BD_M1_FINAL_MATRIZ=BD_M1_FINAL_MATRIZ.drop(columns=['NEC_ATUAL'],errors='ignore')

    NEC_CONSOLIDADA_FINAL_M1=NEC_BASE_TECNICA_M1.copy().merge(
        ESTADO_NEC_LOCAL_M1[['MES_REF','ID_PROD_UNID_FAT','NEC_ATUAL','RED_DIRETA_BRUTA','RED_ESTRUTURAL']],
        on=['MES_REF','ID_PROD_UNID_FAT'],how='left'
    )
    NEC_CONSOLIDADA_FINAL_M1['NEC_ATUAL']=_solver_num(NEC_CONSOLIDADA_FINAL_M1['NEC_ATUAL'])
    for c in ['NEC_PCS_CONSOLIDADA','NEC_PCS']:
        if c in NEC_CONSOLIDADA_FINAL_M1.columns:
            NEC_CONSOLIDADA_FINAL_M1[c]=NEC_CONSOLIDADA_FINAL_M1['NEC_ATUAL']
    # Decomposicao apenas diagnostica, proporcional, sem prioridade de estoque.
    if 'NEC_DIRETA_PCS' in NEC_CONSOLIDADA_FINAL_M1.columns and 'NEC_COMP_PCS' in NEC_CONSOLIDADA_FINAL_M1.columns:
        d0=_solver_num(NEC_CONSOLIDADA_FINAL_M1['NEC_DIRETA_PCS'])
        c0=_solver_num(NEC_CONSOLIDADA_FINAL_M1['NEC_COMP_PCS'])
        soma=d0+c0
        fator=np.divide(NEC_CONSOLIDADA_FINAL_M1['NEC_ATUAL'].to_numpy(float),soma.to_numpy(float),out=np.zeros(len(soma)),where=soma.to_numpy(float)>TOL)
        fator=np.minimum(fator,1.0)
        NEC_CONSOLIDADA_FINAL_M1['NEC_DIRETA_PCS']=d0*fator
        NEC_CONSOLIDADA_FINAL_M1['NEC_COMP_PCS']=c0*fator
        if 'NEC_EXPLODIDA_PCS' in NEC_CONSOLIDADA_FINAL_M1.columns:
            NEC_CONSOLIDADA_FINAL_M1['NEC_EXPLODIDA_PCS']=NEC_CONSOLIDADA_FINAL_M1['NEC_COMP_PCS']

    # Plano tecnico final: remove rotas auxiliares que continuaram zeradas.
    PLANO_LOCAL_M1,_tot_final_local,_ru,_fu=_solver_totais_plano(PLANO_LOCAL_M1)
    PLANO_LOCAL_M1=PLANO_LOCAL_M1[PLANO_LOCAL_M1['_LINHA_ORIGINAL'].fillna(False) | (_solver_num(PLANO_LOCAL_M1['QTD_PRODUZIR_SOLVER'])>TOL)].copy()
    PLANO_LOCAL_M1=PLANO_LOCAL_M1.drop(columns=['_LINHA_ORIGINAL'],errors='ignore').reset_index(drop=True)

    # Atualiza campos de NEC do plano sem vazar estruturas internas novas.
    _meta_nec_final=NEC_CONSOLIDADA_FINAL_M1[['MES_REF','ID_PROD_UNID_FAT']+[c for c in ['NEC_DIRETA_PCS','NEC_COMP_PCS','NEC_EXPLODIDA_PCS','NEC_PCS_CONSOLIDADA','NEC_ATUAL'] if c in NEC_CONSOLIDADA_FINAL_M1.columns]].drop_duplicates(['MES_REF','ID_PROD_UNID_FAT'])
    PLANO_LOCAL_M1=PLANO_LOCAL_M1.drop(columns=[c for c in _meta_nec_final.columns if c not in ['MES_REF','ID_PROD_UNID_FAT'] and c in PLANO_LOCAL_M1.columns],errors='ignore').merge(_meta_nec_final,on=['MES_REF','ID_PROD_UNID_FAT'],how='left')
    if 'NEC_ATUAL' in PLANO_LOCAL_M1.columns:
        PLANO_LOCAL_M1['NEC_PCS']=_solver_num(PLANO_LOCAL_M1['NEC_ATUAL'])
        PLANO_LOCAL_M1=PLANO_LOCAL_M1.drop(columns='NEC_ATUAL')

    # ORI_* no plano para comparacao antes x depois conforme contrato da memoria.
    _ori_plan=COMPONENTES_LOCAL_M1[['MES_REF','ID_PROD_UNID_FAT']+list(MAPA_ORI_LOCAL_M1.values())+['ORI_NEC']+list(MAPA_ORI_LOCAL_M1.keys())].copy()
    PLANO_LOCAL_M1=PLANO_LOCAL_M1.merge(_ori_plan,on=['MES_REF','ID_PROD_UNID_FAT'],how='left')
    for comp,col in MAPA_COMPONENTE_LTP_LOCAL_M1.items():
        if comp in PLANO_LOCAL_M1.columns:
            PLANO_LOCAL_M1=PLANO_LOCAL_M1.rename(columns={comp:comp})

    # Recalcula nao atendimento sobre a NEC final.
    _atendido=PLANO_LOCAL_M1.groupby(['MES_REF','ID_PROD_UNID_FAT'],as_index=False)['QTD_PRODUZIR_SOLVER'].sum().rename(columns={'QTD_PRODUZIR_SOLVER':'QTD_ATENDIDA_SOLVER'})
    NAO_ATEND_FINAL_M1=NEC_CONSOLIDADA_FINAL_M1.merge(_atendido,on=['MES_REF','ID_PROD_UNID_FAT'],how='left')
    NAO_ATEND_FINAL_M1['QTD_ATENDIDA_SOLVER']=_solver_num(NAO_ATEND_FINAL_M1['QTD_ATENDIDA_SOLVER'])
    NAO_ATEND_FINAL_M1['QTD_NAO_ATEND_SOLVER']=(NAO_ATEND_FINAL_M1['NEC_ATUAL']-NAO_ATEND_FINAL_M1['QTD_ATENDIDA_SOLVER']).clip(lower=0.0)

    # Resumos de capacidade do estado tecnico final.
    _prod_pos=PLANO_LOCAL_M1[_solver_num(PLANO_LOCAL_M1['QTD_PRODUZIR_SOLVER'])>TOL].copy()
    RESUMO_REC_FINAL_M1=_prod_pos.groupby('ALOC_REC',as_index=False).agg(HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER','sum')) if not _prod_pos.empty else pd.DataFrame(columns=['ALOC_REC','HR_PRODUZIR_SOLVER'])
    RESUMO_FER_FINAL_M1=_prod_pos.groupby('COD_FER_UNID',as_index=False).agg(HR_PRODUZIR_SOLVER=('HR_PRODUZIR_SOLVER','sum')) if not _prod_pos.empty else pd.DataFrame(columns=['COD_FER_UNID','HR_PRODUZIR_SOLVER'])
    RESUMO_REC_FINAL_M1['HOR_REC'] = RESUMO_REC_FINAL_M1['ALOC_REC'].map(CAP_REC_LOCAL_M1).fillna(0.0)
    RESUMO_FER_FINAL_M1['HOR_FER'] = RESUMO_FER_FINAL_M1['COD_FER_UNID'].map(CAP_FER_LOCAL_M1).fillna(0.0)
    if not RESUMO_REC_FINAL_M1.empty:
        RESUMO_REC_FINAL_M1['ESTOURO_HR_RECURSO']=(RESUMO_REC_FINAL_M1['HR_PRODUZIR_SOLVER']-RESUMO_REC_FINAL_M1['HOR_REC']).clip(lower=0.0)
    if not RESUMO_FER_FINAL_M1.empty:
        RESUMO_FER_FINAL_M1['ESTOURO_HR_FER']=(RESUMO_FER_FINAL_M1['HR_PRODUZIR_SOLVER']-RESUMO_FER_FINAL_M1['HOR_FER']).clip(lower=0.0)

    # Auditoria externa: somente corte liquido final apos recuperacao.
    AUDITORIA_MATRIZ_CORTES_TECNICA=pd.DataFrame(EVENTOS_CORTE_LOCAL_M1)
    if not AUDITORIA_MATRIZ_CORTES_TECNICA.empty:
        AUDITORIA_MATRIZ_CORTES_TECNICA['CORTE_PCS']=_solver_num(AUDITORIA_MATRIZ_CORTES_TECNICA['PCS_CORTE_FINAL'])
        _ratio_final=np.divide(
            AUDITORIA_MATRIZ_CORTES_TECNICA['CORTE_PCS'].to_numpy(float),
            _solver_num(AUDITORIA_MATRIZ_CORTES_TECNICA['PCS_CORTE_APLICADO']).to_numpy(float),
            out=np.zeros(len(AUDITORIA_MATRIZ_CORTES_TECNICA),dtype=float),
            where=_solver_num(AUDITORIA_MATRIZ_CORTES_TECNICA['PCS_CORTE_APLICADO']).to_numpy(float)>TOL
        )
        AUDITORIA_MATRIZ_CORTES_TECNICA['CORTE_HR']=_solver_num(AUDITORIA_MATRIZ_CORTES_TECNICA['CORTE_HR'])*_ratio_final
        AUDITORIA_MATRIZ_CORTES_TECNICA=AUDITORIA_MATRIZ_CORTES_TECNICA[AUDITORIA_MATRIZ_CORTES_TECNICA['CORTE_PCS']>TOL].copy()
        AUDITORIA_MATRIZ_CORTES_TECNICA['VALOR_DEPOIS_PCS']=(AUDITORIA_MATRIZ_CORTES_TECNICA['VALOR_ANTES_PCS']-AUDITORIA_MATRIZ_CORTES_TECNICA['CORTE_PCS']).clip(lower=0.0)
        _cols_aud=['MES_REF','RODADA','COD_PROD','UNID_FAT','ORIGEM_RESTRICAO','ALOC_REC','COD_FER_UNID','CRITERIO_MATRIZ','COMPONENTE_CORTE','VALOR_ANTES_PCS','CORTE_PCS','VALOR_DEPOIS_PCS','CORTE_HR','TIPO_IMPACTO','PA_GERADOR','PI_IMPACTADO']
        AUDITORIA_MATRIZ_CORTES_TECNICA=AUDITORIA_MATRIZ_CORTES_TECNICA[_cols_aud].reset_index(drop=True)
    else:
        AUDITORIA_MATRIZ_CORTES_TECNICA=pd.DataFrame(columns=['MES_REF','RODADA','COD_PROD','UNID_FAT','ORIGEM_RESTRICAO','ALOC_REC','COD_FER_UNID','CRITERIO_MATRIZ','COMPONENTE_CORTE','VALOR_ANTES_PCS','CORTE_PCS','VALOR_DEPOIS_PCS','CORTE_HR','TIPO_IMPACTO','PA_GERADOR','PI_IMPACTADO'])

    # Estado tecnico final publicado para a etapa industrial.
    RESULTADO_V66_FINAL_M1=dict(RESULTADO_BASE_TECNICO_M1)
    RESULTADO_V66_FINAL_M1['plano_producao']=PLANO_LOCAL_M1.copy()
    RESULTADO_V66_FINAL_M1['nao_atend_item']=NAO_ATEND_FINAL_M1.copy()
    RESULTADO_V66_FINAL_M1['nec_consolidada_solver']=NEC_CONSOLIDADA_FINAL_M1.copy()
    RESULTADO_V66_FINAL_M1['estrutura_explodida']=ESTRUTURA_BASE_TECNICA_M1.copy()
    RESULTADO_V66_FINAL_M1['rastro_rateio']=RASTRO_RATEIO_BASE_M1.copy()
    RESULTADO_V66_FINAL_M1['resumo_recursos']=RESUMO_REC_FINAL_M1.copy()
    RESULTADO_V66_FINAL_M1['resumo_ferramentas']=RESUMO_FER_FINAL_M1.copy()

    # Demanda final para calculo do estado M1 sem rerodagem.
    _dem=BD_M1_FINAL_MATRIZ[['MES_REF','ID_PROD_UNID_FAT','COD_PROD','UNID_FAT']].drop_duplicates().merge(
        ESTADO_NEC_LOCAL_M1[['MES_REF','ID_PROD_UNID_FAT','NEC_ATUAL']],on=['MES_REF','ID_PROD_UNID_FAT'],how='left'
    )
    _dem['NEC_BRUTA_FINAL']=_solver_num(_dem['NEC_ATUAL'])
    RESULTADO_V66_FINAL_M1['demanda_final_m1']=_dem.drop(columns='NEC_ATUAL')

    ESTRUTURA_EXPLODIDA_FINAL_M1=ESTRUTURA_BASE_TECNICA_M1.copy()
    RASTRO_RATEIO_FINAL_M1=RASTRO_RATEIO_BASE_M1.copy()
    DIAGNOSTICO_MATRIZ_FINAL_M1=pd.DataFrame()
    DIAGNOSTICO_MATRIZ_ROTAS_FINAL_M1=pd.DataFrame()

    # Compatibilidade para barreira final/saida.
    AUDITORIA_MATRIZ_CORTES=AUDITORIA_MATRIZ_CORTES_TECNICA.copy()
    MAT_DE_ONDE_CORTAR_M1=pd.DataFrame()
    MAT_DE_ONDE_CORTAR_FER_M1=pd.DataFrame()

    # Valida fechamento tecnico sem tolerancia artificial.
    _plan_tot=PLANO_LOCAL_M1.groupby(['MES_REF','ID_PROD_UNID_FAT'],as_index=False)['QTD_PRODUZIR_SOLVER'].sum().rename(columns={'QTD_PRODUZIR_SOLVER':'QTD_PLANO'})
    _chk=ESTADO_NEC_LOCAL_M1.merge(_plan_tot,on=['MES_REF','ID_PROD_UNID_FAT'],how='left')
    _chk['QTD_PLANO']=_solver_num(_chk['QTD_PLANO'])
    # Despreza exclusivamente ruido numerico submicroscopico por chave.
    # Nao permite compensacao entre itens nem altera quantidades/capacidades.
    _gap_por_item=(_solver_num(_chk['NEC_ATUAL'])-_chk['QTD_PLANO']).clip(lower=0.0)
    _gap_final=float(_gap_por_item.where(_gap_por_item>1e-6,0.0).sum())
    _exc_rec=float(RESUMO_REC_FINAL_M1.get('ESTOURO_HR_RECURSO',pd.Series(dtype=float)).sum()) if not RESUMO_REC_FINAL_M1.empty else 0.0
    _exc_fer=float(RESUMO_FER_FINAL_M1.get('ESTOURO_HR_FER',pd.Series(dtype=float)).sum()) if not RESUMO_FER_FINAL_M1.empty else 0.0
    if _gap_final>TOL or _exc_rec>TOL_LOCAL_HR or _exc_fer>TOL_LOCAL_HR:
        raise RuntimeError(f'FECHAMENTO_TECNICO_LOCAL_M1_INVALIDO: gap={_gap_final:.9f}; REC={_exc_rec:.9f}h; FER={_exc_fer:.9f}h')

    _TEMPO_FECHAMENTO_TECNICO_LOCAL_M1=time.perf_counter()-_T0_FECH_TEC_LOCAL
    _finalizar_etapa_v58('FECHAMENTO_TECNICO_LOCAL_M1')
    LOG_RECONCILIACAO_LOCAL_M1=pd.DataFrame(LOG_GIROS_LOCAL_M1)
    print(
        f'       FECHAMENTO TECNICO LOCAL M1: {_TEMPO_FECHAMENTO_TECNICO_LOCAL_M1:.3f}s | '
        f'giros={len(LOG_RECONCILIACAO_LOCAL_M1)} | cortes={len(EVENTOS_CORTE_LOCAL_M1)} | '
        f'pcs_recuperado={PCS_RECUPERADO_LOCAL_M1:.3f} | gap_final={_gap_final:.6f}'
    )

    # Fechamento tecnico aprovado: nao existe residual objetivo antes da etapa industrial.
    _CORTE_HR_REC_FINAL_M1 = 0.0
    _CORTE_HR_FER_FINAL_M1 = 0.0
    _CORTE_HR_FINAL_M1 = 0.0
    MATRIZ_CORTES_M1_TOL_HR = TOL_LOCAL_HR
    STATUS_RECONCILIACAO_LOCAL_M1 = 'FECHOU_LOCAL_SEM_RERODAGEM_GLOBAL'

    if 'LOG_TEMPO_PROCESSAMENTO_M1' not in globals() or not isinstance(LOG_TEMPO_PROCESSAMENTO_M1, pd.DataFrame):
        LOG_TEMPO_PROCESSAMENTO_M1 = pd.DataFrame(columns=['ETAPA','TEMPO_SEG'])
    LOG_TEMPO_PROCESSAMENTO_M1 = pd.concat([LOG_TEMPO_PROCESSAMENTO_M1, pd.DataFrame([
        {'ETAPA':'FECHAMENTO_TECNICO_LOCAL','TEMPO_SEG':_TEMPO_FECHAMENTO_TECNICO_LOCAL_M1},
    ])], ignore_index=True)


    print("[03/06] Fechamento industrial final do M1...")
    _iniciar_etapa_v58('FECHAMENTO_INDUSTRIAL_M1')
    _TEMPO_FECHAMENTO_INDUSTRIAL_INICIO = time.perf_counter()

    if _CORTE_HR_FINAL_M1 > MATRIZ_CORTES_M1_TOL_HR:
        raise RuntimeError('FECHAMENTO_INDUSTRIAL_BLOQUEADO_POR_RESIDUAL_TECNICO')

    PLANO_TECNICO_FINAL_M1 = RESULTADO_V66_FINAL_M1.get('plano_producao', pd.DataFrame()).copy()
    if PLANO_TECNICO_FINAL_M1.empty:
        raise RuntimeError('PLANO_TECNICO_FINAL_M1_VAZIO')

    for _c in ['QTD_PRODUZIR_SOLVER','HR_PRODUZIR_SOLVER','PCS_HORA','HOR_REC','HOR_FER','LIMIT_PCS','LOTE_MIN','QTD_EMB']:
        if _c in PLANO_TECNICO_FINAL_M1.columns:
            PLANO_TECNICO_FINAL_M1[_c] = pd.to_numeric(PLANO_TECNICO_FINAL_M1[_c], errors='coerce').fillna(0.0)

    _CHAVE_ITEM_IND = ['MES_REF','ID_PROD_UNID_FAT']
    PLANO_TECNICO_FINAL_M1['QTD_TECNICA'] = PLANO_TECNICO_FINAL_M1['QTD_PRODUZIR_SOLVER'].clip(lower=0.0)
    _qtd_tec_item = PLANO_TECNICO_FINAL_M1.groupby(_CHAVE_ITEM_IND, as_index=False)['QTD_TECNICA'].sum().rename(columns={'QTD_TECNICA':'QTD_TECNICA_ITEM'})

    # Regras industriais por item. max() e seguro porque sao atributos do item repetidos nas rotas.
    _cols_regra = [c for c in ['LIMIT_PCS','LOTE_MIN','QTD_EMB'] if c in PLANO_TECNICO_FINAL_M1.columns]
    _regras_item = PLANO_TECNICO_FINAL_M1.groupby(_CHAVE_ITEM_IND, as_index=False)[_cols_regra].max() if _cols_regra else _qtd_tec_item[_CHAVE_ITEM_IND].copy()
    _regras_item = _qtd_tec_item.merge(_regras_item, on=_CHAVE_ITEM_IND, how='left')
    for _c in ['LIMIT_PCS','LOTE_MIN','QTD_EMB']:
        if _c not in _regras_item.columns: _regras_item[_c] = 0.0
        _regras_item[_c] = pd.to_numeric(_regras_item[_c], errors='coerce').fillna(0.0).clip(lower=0.0)

    # LIMIT_PCS, LOTE_MIN e QTD_EMB pertencem somente ao fechamento industrial.
    def _qtd_industrial_floor_v66_25(qtd, limit_pcs, lote_min, qtd_emb):
        qtd=max(0.0,float(qtd)); minimo=max(0.0,float(limit_pcs),float(lote_min)); passo=max(1.0,float(qtd_emb)) if float(qtd_emb)>TOL else 1.0
        if qtd <= TOL: return 0.0
        q=float(np.floor((qtd+TOL)/passo)*passo)
        return q if q+TOL >= minimo else 0.0

    def _qtd_industrial_ceil_v66_25(qtd, limit_pcs, lote_min, qtd_emb):
        qtd=max(0.0,float(qtd)); minimo=max(0.0,float(limit_pcs),float(lote_min)); passo=max(1.0,float(qtd_emb)) if float(qtd_emb)>TOL else 1.0
        if qtd <= TOL and minimo <= TOL: return 0.0
        alvo=max(qtd,minimo)
        return float(np.ceil((alvo-TOL)/passo)*passo)

    _regras_item['QTD_OPERACIONAL_FLOOR'] = np.fromiter((_qtd_industrial_floor_v66_25(r.QTD_TECNICA_ITEM,r.LIMIT_PCS,r.LOTE_MIN,r.QTD_EMB) for r in _regras_item.itertuples(index=False)), dtype=float, count=len(_regras_item))
    _regras_item['QTD_OPERACIONAL_CEIL'] = np.fromiter((_qtd_industrial_ceil_v66_25(r.QTD_TECNICA_ITEM,r.LIMIT_PCS,r.LOTE_MIN,r.QTD_EMB) for r in _regras_item.itertuples(index=False)), dtype=float, count=len(_regras_item))
    _regras_item['QTD_OPERACIONAL'] = _regras_item['QTD_OPERACIONAL_FLOOR']
    _regras_item['DELTA_INDUSTRIAL'] = _regras_item['QTD_OPERACIONAL'] - _regras_item['QTD_TECNICA_ITEM']

    # Distribuicao local: preserva a proporcao tecnica entre rotas; nao reabre o Solver nem redistribui a fabrica.
    def _materializar_totais_industriais_v66_25(totais_item):
        p=PLANO_TECNICO_FINAL_M1.copy()
        p=p.merge(totais_item[_CHAVE_ITEM_IND+['QTD_OPERACIONAL']], on=_CHAVE_ITEM_IND, how='left')
        tot_tec=p.groupby(_CHAVE_ITEM_IND)['QTD_TECNICA'].transform('sum')
        share=np.divide(p['QTD_TECNICA'].to_numpy(float),tot_tec.to_numpy(float),out=np.zeros(len(p),dtype=float),where=tot_tec.to_numpy(float)>TOL)
        p['QTD_OPERACIONAL_LINHA']=p['QTD_OPERACIONAL'].fillna(0.0).to_numpy(float)*share
        pcs_h=pd.to_numeric(p.get('PCS_HORA',0),errors='coerce').fillna(0.0).to_numpy(float)
        p['HR_OPERACIONAL_LINHA']=np.divide(p['QTD_OPERACIONAL_LINHA'].to_numpy(float),pcs_h,out=np.zeros(len(p),dtype=float),where=pcs_h>TOL)
        return p

    def _capacidade_ok_v66_25(p):
        prod=p[p['HR_OPERACIONAL_LINHA']>TOL].copy()
        if prod.empty: return True,0.0,0.0
        rec=prod.groupby(['MES_REF','ALOC_REC'],as_index=False).agg(HR=('HR_OPERACIONAL_LINHA','sum'),CAP=('HOR_REC','max'))
        fer=prod.groupby(['MES_REF','COD_FER_UNID'],as_index=False).agg(HR=('HR_OPERACIONAL_LINHA','sum'),CAP=('HOR_FER','max'))
        exc_rec=float((rec['HR']-rec['CAP']).clip(lower=0.0).max()) if len(rec) else 0.0
        exc_fer=float((fer['HR']-fer['CAP']).clip(lower=0.0).max()) if len(fer) else 0.0
        return max(exc_rec,exc_fer)<=TOL,exc_rec,exc_fer

    # Mapa de delta PA->PI a partir da explosao tecnica ja calculada. Nao ha nova explosao.
    _MAPA_DELTA_ESTRUTURAL_M1 = pd.DataFrame()
    if isinstance(ESTRUTURA_EXPLODIDA_FINAL_M1,pd.DataFrame) and not ESTRUTURA_EXPLODIDA_FINAL_M1.empty:
        _e=ESTRUTURA_EXPLODIDA_FINAL_M1.copy()
        req={'ID_PROD_UNID_FAT_ORIGEM','COD_INSUMO','NEC_RAIZ_PCS','NEC_GERADA_BRUTA_PCS'}
        if req.issubset(_e.columns):
            _e['NEC_RAIZ_PCS']=pd.to_numeric(_e['NEC_RAIZ_PCS'],errors='coerce').fillna(0.0)
            _e['NEC_GERADA_BRUTA_PCS']=pd.to_numeric(_e['NEC_GERADA_BRUTA_PCS'],errors='coerce').fillna(0.0)
            _e['COEF_DELTA']=np.divide(_e['NEC_GERADA_BRUTA_PCS'].to_numpy(float),_e['NEC_RAIZ_PCS'].to_numpy(float),out=np.zeros(len(_e),dtype=float),where=_e['NEC_RAIZ_PCS'].to_numpy(float)>TOL)
            _e=_e[_e['COEF_DELTA']>TOL].copy()
            _unid_child = _e['UNID_PROD'].astype(str) if 'UNID_PROD' in _e.columns else pd.Series('',index=_e.index)
            _e['CHAVE_FILHO_COD_UNID']=_e['COD_INSUMO'].astype(str)+'|'+_unid_child
            _MAPA_DELTA_ESTRUTURAL_M1=_e.groupby(['ID_PROD_UNID_FAT_ORIGEM','CHAVE_FILHO_COD_UNID'],as_index=False)['COEF_DELTA'].sum()

    # Chave auxiliar COD_PROD|UNID_FAT -> ID para localizar o PI no plano final.
    _item_ref=PLANO_TECNICO_FINAL_M1[_CHAVE_ITEM_IND+['COD_PROD','UNID_FAT']].drop_duplicates().copy()
    _item_ref['CHAVE_COD_UNID']=_item_ref['COD_PROD'].astype(str)+'|'+_item_ref['UNID_FAT'].astype(str)
    _id_por_cod_unid=dict(zip(_item_ref['CHAVE_COD_UNID'],_item_ref['ID_PROD_UNID_FAT']))

    # Busca gulosa e deterministica de melhor encaixe: cada item tenta subir do multiplo inferior para o superior.
    # Para PA, o delta positivo e propagado somente pelas relacoes ja materializadas; se algum PI nao puder ser
    # elevado industrialmente dentro da capacidade global, a combinacao inteira e descartada.
    _LOG_CANDIDATOS_INDUSTRIAIS=[]
    _totais=_regras_item.copy()
    for _r in _regras_item.sort_values(['MES_REF','ID_PROD_UNID_FAT']).itertuples(index=False):
        atual=float(_totais.loc[(_totais['MES_REF'].eq(_r.MES_REF))&_totais['ID_PROD_UNID_FAT'].eq(_r.ID_PROD_UNID_FAT),'QTD_OPERACIONAL'].iloc[0])
        cand=float(_r.QTD_OPERACIONAL_CEIL)
        if cand <= atual+TOL:
            continue
        if float(_r.QTD_TECNICA_ITEM) <= TOL:
            _LOG_CANDIDATOS_INDUSTRIAIS.append({'MES_REF':_r.MES_REF,'ID_PROD_UNID_FAT':_r.ID_PROD_UNID_FAT,'QTD_ATUAL':atual,'QTD_CANDIDATA':cand,'STATUS':'DESCARTADO','MOTIVO':'SEM_ROTA_TECNICA_ATIVA_NAO_REABRE_SOLVER','EXCESSO_REC_HR':np.nan,'EXCESSO_FER_HR':np.nan})
            continue
        prop=_totais.copy()
        prop.loc[(prop['MES_REF'].eq(_r.MES_REF))&prop['ID_PROD_UNID_FAT'].eq(_r.ID_PROD_UNID_FAT),'QTD_OPERACIONAL']=cand
        motivo='CANDIDATO_DIRETO'
        valido_estr=True
        # Propaga somente delta positivo do PA. Se nao existe mapa, o item e tratado como folha.
        if not _MAPA_DELTA_ESTRUTURAL_M1.empty:
            _m=_MAPA_DELTA_ESTRUTURAL_M1[_MAPA_DELTA_ESTRUTURAL_M1['ID_PROD_UNID_FAT_ORIGEM'].astype(str).eq(str(_r.ID_PROD_UNID_FAT))]
            if not _m.empty:
                delta_pa=max(0.0,cand-float(_r.QTD_TECNICA_ITEM))
                for _d in _m.itertuples(index=False):
                    _id_filho=_id_por_cod_unid.get(str(_d.CHAVE_FILHO_COD_UNID))
                    if not _id_filho:
                        valido_estr=False; motivo='DELTA_PI_SEM_ROTA_TECNICA'; break
                    _mask=prop['ID_PROD_UNID_FAT'].astype(str).eq(str(_id_filho))
                    if not _mask.any():
                        valido_estr=False; motivo='DELTA_PI_FORA_PLANO'; break
                    _rr=prop.loc[_mask].iloc[0]
                    alvo=float(_rr['QTD_OPERACIONAL'])+delta_pa*float(_d.COEF_DELTA)
                    alvo_val=_qtd_industrial_ceil_v66_25(alvo,_rr['LIMIT_PCS'],_rr['LOTE_MIN'],_rr['QTD_EMB'])
                    prop.loc[_mask,'QTD_OPERACIONAL']=np.maximum(prop.loc[_mask,'QTD_OPERACIONAL'].to_numpy(float),alvo_val)
                if valido_estr: motivo='DELTA_PA_PI_VALIDADO'
        if valido_estr:
            ptest=_materializar_totais_industriais_v66_25(prop)
            capok,exr,exf=_capacidade_ok_v66_25(ptest)
        else:
            capok=False; exr=exf=np.nan
        if valido_estr and capok:
            _totais=prop
            status='ACEITO'
        else:
            status='DESCARTADO'
        _LOG_CANDIDATOS_INDUSTRIAIS.append({'MES_REF':_r.MES_REF,'ID_PROD_UNID_FAT':_r.ID_PROD_UNID_FAT,'QTD_ATUAL':atual,'QTD_CANDIDATA':cand,'STATUS':status,'MOTIVO':motivo if valido_estr else motivo,'EXCESSO_REC_HR':exr,'EXCESSO_FER_HR':exf})

    PLANO_OPERACIONAL_FINAL_M1_INTERNAL=_materializar_totais_industriais_v66_25(_totais)
    PLANO_OPERACIONAL_FINAL_M1_INTERNAL['DELTA_INDUSTRIAL']=PLANO_OPERACIONAL_FINAL_M1_INTERNAL['QTD_OPERACIONAL_LINHA']-PLANO_OPERACIONAL_FINAL_M1_INTERNAL['QTD_TECNICA']
    LOG_CANDIDATOS_INDUSTRIAIS_M1=pd.DataFrame(_LOG_CANDIDATOS_INDUSTRIAIS)

    # Contrato externo: sobrescreve somente os campos operacionais existentes e remove campos internos.
    plano_producao=PLANO_OPERACIONAL_FINAL_M1_INTERNAL.copy()
    plano_producao['QTD_PRODUZIR_SOLVER']=plano_producao['QTD_OPERACIONAL_LINHA']
    plano_producao['HR_PRODUZIR_SOLVER']=plano_producao['HR_OPERACIONAL_LINHA']
    if 'PESO_PROD_KG' in plano_producao.columns and 'VOL_PRODUZIR_SOLVER' in plano_producao.columns:
        plano_producao['VOL_PRODUZIR_SOLVER']=plano_producao['QTD_PRODUZIR_SOLVER']*pd.to_numeric(plano_producao['PESO_PROD_KG'],errors='coerce').fillna(0.0)
    # 04H.1 - v66_28: sincroniza colunas ja existentes com a quantidade operacional
    # APOS o fechamento tecnico/local e o fechamento industrial definitivos.
    # Antes, apenas QTD_PRODUZIR_SOLVER era atualizada aqui; as colunas de
    # viabilidade/realocacao permaneciam com valores da fotografia inicial.
    # Nao altera producao, capacidade, cortes, rotas ou colunas da saida.
    _qtd_final_export_v6628 = pd.to_numeric(plano_producao['QTD_PRODUZIR_SOLVER'], errors='coerce').fillna(0.0).clip(lower=0.0)
    for _col_sincronia_v6628 in ['QTD_PRODUZIR_FINAL_VIAVEL_EST', 'QTD_FINAL_APOS_REALOCACAO']:
        if _col_sincronia_v6628 in plano_producao.columns:
            plano_producao[_col_sincronia_v6628] = _qtd_final_export_v6628
    if 'QTD_GAP_FINAL_APOS_REALOCACAO' in plano_producao.columns:
        # Conserva a mesma regra por linha ja usada no 06C, agora com NEC
        # consolidada depois da Matriz e QTD definitiva (nao a fotografia antiga).
        if 'NEC_PCS_CONSOLIDADA' in plano_producao.columns:
            _nec_ref_export_v6628 = pd.to_numeric(plano_producao['NEC_PCS_CONSOLIDADA'], errors='coerce').fillna(0.0)
        elif 'NEC_PCS' in plano_producao.columns:
            _nec_ref_export_v6628 = pd.to_numeric(plano_producao['NEC_PCS'], errors='coerce').fillna(0.0)
        else:
            _nec_ref_export_v6628 = pd.Series(0.0, index=plano_producao.index)
        plano_producao['QTD_GAP_FINAL_APOS_REALOCACAO'] = (_nec_ref_export_v6628 - _qtd_final_export_v6628).clip(lower=0.0)

    for _c in ['QTD_TECNICA','QTD_OPERACIONAL','QTD_OPERACIONAL_LINHA','HR_OPERACIONAL_LINHA','DELTA_INDUSTRIAL']:
        if _c in plano_producao.columns: plano_producao=plano_producao.drop(columns=_c)

    _TEMPO_FECHAMENTO_INDUSTRIAL_M1=time.perf_counter()-_TEMPO_FECHAMENTO_INDUSTRIAL_INICIO
    _finalizar_etapa_v58('FECHAMENTO_INDUSTRIAL_M1')
    print(f"       FECHAMENTO INDUSTRIAL M1: {_TEMPO_FECHAMENTO_INDUSTRIAL_M1:.3f}s | candidatos={len(LOG_CANDIDATOS_INDUSTRIAIS_M1)} | aceitos={int(LOG_CANDIDATOS_INDUSTRIAIS_M1['STATUS'].eq('ACEITO').sum()) if not LOG_CANDIDATOS_INDUSTRIAIS_M1.empty else 0}")

    # ------------------------------------------------------------
    # Estado operacional final do M1 - fonte unica para futuro M2
    # ------------------------------------------------------------
    print("[04/06] Calculando estado operacional final do M1...")
    _iniciar_etapa_v58('ESTADO_OPERACIONAL_FINAL_M1')
    _TEMPO_ESTADO_FINAL_INICIO=time.perf_counter()
    _RESULTADO_OPERACIONAL_FINAL_M1=dict(RESULTADO_V66_FINAL_M1)
    _RESULTADO_OPERACIONAL_FINAL_M1['plano_producao']=plano_producao.copy()

    # Ajusta somente o delta estrutural causado pelo fechamento industrial; nao reexplode a fabrica.
    _dem_final_op=_RESULTADO_OPERACIONAL_FINAL_M1.get('demanda_final_m1',pd.DataFrame()).copy()
    if isinstance(_dem_final_op,pd.DataFrame) and not _dem_final_op.empty and 'NEC_BRUTA_FINAL' in _dem_final_op.columns and not _MAPA_DELTA_ESTRUTURAL_M1.empty:
        _delta_pais=_totais[_CHAVE_ITEM_IND+['QTD_OPERACIONAL','QTD_TECNICA_ITEM']].copy()
        _delta_pais['DELTA_PAI']=_delta_pais['QTD_OPERACIONAL']-_delta_pais['QTD_TECNICA_ITEM']
        _delta_pais=_delta_pais[np.abs(_delta_pais['DELTA_PAI'])>TOL]
        if not _delta_pais.empty:
            _dx=_delta_pais.merge(_MAPA_DELTA_ESTRUTURAL_M1,left_on='ID_PROD_UNID_FAT',right_on='ID_PROD_UNID_FAT_ORIGEM',how='inner')
            if not _dx.empty:
                _dx['DELTA_NEC_FILHO']=_dx['DELTA_PAI']*_dx['COEF_DELTA']
                _dx['ID_FILHO']=_dx['CHAVE_FILHO_COD_UNID'].map(_id_por_cod_unid)
                _dx=_dx[_dx['ID_FILHO'].notna()]
                _delta_child=_dx.groupby(['MES_REF','ID_FILHO'],as_index=False)['DELTA_NEC_FILHO'].sum()
                _dem_final_op=_dem_final_op.merge(_delta_child,left_on=['MES_REF','ID_PROD_UNID_FAT'],right_on=['MES_REF','ID_FILHO'],how='left')
                _dem_final_op['DELTA_NEC_FILHO']=pd.to_numeric(_dem_final_op['DELTA_NEC_FILHO'],errors='coerce').fillna(0.0)
                _dem_final_op['NEC_BRUTA_FINAL']=(pd.to_numeric(_dem_final_op['NEC_BRUTA_FINAL'],errors='coerce').fillna(0.0)+_dem_final_op['DELTA_NEC_FILHO']).clip(lower=0.0)
                _dem_final_op=_dem_final_op.drop(columns=['ID_FILHO','DELTA_NEC_FILHO'],errors='ignore')
    _RESULTADO_OPERACIONAL_FINAL_M1['demanda_final_m1']=_dem_final_op
    ESTADO_OPERACIONAL_FINAL_M1=_calcular_estado_proximo_periodo_v58(BD_M1_FINAL_MATRIZ.copy(),_RESULTADO_OPERACIONAL_FINAL_M1)
    _TEMPO_ESTADO_FINAL_M1=time.perf_counter()-_TEMPO_ESTADO_FINAL_INICIO
    _finalizar_etapa_v58('ESTADO_OPERACIONAL_FINAL_M1')
    print(f"       ESTADO FINAL M1: {_TEMPO_ESTADO_FINAL_M1:.3f}s | chaves={len(ESTADO_OPERACIONAL_FINAL_M1)}")

    # ------------------------------------------------------------
    # Barreira final de validacao - leve, vetorizada, sem rerodagem
    # ------------------------------------------------------------
    print("[05/06] Validando estado operacional final do M1...")
    _iniciar_etapa_v58('VALIDACAO_FINAL_M1')
    _TEMPO_VALIDACAO_FINAL_INICIO=time.perf_counter()
    _valid=[]
    _prod=plano_producao[pd.to_numeric(plano_producao['QTD_PRODUZIR_SOLVER'],errors='coerce').fillna(0.0)>TOL].copy()
    _rec=_prod.groupby(['MES_REF','ALOC_REC'],as_index=False).agg(HR=('HR_PRODUZIR_SOLVER','sum'),CAP=('HOR_REC','max')) if not _prod.empty else pd.DataFrame(columns=['HR','CAP'])
    _fer=_prod.groupby(['MES_REF','COD_FER_UNID'],as_index=False).agg(HR=('HR_PRODUZIR_SOLVER','sum'),CAP=('HOR_FER','max')) if not _prod.empty else pd.DataFrame(columns=['HR','CAP'])
    _exc_rec=float((_rec['HR']-_rec['CAP']).clip(lower=0).max()) if len(_rec) else 0.0
    _exc_fer=float((_fer['HR']-_fer['CAP']).clip(lower=0).max()) if len(_fer) else 0.0
    _valid.append(('CAPACIDADE_REC',_exc_rec<=TOL,_exc_rec))
    _valid.append(('CAPACIDADE_FER',_exc_fer<=TOL,_exc_fer))

    _tot_final=plano_producao.groupby(_CHAVE_ITEM_IND,as_index=False)['QTD_PRODUZIR_SOLVER'].sum().merge(_regras_item[_CHAVE_ITEM_IND+['LIMIT_PCS','LOTE_MIN','QTD_EMB']],on=_CHAVE_ITEM_IND,how='left')
    _q=_tot_final['QTD_PRODUZIR_SOLVER'].to_numpy(float); _lim=_tot_final['LIMIT_PCS'].fillna(0).to_numpy(float); _lot=_tot_final['LOTE_MIN'].fillna(0).to_numpy(float); _emb=_tot_final['QTD_EMB'].fillna(0).to_numpy(float)
    _min=np.maximum(_lim,_lot); _pos=_q>TOL
    _ok_min=(~_pos)|(_q+TOL>=_min)
    _passo=np.where(_emb>TOL,_emb,1.0); _rest=np.abs(_q/_passo-np.round(_q/_passo)); _ok_mult=(~_pos)|(_rest<=1e-7)
    _valid.append(('LOTE_MIN_LIMIT_PCS',bool(_ok_min.all()),int((~_ok_min).sum())))
    _valid.append(('QTD_EMB_MULTIPLO',bool(_ok_mult.all()),int((~_ok_mult).sum())))

    _recup_aberto=0
    if isinstance(REGISTRO_CORTE_RECUPERAVEL_M1,pd.DataFrame) and not REGISTRO_CORTE_RECUPERAVEL_M1.empty:
        _recup_aberto=int((to_num(REGISTRO_CORTE_RECUPERAVEL_M1.get('PCS_CORTE_RECUPERAVEL',0))>TOL).sum())
    _valid.append(('CORTE_RECUPERAVEL_ENCERRADO',_recup_aberto==0,_recup_aberto))

    # ORI_* imutaveis: compara o congelamento com os cinco componentes originais da entrada M1.
    _ori_div=0
    if 'COMPONENTES_LOCAL_M1' in globals() and isinstance(COMPONENTES_LOCAL_M1,pd.DataFrame) and not COMPONENTES_LOCAL_M1.empty:
        _base_ori=BD_M1_BASE_LOCAL.copy()
        _base_ori=_base_ori.groupby(['MES_REF','ID_PROD_UNID_FAT'],as_index=False).agg({
            'LTP_CART_ARR_MES_ANT':'max','LTP_CART_MES_ATUAL':'max','LTP_SALDO_PREV_PCS':'max',
            'LTP_SALDO_PREV_PROX_MES_PCS':'max','LTP_EST_SEG_PCS':'max'
        })
        _cmp_ori=COMPONENTES_LOCAL_M1[['MES_REF','ID_PROD_UNID_FAT','ORI_C_ARR','ORI_C_AT','ORI_PREV_VEN','ORI_PREV_PROX','ORI_EST_SEG']].copy()
        _oo=_base_ori.merge(_cmp_ori,on=['MES_REF','ID_PROD_UNID_FAT'],how='outer').fillna(0.0)
        _pares_ori=[
            ('LTP_CART_ARR_MES_ANT','ORI_C_ARR'),('LTP_CART_MES_ATUAL','ORI_C_AT'),
            ('LTP_SALDO_PREV_PCS','ORI_PREV_VEN'),('LTP_SALDO_PREV_PROX_MES_PCS','ORI_PREV_PROX'),
            ('LTP_EST_SEG_PCS','ORI_EST_SEG')
        ]
        for _a,_b in _pares_ori:
            _ori_div+=int((np.abs(pd.to_numeric(_oo[_a],errors='coerce').fillna(0)-pd.to_numeric(_oo[_b],errors='coerce').fillna(0))>TOL).sum())
    _valid.append(('ORI_IMUTAVEIS',_ori_div==0,_ori_div))

    # Duplicidade da chave de alocacao positiva.
    _dup_cols=[c for c in ['MES_REF','ID_PROD_UNID_FAT','ALOC_REC','COD_FER_UNID','PRIOR_MATPAR','PRIOR_ROT'] if c in _prod.columns]
    _dup=int(_prod.duplicated(subset=_dup_cols,keep=False).sum()) if _dup_cols else 0
    _valid.append(('DUPLICIDADE_PLANO_POSITIVO',_dup==0,_dup))

    # Campos internos nao podem vazar para o cliente.
    _vaz=[c for c in ['QTD_TECNICA','QTD_OPERACIONAL','DELTA_INDUSTRIAL','QTD_OPERACIONAL_LINHA','HR_OPERACIONAL_LINHA'] if c in plano_producao.columns]
    _valid.append(('CONTRATO_EXTERNO_SEM_CAMPOS_INTERNOS',len(_vaz)==0,','.join(_vaz)))

    # Corte fisico unico / sem sobrecorte: corte liquido final deve fechar exatamente ORI - componente final.
    _sobrecorte=0
    _dif_corte_max=0.0
    if 'COMPONENTES_LOCAL_M1' in globals() and isinstance(COMPONENTES_LOCAL_M1,pd.DataFrame) and not COMPONENTES_LOCAL_M1.empty:
        _aud_reg=REGISTRO_CORTE_RECUPERAVEL_M1.copy() if isinstance(REGISTRO_CORTE_RECUPERAVEL_M1,pd.DataFrame) else pd.DataFrame()
        for _comp,_ori in MAPA_ORI_LOCAL_M1.items():
            _exp=COMPONENTES_LOCAL_M1[['ID_PROD_UNID_FAT',_comp,_ori]].copy()
            _exp['CORTE_ESPERADO']=(pd.to_numeric(_exp[_ori],errors='coerce').fillna(0)-pd.to_numeric(_exp[_comp],errors='coerce').fillna(0)).clip(lower=0.0)
            if not _aud_reg.empty:
                _a=_aud_reg[_aud_reg['COMPONENTE_CORTE'].eq(_comp)].groupby('ID_PROD_UNID_FAT',as_index=False)['PCS_CORTE_FINAL'].sum().rename(columns={'PCS_CORTE_FINAL':'CORTE_AUDITADO'})
            else:
                _a=pd.DataFrame(columns=['ID_PROD_UNID_FAT','CORTE_AUDITADO'])
            _z=_exp.merge(_a,on='ID_PROD_UNID_FAT',how='left')
            _z['CORTE_AUDITADO']=pd.to_numeric(_z['CORTE_AUDITADO'],errors='coerce').fillna(0.0)
            _dif=np.abs(_z['CORTE_ESPERADO']-_z['CORTE_AUDITADO'])
            _dif_corte_max=max(_dif_corte_max,float(_dif.max()) if len(_dif) else 0.0)
            _sobrecorte+=int((_z['CORTE_AUDITADO']>pd.to_numeric(_z[_ori],errors='coerce').fillna(0)+TOL).sum())
    _valid.append(('CORTE_FISICO_UNICO_FECHA_COMPONENTES',_dif_corte_max<=1e-6,_dif_corte_max))
    _valid.append(('AUSENCIA_SOBRECORTE',_sobrecorte==0,_sobrecorte))

    # Coerencia HR x QTD / PCS_HORA no plano operacional final.
    _hr_div=0.0
    if not _prod.empty:
        _pcs=pd.to_numeric(_prod['PCS_HORA'],errors='coerce').fillna(0.0).to_numpy(float)
        _qv=pd.to_numeric(_prod['QTD_PRODUZIR_SOLVER'],errors='coerce').fillna(0.0).to_numpy(float)
        _hrv=pd.to_numeric(_prod['HR_PRODUZIR_SOLVER'],errors='coerce').fillna(0.0).to_numpy(float)
        _hr_calc=np.divide(_qv,_pcs,out=np.zeros(len(_prod),dtype=float),where=_pcs>TOL)
        _hr_div=float(np.max(np.abs(_hrv-_hr_calc))) if len(_prod) else 0.0
    _valid.append(('COERENCIA_QTD_HR_PLANO',_hr_div<=1e-7,_hr_div))

    # Delta industrial PA->PI: todo aumento de PA deve estar sustentado pelo aumento operacional dos filhos afetados.
    _delta_estr_falha=0
    _delta_estr_max=0.0
    if isinstance(_MAPA_DELTA_ESTRUTURAL_M1,pd.DataFrame) and not _MAPA_DELTA_ESTRUTURAL_M1.empty:
        _it=_totais[_CHAVE_ITEM_IND+['QTD_TECNICA_ITEM','QTD_OPERACIONAL']].copy()
        _it['DELTA_OP']=(pd.to_numeric(_it['QTD_OPERACIONAL'],errors='coerce').fillna(0)-pd.to_numeric(_it['QTD_TECNICA_ITEM'],errors='coerce').fillna(0))
        _pais=_it[_it['DELTA_OP']>TOL][['ID_PROD_UNID_FAT','DELTA_OP']].rename(columns={'ID_PROD_UNID_FAT':'ID_PROD_UNID_FAT_ORIGEM'})
        if not _pais.empty:
            _req=_pais.merge(_MAPA_DELTA_ESTRUTURAL_M1,on='ID_PROD_UNID_FAT_ORIGEM',how='inner')
            if not _req.empty:
                _req['DELTA_REQ_FILHO']=_req['DELTA_OP']*pd.to_numeric(_req['COEF_DELTA'],errors='coerce').fillna(0.0)
                _req['ID_FILHO']=_req['CHAVE_FILHO_COD_UNID'].map(_id_por_cod_unid)
                _req=_req[_req['ID_FILHO'].notna()]
                _req=_req.groupby('ID_FILHO',as_index=False)['DELTA_REQ_FILHO'].sum()
                _filhos=_it[['ID_PROD_UNID_FAT','DELTA_OP']].rename(columns={'ID_PROD_UNID_FAT':'ID_FILHO','DELTA_OP':'DELTA_FILHO'})
                _req=_req.merge(_filhos,on='ID_FILHO',how='left')
                _req['DELTA_FILHO']=pd.to_numeric(_req['DELTA_FILHO'],errors='coerce').fillna(0.0)
                _req['DEFICIT_DELTA']=(pd.to_numeric(_req['DELTA_REQ_FILHO'],errors='coerce').fillna(0.0)-_req['DELTA_FILHO']).clip(lower=0.0)
                _delta_estr_max=float(_req['DEFICIT_DELTA'].max()) if len(_req) else 0.0
                _delta_estr_falha=int((_req['DEFICIT_DELTA']>1e-6).sum())
    _valid.append(('DELTA_INDUSTRIAL_PA_PI',_delta_estr_falha==0,_delta_estr_max))

    # Estado final para M2: chave unica, estoque/carteira nao negativos e mutuamente exclusivos.
    _estado_dup=0; _estado_neg=0; _estado_duplo=0; _estado_dif=0.0
    if isinstance(ESTADO_OPERACIONAL_FINAL_M1,pd.DataFrame) and not ESTADO_OPERACIONAL_FINAL_M1.empty:
        _estado_dup=int(ESTADO_OPERACIONAL_FINAL_M1.duplicated(subset=['COD_PROD','UNID_FAT'],keep=False).sum())
        _est=pd.to_numeric(ESTADO_OPERACIONAL_FINAL_M1['ESTOQUE_INICIAL_PROX_PERIODO'],errors='coerce').fillna(0.0)
        _arr=pd.to_numeric(ESTADO_OPERACIONAL_FINAL_M1['CARTEIRA_ARRASTE_PROX_PERIODO'],errors='coerce').fillna(0.0)
        _saldo=pd.to_numeric(ESTADO_OPERACIONAL_FINAL_M1['SALDO_PROX_PERIODO'],errors='coerce').fillna(0.0)
        _estado_neg=int(((_est<-TOL)|(_arr<-TOL)).sum())
        _estado_duplo=int(((_est>TOL)&(_arr>TOL)).sum())
        _estado_dif=float(np.max(np.abs(_saldo-(_est-_arr)))) if len(_saldo) else 0.0
    _valid.append(('ESTADO_M1_CHAVE_UNICA',_estado_dup==0,_estado_dup))
    _valid.append(('ESTOQUE_CARTEIRA_VALIDOS',_estado_neg==0 and _estado_duplo==0 and _estado_dif<=1e-7,f'neg={_estado_neg}; duplo={_estado_duplo}; dif={_estado_dif:.3e}'))

    # 04H.2 - barreira leve de sincronizacao das colunas publicadas (v66_28).
    _qtd_final_valid_v6628 = pd.to_numeric(plano_producao['QTD_PRODUZIR_SOLVER'], errors='coerce').fillna(0.0)
    for _col_valid_v6628 in ['QTD_PRODUZIR_FINAL_VIAVEL_EST', 'QTD_FINAL_APOS_REALOCACAO']:
        if _col_valid_v6628 in plano_producao.columns:
            _div_v6628 = np.abs(pd.to_numeric(plano_producao[_col_valid_v6628], errors='coerce').fillna(0.0) - _qtd_final_valid_v6628)
            _max_div_v6628 = float(_div_v6628.max()) if len(_div_v6628) else 0.0
            _valid.append(('SINCRONIA_' + _col_valid_v6628, _max_div_v6628 <= TOL, _max_div_v6628))

    VALIDACAO_FINAL_M1=pd.DataFrame(_valid,columns=['VALIDACAO','PASSOU','VALOR_DIAGNOSTICO'])
    _TEMPO_VALIDACAO_FINAL_M1=time.perf_counter()-_TEMPO_VALIDACAO_FINAL_INICIO
    _finalizar_etapa_v58('VALIDACAO_FINAL_M1')
    print(VALIDACAO_FINAL_M1.to_string(index=False))
    print(f"       VALIDACAO FINAL M1: {_TEMPO_VALIDACAO_FINAL_M1:.3f}s")
    if not bool(VALIDACAO_FINAL_M1['PASSOU'].all()):
        _falhas=', '.join(VALIDACAO_FINAL_M1.loc[~VALIDACAO_FINAL_M1['PASSOU'],'VALIDACAO'].astype(str))
        raise RuntimeError(f'BARREIRA_FINAL_M1_REPROVADA: {_falhas}')

    # ------------------------------------------------------------
    # Auditoria externa aprovada: corte liquido final da Matriz apos recuperacao local
    # ------------------------------------------------------------
    AUDITORIA_MATRIZ_CORTES = AUDITORIA_MATRIZ_CORTES_TECNICA.copy()

    # Completa o log de tempos do M1 sem expor estruturas internas no Excel.
    LOG_TEMPO_PROCESSAMENTO_M1=pd.concat([LOG_TEMPO_PROCESSAMENTO_M1,pd.DataFrame([
        {'ETAPA':'FECHAMENTO_INDUSTRIAL','TEMPO_SEG':_TEMPO_FECHAMENTO_INDUSTRIAL_M1},
        {'ETAPA':'ESTADO_OPERACIONAL_FINAL','TEMPO_SEG':_TEMPO_ESTADO_FINAL_M1},
        {'ETAPA':'VALIDACAO_FINAL','TEMPO_SEG':_TEMPO_VALIDACAO_FINAL_M1},
    ])],ignore_index=True)

    # The operational state must be generated after industrial closure, not from the preliminary global pass.
    if ESTADO_OPERACIONAL_FINAL_M1.empty:
        raise RuntimeError(f'ESTADO_FINAL_VAZIO: {_mes_ref_v58:%Y-%m}')
    for _r_v58 in ESTADO_OPERACIONAL_FINAL_M1.itertuples(index=False):
        _chave_v58 = _chave_estado_v58(_r_v58.COD_PROD, _r_v58.UNID_FAT)
        _estado_itens_v58[_chave_v58] = {'ESTOQUE': float(_r_v58.ESTOQUE_INICIAL_PROX_PERIODO), 'ARRASTE': float(_r_v58.CARTEIRA_ARRASTE_PROX_PERIODO)}
    _saldos_periodos_v58.append(ESTADO_OPERACIONAL_FINAL_M1.copy())
    _planos_finais_periodos.append(plano_producao.copy())
    _auditorias_finais_periodos.append(AUDITORIA_MATRIZ_CORTES.copy())
    _validacoes_finais_periodos.append(VALIDACAO_FINAL_M1.assign(MES_REF=_mes_ref_v58).copy())
    _log_tempo_periodos.append(LOG_TEMPO_PROCESSAMENTO_M1.assign(MES_REF=_mes_ref_v58).copy())
    _tempos_periodos_v58.append({'MES_REF': _mes_ref_v58, 'SEGUNDOS': time.perf_counter()-_inicio_periodo_v58})
    print(f'       PERIODO {_idx_periodo_v58+1}/{len(periodos_v58)} FECHADO: {_mes_ref_v58:%Y-%m}')


# Only operationally approved periods contribute to the final output.
plano_producao = pd.concat(_planos_finais_periodos, ignore_index=True, sort=False)
AUDITORIA_MATRIZ_CORTES = pd.concat(_auditorias_finais_periodos, ignore_index=True, sort=False)
VALIDACAO_FINAL_CICLO = pd.concat(_validacoes_finais_periodos, ignore_index=True, sort=False)
LOG_TEMPO_PROCESSAMENTO_CICLO = pd.concat(_log_tempo_periodos, ignore_index=True, sort=False)
saldo_encadeamento_v58 = pd.concat(_saldos_periodos_v58, ignore_index=True, sort=False)
if not bool(VALIDACAO_FINAL_CICLO['PASSOU'].all()):
    raise RuntimeError('CICLO_COMPLETO_VALIDACAO_REPROVADA')
print(f'       CICLO CONCLUIDO - {len(periodos_v58)} periodo(s) | {len(plano_producao):,} linhas de plano')
_finalizar_etapa_v58('SOLVER_ENCADEAMENTO')


# 05. Montagem e exportação do resultado


In [ ]:
print("[06/06] Gravando arquivo final...")
_iniciar_etapa_v58("EXPORTACAO_EXCEL")
# ============================================================
# 07. EXPORTAÇÃO
# ============================================================



# Simplificação rotina:
# - mantém a limpeza da guia 15: não leva os fatores FATOR_FINAL_VIAVEL_ORIGEM e FATOR_AJUSTE_SOBRE_SOLVER;
# - as guias 17_RESUMO_REALOCACAO e 17_RESUMO_REALOCACAO_POOL seguem removidas do arquivo final;
OUTPUT_SOLVER_FILE.parent.mkdir(parents=True, exist_ok=True)

_GUIAS_DADOS_EXPORTAR = {
    "03_PLANO_PRODUCAO": plano_producao,
    "04_AUDITORIA_MATRIZ_CORTES": AUDITORIA_MATRIZ_CORTES,
}

_guias_ativas = [nome for nome, ativo in GUIAS_EXPORTAR.items() if ativo]
if not _guias_ativas:
    raise RuntimeError("NENHUMA_GUIA_ATIVA_PARA_EXPORTACAO")

with pd.ExcelWriter(OUTPUT_SOLVER_FILE, engine="openpyxl") as writer:
    for _nome_guia in _guias_ativas:
        _GUIAS_DADOS_EXPORTAR[_nome_guia].to_excel(writer, sheet_name=_nome_guia, index=False)

# Formatação básica
wb = load_workbook(OUTPUT_SOLVER_FILE)
header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells[:1000]:
            if cell.value is not None:
                max_len = max(max_len, min(len(str(cell.value)), 60))
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 10), 45)
wb.save(OUTPUT_SOLVER_FILE)


_finalizar_etapa_v58("EXPORTACAO_EXCEL")
if 'CURVA_ABC' in plano_producao.columns and plano_producao['CURVA_ABC'].isna().all():
    print('       ATENCAO - CURVA_ABC nao encontrada/preenchida na base de entrada')
if 'CURVA_123' in plano_producao.columns and plano_producao['CURVA_123'].isna().all():
    print('       ATENCAO - CURVA_123 nao encontrada/preenchida na base de entrada')
print(f"       OK - Arquivo gravado: {OUTPUT_SOLVER_FILE.name}")


# 06. Finalização e tempos por fase


In [ ]:
print("[FINAL] Consolidando tempos por fase...")
tempo_total = time.perf_counter() - _TEMPO_TOTAL_INICIO_V58
segundos_total = int(round(tempo_total))
horas, resto = divmod(segundos_total, 3600)
minutos, segundos = divmod(resto, 60)
print("=" * 72)
print(" PROCESSAMENTO MENSAL CONCLUIDO COM SUCESSO")
print(f" TEMPO TOTAL: {horas:02d}:{minutos:02d}:{segundos:02d} ({tempo_total:.3f}s)")
print("-" * 72)
print(" TEMPOS POR MACROFASE")
for _nome_v58, _seg_v58 in TEMPOS_ETAPAS_V58.items():
    print(f"  {_nome_v58:<36} {_seg_v58:>12.3f}s")
if 'LOG_TEMPO_PROCESSAMENTO_M1' in globals() and isinstance(LOG_TEMPO_PROCESSAMENTO_M1, pd.DataFrame) and not LOG_TEMPO_PROCESSAMENTO_M1.empty:
    print("-" * 72)
    print(" TEMPOS INTERNOS M1")
    _lt=LOG_TEMPO_PROCESSAMENTO_M1.groupby('ETAPA',as_index=False)['TEMPO_SEG'].sum()
    for _r in _lt.itertuples(index=False):
        print(f"  {str(_r.ETAPA):<36} {float(_r.TEMPO_SEG):>12.3f}s")

if 'LOG_RECUPERACAO_PERFORMANCE_M1' in globals():
    _lr=LOG_RECUPERACAO_PERFORMANCE_M1
    print("-" * 72)
    print(" RECUPERACAO LOCAL - v66_26")
    print(f"  Testes de restauracao: {_lr['TESTES_RESTAURACAO']:,}")
    print(f"  Bissecoes executadas: {_lr['EVENTOS_COM_BISSECAO']:,}")
    print(f"  Bissecoes puladas (zero): {_lr['EVENTOS_BISSECAO_PULADA_ZERO']:,}")
    print(f"  Tempo recuperacao: {_lr['TEMPO_RECUPERACAO_SEG']:.3f}s")

print("-" * 72)
print(f"  Auditoria Matriz: {len(AUDITORIA_MATRIZ_CORTES):,} evento(s)")
print(f"  Estado operacional M1: {len(ESTADO_OPERACIONAL_FINAL_M1):,} chave(s)")
print(f'  PERIODOS CONCLUIDOS: {len(periodos_v58)}')
print("=" * 72)